# Batch: a list of runs, unattended

This notebook solves a list of runs one after another and writes each to `results/`, exactly
as the solver notebook does for one run, then puts them all in one zip for the ThermaSched
site. Use the solver notebook to look at one run step by step; use this one to produce a set
of results, such as every thesis scenario.

The notebook is self-contained: section 0 holds a copy of the model's code. It needs a
project folder with `scenarios/` in it, written by the cleaning notebook (`data_prep.ipynb`),
and scenarios built in the solver notebook can be listed too.

**How to run.**

- **Google Colab:** upload this notebook, and use the same Google Drive folder as the other
  notebooks (by default `My Drive/thermasched/`; set `COLAB_DRIVE_FOLDER` below if it is
  elsewhere). Press *Runtime › Run all* and keep the tab open. Colab asks for permission to
  mount Drive, and HiGHS (the solver) is installed on the first run.
- **Locally:** Python 3.11 or later with `highspy`, `numpy`, `pandas`, `plotly` and Jupyter
  (`pip install highspy numpy pandas plotly jupyter`). Keep this notebook in `notebooks/`
  inside the project folder, or start Jupyter in the project folder, and press **Run All**.

**Re-planning.** By default (`PER_TICK = True`) every run re-plans at every 5-minute period
with only the workloads that have arrived, as the solver notebook explains in its section 4.
A per-tick run saves its state after every period, so a run cut off half-way goes on from
where it stopped.

**Run time.** Per tick, a run takes about its number of periods times `TICK_TIME_LIMIT_S`
at most (about five hours for a 288-period day at 60 seconds), plus up to
`TICK_NO_SCHEDULE_LIMIT_S` for each period that finds no schedule within that limit (rare);
most periods solve well within the limit, so a run usually takes less. For the whole-day plan (`PER_TICK = False`), a run with
`reassignment="none"` takes seconds and every other run takes up to its time limit, 30
minutes for the runs below.
Section 1 shows the plan and an upper bound before anything is solved.

**If the session stops** (Colab disconnects, the laptop sleeps): run all cells again. Each run
is saved to Google Drive or disk as soon as it finishes, and runs already done with this
release on the same scenario build are skipped, so the list picks up where it stopped.

**Release.** Section 0 prints the ThermaSched release. Every run records it, with a
fingerprint of the scenario it was solved on, and the site shows both.

## Settings

**`RUNS`** is the list to solve, in order. Each entry is a label and the switches that differ
from their defaults. The switches are the settings of the solver notebook's `RUN`, with the
same names and defaults:

| Switch | Default | What it does |
|---|---|---|
| `scenario` | — | folder under `scenarios/` |
| `baseline_rule` | `"least_loaded"` | `"least_loaded"`, `"fastest_first"` or `"random"` (uses `seed`) |
| `baseline_supply_temp_C` | `18.0` | the baseline's fixed CRAC supply air temperature, °C |
| `reassignment` | `"free"` | `"free"` (any server), `"same_zone"`, or `"none"` (baseline server) |
| `placement` | `"thermal_aware"` | or `"thermal_blind"`: servers are chosen with server temperature left out, then start times and cooling are solved on those servers with every thermal limit (two passes) |
| `supply_temp_mode` | `"optimize"` | `"optimize"` per zone and period, or `"fixed"` at `fixed_supply_temp_C` (default: the baseline's) |
| `H_min_C` | the scenario's (10) | minimum thermal headroom, °C; the server temperature cap is 90 − H_min |
| `time_limit_s`, `mip_rel_gap` | `600.0`, `0.001` | when the solver stops |
| `cold_plates`, `cold_plate_count` | `True`, `None` | model cold plates (plates per server from `servers.csv`, or this many on every server); on in every Chapter 6 run |
| `cp_mode` | `"optimize"` | `"fixed"` holds the cold plates at `fixed_cp_flow_Lmin` / `fixed_cp_coolant_C` (default: the baseline's 0.30 L/min, 20 °C) |
| `priority` | `True` | the priority rules, Eq. (56)-(57) |
| `zone_map` | `None` | zone of each server in order, e.g. `(1, 1, 2, 2, 2, 2)` on six servers |
| `ci_flat`, `carbon_file` | `False`, `None` | carbon intensity held at the mean, or read from a CSV with columns `t`, `CI_t_g_per_kWh` |

The list below is the thesis set: every test of sections 6.1 to 6.4, labelled as on the
site, on the six-server fleet (two R840s at 2.1 GHz, two R840s at 2.4 GHz, two C6320s at
2.6 GHz with cold plates) on a light, normal and heavy day (`servers6_light`, `servers6`,
`servers6_heavy`), 36 runs. For the ten-server fleet (3 / 4 / 3 servers), point `DAYS` at
`servers10_light`, `servers10` and `servers10_heavy`, and give 6.1 Regrouped zones
`zone_map=(1, 1, 1, 2, 2, 2, 2, 2, 2, 2)`: the three 2.1 GHz R840s in zone 1, the rest in
zone 2. The deeper carbon swing of 6.4 runs on each day's `_swing`
scenario, which holds the same workloads under a carbon intensity that swings ±50 % of its mean
over the day instead of ±25 %. The site labels a run by its
switches and scenario, not by the label here. Delete or comment out entries to run fewer.
**`PER_TICK`** re-plans every run at every period (`True`, the default) or plans the whole
day at once (`False`); **`TICK_TIME_LIMIT_S`** is the solver time limit of each period's
solve, and a period with no schedule by then is solved again up to
**`TICK_NO_SCHEDULE_LIMIT_S`** in all. Per tick, `time_limit_s` plays no part.
**`DEFAULTS`** is applied to every run before its own switches, for example
`dict(time_limit_s=1200.0)` to give every run 20 minutes.

Each run is written to `results/<scenario>/nb_<run name>/`, the name built from its switches
as in the solver notebook, so a run made here and the same run made there share a folder.

In [ ]:
from pathlib import Path

# Project folder: the one that holds scenarios/; results/ and exports/ are written into it. On
# Colab it is a folder on Google Drive, under "My Drive". Locally it is found on its own when
# Jupyter starts in it or in its notebooks/ folder.
COLAB_DRIVE_FOLDER = "thermasched"
LOCAL_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# The six-server fleet on each load level
DAYS = {"light day": "servers6_light", "normal day": "servers6", "heavy day": "servers6_heavy"}
# Chapter 6 of the thesis, sections 6.1 to 6.4. The 6.1 Base is also the base of 6.2, 6.3 and
# 6.4 (cold plates are on in every run, so it has both cooling systems optimized).
CHAPTER6 = [
    ("6.1 Base",                                 dict()),
    ("6.1 Regrouped zones",                      dict(zone_map=(1, 1, 2, 2, 2, 2))),
    ("6.1 Same-zone moves only",                 dict(reassignment="same_zone")),
    ("6.2 CRAC changed, cold plates fixed",      dict(cp_mode="fixed")),
    ("6.2 Cold plates changed, CRAC fixed",      dict(supply_temp_mode="fixed")),
    ("6.3 High safety margin, 19 °C",            dict(H_min_C=19.0)),
    ("6.3 Medium safety margin, 15 °C",          dict(H_min_C=15.0)),
    ("6.3 Low safety margin, 5 °C",              dict(H_min_C=5.0)),
    ("6.4 No reassignment",                      dict(reassignment="none")),
    ("6.4 Moves chosen without temperature",     dict(placement="thermal_blind")),
    ("6.4 Flat carbon intensity",                dict(ci_flat=True)),
    # the same day's workloads on a grid with a deeper daily carbon swing: its _swing scenario
    ("6.4 Deeper daily carbon swing",            dict(swing=True)),
]


def chapter6_run(scenario, switches):
    switches = dict(switches)
    if switches.pop("swing", False):
        scenario += "_swing"
    return dict(scenario=scenario, time_limit_s=1800.0, **switches)


# Each test on each day, 30 minutes each for a whole-day plan: 36 runs
RUNS = [(f"{label}, {day}", chapter6_run(scenario, switches))
        for day, scenario in DAYS.items() for label, switches in CHAPTER6]
DEFAULTS = dict()            # switches applied to every run first, e.g. dict(time_limit_s=1200.0)
PER_TICK = True              # re-plan at every 5-minute period with only the workloads that have
                             #   arrived; False: one plan for the whole day, every arrival known
TICK_TIME_LIMIT_S = 60.0     # per tick: solver time limit of each period's solve, s
TICK_NO_SCHEDULE_LIMIT_S = 300.0  # per tick: a period with no schedule by then is solved again,
                             #   up to this many seconds in all

SKIP_DONE = True             # skip runs already in results/ from this release and scenario build
ZIP_NAME = "batch"           # exports/<ZIP_NAME>.zip holds every run of the list that has results
DOWNLOAD_ZIP = True          # on Colab, download the zip at the end

## 0. Setup

Finds the project folder (on Colab, mounts Google Drive), installs HiGHS if it is missing,
and writes the model code in the cells below to a temporary folder `LIB`, from which the rest
of the notebook imports it. There is nothing to change here.

The code cells after it are the model code of this release. Edits made there are lost when
the notebook is replaced by a newer release. In Colab or Jupyter, collapse this section to
hide them.

In [ ]:
import importlib.util
import shutil
import subprocess
import sys
import tempfile

IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    ROOT = Path("/content/drive/MyDrive") / COLAB_DRIVE_FOLDER
else:
    ROOT = LOCAL_ROOT
SCENARIOS_DIR = ROOT / "scenarios"
RESULTS_DIR = ROOT / "results"
EXPORTS_DIR = ROOT / "exports"                   # zips of runs, for the site
if not SCENARIOS_DIR.is_dir():
    raise FileNotFoundError(f"no scenarios/ folder in {ROOT}; run the cleaning notebook first")

# HiGHS solves the MILP. numpy, pandas and plotly come with Colab.
if importlib.util.find_spec("highspy") is None:
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "highspy"], check=True)

# A fresh copy of the model code, imported from LIB ahead of any other copy.
LIB = Path(tempfile.gettempdir()) / "thermasched_batch"
shutil.rmtree(LIB, ignore_errors=True)
for package in ("scheduler", "dashboard"):
    (LIB / package).mkdir(parents=True)
    (LIB / package / "__init__.py").write_text("")
    for name in [m for m in sys.modules if m == package or m.startswith(package + ".")]:
        del sys.modules[name]
if str(LIB) in sys.path:
    sys.path.remove(str(LIB))
sys.path.insert(0, str(LIB))
print(f"Project folder: {ROOT}\nModel code written to: {LIB}")

### Model code

The model code of this release; see above.

In [ ]:
%%writefile {LIB}/scheduler/version.py
"""Release version, written into every scenario (params.json) and run (summary.json).

VERSION must equal ``version`` in pyproject.toml and have an entry in CHANGELOG.md
(tests/test_release.py checks both). The notebooks carry a copy of it, so a run made with
an old notebook shows its version on the dashboard.

SCHEMA_VERSION is the layout of summary.json. Raise it when a field is renamed or removed,
and keep the dashboard able to read every earlier layout.
"""

VERSION = "1.5.0"
SCHEMA_VERSION = 1


In [ ]:
%%writefile {LIB}/scheduler/types.py
"""Shared data structures for the thermal-aware scheduling prototype.

This file is the contract between the loader, the baseline, the evaluator, the
optimizer, and the report writer. Equation numbers refer to docs/PROJECT_CONTEXT.md.

Conventions (all modules):
- Periods are numbered 1..T as in the scenario files. Arrays over periods have length T
  and are indexed by ``t - 1``.
- Servers are numbered 1..S, zones 1..Z, workloads 1..I, all contiguous. Arrays over
  servers are indexed by ``s - 1`` and over zones by ``z - 1``.
- A workload that starts in period ``start`` and runs ``n`` periods occupies periods
  ``start, ..., start + n - 1`` and has ``end = start + n`` (exclusive, the same as
  ``end`` in ``baseline_schedule.csv``). It is on time when ``end <= D_i``.
- ``T[s, t]`` is the server temperature at the end of period t, produced by the power
  drawn during period t. ``DPF[s, t]`` is taken from ``T[s, t]`` of the same period.
- Units: power in W, ``Q_crac`` and ``P_crac`` in kW, ``dt`` in seconds, energy in kWh,
  carbon intensity in g/kWh, carbon in g, work in MHz*s, temperature in degC.
"""

from __future__ import annotations

from dataclasses import asdict, dataclass, field

import numpy as np

BASELINE_RULES = ("least_loaded", "fastest_first", "random")
REASSIGNMENT_MODES = ("free", "same_zone", "none")
SUPPLY_TEMP_MODES = ("optimize", "fixed")
COLD_PLATE_MODES = ("optimize", "fixed")
# thermal_blind: servers are chosen with server temperature left out, then the full model sets
# start times and cooling on those servers (two passes, see run.solve_thermal_blind).
PLACEMENT_MODES = ("thermal_aware", "thermal_blind")

# Constraint families that the infeasibility diagnosis relaxes one at a time.
RELAX_DEADLINES = "deadlines_and_waiting"   # (27), (28): windows widened to the horizon
RELAX_THERMAL = "thermal_cap"               # (31)-(33): T <= T_max - H_min dropped
RELAX_CAPACITY = "capacity"                 # (12), (13)
RELAX_COOLING = "cooling_capacity"          # (54): Q_crac <= Q_crac_kW
RELAX_PRIORITY = "priority"                 # (56), (57): start order dropped
RELAX_FAMILIES = (RELAX_DEADLINES, RELAX_THERMAL, RELAX_CAPACITY, RELAX_COOLING, RELAX_PRIORITY)

# Solver outcomes reported by the optimizer.
STATUS_OPTIMAL = "optimal"            # solved to the requested gap
STATUS_TIME_LIMIT = "time_limit"      # stopped on the time limit with a valid schedule
STATUS_INFEASIBLE = "infeasible"      # proven: no valid schedule exists
STATUS_NO_SOLUTION = "no_solution"    # stopped on the time limit with no schedule found

# The solver outcomes in plain words, wherever a run's status is shown.
STATUS_TEXT = {
    STATUS_OPTIMAL: "Proven best",
    STATUS_TIME_LIMIT: "Stopped at time limit",
    STATUS_INFEASIBLE: "No valid schedule (proven)",
    STATUS_NO_SOLUTION: "Stopped, no schedule found",
}


def status_text(status: str | None) -> str:
    """'time_limit' -> 'Stopped at time limit'; an unknown status is returned as it is."""
    return STATUS_TEXT.get(status or "", status or "unknown")


class NotDefinedYet(NotImplementedError):
    """Raised when a switch asks for a term the thesis does not define yet."""


@dataclass(frozen=True)
class Workload:
    i: int
    workload_id: str
    flavor: str
    A: int                # first period it may run
    D: int                # period by which it must be finished (end <= D)
    AD: int               # maximum allowable delay in periods, Eq. (22)
    ET_base: int          # baseline execution time in periods at f_ref
    RT: int               # 1 time-sensitive, 0 delay-tolerant
    B: int                # 1 already started: stays on its original server
    X0: int | None        # server in the scenario's own least-loaded baseline
    C_req: int            # cores
    M_req_MB: float
    W_MHz_s: float        # work required
    usage_imputed: int = 0


@dataclass(frozen=True)
class Server:
    s: int
    name: str
    zone: int
    f_MHz: float
    C_cores: int
    M_MB: float
    P_idle_W: float
    P_max_W: float
    T0_C: float
    CT_cold_plate: int    # CT_s: 1 if the server has cold-plate cooling
    th_a_C: float         # a_s
    th_R_C_per_W: float   # R_s
    th_alpha: float       # alpha_s per period
    n_cold_plates: int | None = None   # editable count; None: CT_cold_plate (0 or 1)

    @property
    def plates(self) -> int:
        """Cold plates installed on the server."""
        return self.CT_cold_plate if self.n_cold_plates is None else self.n_cold_plates


@dataclass(frozen=True)
class Constants:
    """Constants of docs/PROJECT_CONTEXT.md section 4.1 (params.json, constants_as_printed)."""
    R_dvfs: float
    F_throttle: float
    T_max_server_C: float
    H_min_C: float
    T_dvfs_C: float
    T_throttle_C: float
    Q_crac_kW: float
    COP_coeffs: tuple[float, float, float]   # COP(T) = c0*T^2 + c1*T + c2, T in degC
    T_crac_min_C: float
    T_crac_max_C: float
    T_max_zone_C: float
    ramp_short_lag: int = 3       # Eq. (35): 15 minutes
    ramp_short_C: float = 5.0
    ramp_long_lag: int = 12       # Eq. (36): one hour (decided, section 10)
    ramp_long_C: float = 20.0


@dataclass(frozen=True)
class ColdPlateSpec:
    """Cold-plate model agreed with the thesis group (docs/PROJECT_CONTEXT.md section 10).

    Every plate is identical and has its own pump. Per plate and period the heat removed
    is the least of: Q_max_per_plate_W, the server's heat P_IT, and
    G_k * (T[s,t-1] - coolant), where G_k = min(1/R_k, flow_k * density * c_pw) for the
    chosen flow setting k. Pump power is pump_ref_W * (flow_k / pump_ref_flow_Lmin)^3, Eq. (48).
    """
    # Wang et al. (2026), Fig. 10(a), CPU 1: R = (T_cpu,max - T_in) / P_cpu at 200 W
    flow_settings_Lmin: tuple[float, ...] = (0.10, 0.15, 0.20, 0.25, 0.30)
    R_C_per_W: tuple[float, ...] = (0.1543, 0.1143, 0.0943, 0.0843, 0.0793)
    flow_min_Lmin: float = 0.05          # Eq. (47); the table's printed 0.005 is a typo
    flow_max_Lmin: float = 0.35
    Q_max_per_plate_W: float = 300.0     # Eq. (55), 0.3 kW per cold plate
    coolant_min_C: float = 20.0          # Eq. (49), 293.15 K
    coolant_max_C: float = 39.95         # 313.1 K as printed
    pump_ref_W: float = 0.657e-3         # Wang and Tao (2025): 0.657 mW at 0.264 L/min
    pump_ref_flow_Lmin: float = 0.264
    c_pw_J_per_kgC: float = 4182.0       # printed with a "kJ" label, read as J
    water_density_kg_per_L: float = 1.0

    def validate(self) -> None:
        n = len(self.flow_settings_Lmin)
        if n == 0 or len(self.R_C_per_W) != n:
            raise ValueError("cold plate: flow_settings_Lmin and R_C_per_W must be non-empty "
                             "and of equal length")
        if any(r <= 0 for r in self.R_C_per_W):
            raise ValueError("cold plate: every R_C_per_W must be positive")
        if any(not self.flow_min_Lmin - 1e-9 <= f <= self.flow_max_Lmin + 1e-9
               for f in self.flow_settings_Lmin):
            raise ValueError(f"cold plate: flow settings {self.flow_settings_Lmin} must lie in "
                             f"[{self.flow_min_Lmin:g}, {self.flow_max_Lmin:g}] L/min, Eq. (47)")
        if self.coolant_min_C > self.coolant_max_C:
            raise ValueError("cold plate: coolant_min_C exceeds coolant_max_C")

    def conductance_W_per_C(self) -> np.ndarray:
        """Per plate and flow setting: min(1/R, water heat capacity rate), W per degC."""
        flow = np.asarray(self.flow_settings_Lmin, dtype=float)
        water = flow / 60.0 * self.water_density_kg_per_L * self.c_pw_J_per_kgC   # Eq. (53)
        return np.minimum(1.0 / np.asarray(self.R_C_per_W, dtype=float), water)

    def pump_W(self) -> np.ndarray:
        """Per plate and flow setting, Eq. (48)."""
        flow = np.asarray(self.flow_settings_Lmin, dtype=float)
        return self.pump_ref_W * (flow / self.pump_ref_flow_Lmin) ** 3

    def level_of(self, flow_Lmin: float) -> int:
        """Index of a flow setting; raises if the flow is not one of the settings."""
        for k, f in enumerate(self.flow_settings_Lmin):
            if abs(f - flow_Lmin) < 1e-9:
                return k
        raise ValueError(f"cold-plate flow {flow_Lmin:g} L/min is not one of the settings "
                         f"{self.flow_settings_Lmin}")


@dataclass(frozen=True)
class ColdPlateOps:
    """Cold-plate operating decisions, shape (S, T). NaN for servers without cold plates."""
    flow_Lmin: np.ndarray
    coolant_C: np.ndarray


@dataclass(frozen=True)
class Scenario:
    name: str
    T: int
    dt_seconds: float
    workloads: tuple[Workload, ...]   # sorted by i; workloads[i - 1].i == i
    servers: tuple[Server, ...]       # sorted by s; servers[s - 1].s == s
    n_zones: int
    CI_g_per_kWh: np.ndarray          # shape (T,)
    constants: Constants
    cold_plate: ColdPlateSpec = field(default_factory=ColdPlateSpec)
    AV: np.ndarray | None = None      # (S, T) 0/1 availability, Eq. (11); None: all 1
    fingerprint: str | None = None    # build.scenario_fingerprint of the folder it was read from
    built_with: str | None = None     # release that built the folder (params.json built_with)

    @property
    def I(self) -> int:
        return len(self.workloads)

    @property
    def S(self) -> int:
        return len(self.servers)

    @property
    def Z(self) -> int:
        return self.n_zones

    @property
    def dt_hours(self) -> float:
        return self.dt_seconds / 3600.0

    def servers_in_zone(self, z: int) -> tuple[Server, ...]:
        return tuple(sv for sv in self.servers if sv.zone == z)

    def availability(self) -> np.ndarray:
        """AV[s, t] as a (S, T) 0/1 int array, Eq. (11)."""
        if self.AV is None:
            return np.ones((self.S, self.T), dtype=int)
        return np.asarray(self.AV, dtype=int)


@dataclass
class RunSettings:
    """Every switch of a run. Written verbatim to summary.json."""
    scenario: str = "small"
    run_name: str = "default"
    # baseline
    baseline_rule: str = "least_loaded"
    seed: int = 42
    baseline_supply_temp_C: float = 18.0
    # optimized model
    reassignment: str = "free"
    placement: str = "thermal_aware"
    supply_temp_mode: str = "optimize"
    fixed_supply_temp_C: float | None = None   # None: use baseline_supply_temp_C
    H_min_C: float | None = None               # None: use the scenario constant
    time_limit_s: float = 600.0
    mip_rel_gap: float = 1e-3
    # cold plates, on by default: the servers.csv plates (the C6320s) are modelled in every run
    # (off: Q_cp and pump power are zero everywhere)
    cold_plates: bool = True
    cold_plate_count: int | None = None        # plates on every server; None: servers.csv
    baseline_cp_flow_Lmin: float = 0.30        # baseline runs its cold plates fixed at these
    baseline_cp_coolant_C: float = 20.0
    cp_mode: str = "optimize"                  # "fixed": the optimized run holds them too
    fixed_cp_flow_Lmin: float | None = None    # None: use baseline_cp_flow_Lmin
    fixed_cp_coolant_C: float | None = None    # None: use baseline_cp_coolant_C
    # priority, Eq. (56)-(57)
    priority: bool = True
    # scenario overrides (applied by io.apply_overrides)
    zone_map: tuple[int, ...] | None = None    # zone per server, replaces servers.csv
    carbon_file: str | None = None             # CSV with t, CI_t_g_per_kWh
    ci_flat: bool = False                      # CI_t held at the scenario's mean
    # slot that is not defined yet
    fan_power: bool = False
    # per-tick re-planning (docs/PLAN.md section 2): re-solve every period, keep its decisions
    per_tick: bool = False
    tick_time_limit_s: float = 60.0            # HiGHS time cap of each tick's solve
    # a tick with no schedule at tick_time_limit_s is solved again until this much time in all
    # (at or below tick_time_limit_s: no second solve)
    tick_no_schedule_limit_s: float = 300.0

    def validate(self) -> None:
        if self.baseline_rule not in BASELINE_RULES:
            raise ValueError(f"baseline_rule must be one of {BASELINE_RULES}")
        if self.reassignment not in REASSIGNMENT_MODES:
            raise ValueError(f"reassignment must be one of {REASSIGNMENT_MODES}")
        if self.placement not in PLACEMENT_MODES:
            raise ValueError(f"placement must be one of {PLACEMENT_MODES}")
        if self.supply_temp_mode not in SUPPLY_TEMP_MODES:
            raise ValueError(f"supply_temp_mode must be one of {SUPPLY_TEMP_MODES}")
        if self.cp_mode not in COLD_PLATE_MODES:
            raise ValueError(f"cp_mode must be one of {COLD_PLATE_MODES}")
        if self.carbon_file is not None and self.ci_flat:
            raise ValueError("carbon_file and ci_flat cannot both be set")
        if self.zone_map is not None and any(int(z) < 1 for z in self.zone_map):
            raise ValueError("zone_map: zones are numbered from 1")
        if self.cold_plate_count is not None:
            if not self.cold_plates:
                raise ValueError("cold_plate_count is set but cold_plates is off")
            if self.cold_plate_count < 0:
                raise ValueError("cold_plate_count must be 0 or more")
        if not self.tick_time_limit_s > 0:
            raise ValueError("tick_time_limit_s must be positive")
        if not self.tick_no_schedule_limit_s > 0:
            raise ValueError("tick_no_schedule_limit_s must be positive")
        if self.fan_power:
            raise NotDefinedYet(
                "fan_power: not defined yet. The model document has no fan-power "
                "equation; airflow is not modelled.")

    def h_min(self, scenario: Scenario) -> float:
        return scenario.constants.H_min_C if self.H_min_C is None else self.H_min_C

    def thermal_cap_C(self, scenario: Scenario) -> float:
        """Highest allowed server temperature: T_max - H_min, Eq. (31)-(33)."""
        return scenario.constants.T_max_server_C - self.h_min(scenario)

    def optimized_fixed_supply_temp_C(self) -> float:
        if self.fixed_supply_temp_C is None:
            return self.baseline_supply_temp_C
        return self.fixed_supply_temp_C

    def optimized_fixed_cp(self) -> tuple[float, float]:
        """(flow L/min, coolant degC) held by the optimized run when cp_mode is fixed."""
        flow = self.baseline_cp_flow_Lmin if self.fixed_cp_flow_Lmin is None else self.fixed_cp_flow_Lmin
        cool = (self.baseline_cp_coolant_C if self.fixed_cp_coolant_C is None
                else self.fixed_cp_coolant_C)
        return float(flow), float(cool)

    def as_dict(self) -> dict:
        return asdict(self)


def cold_plate_counts(scenario: Scenario, settings: RunSettings) -> np.ndarray:
    """Cold plates per server, shape (S,): zero everywhere when cold plates are off."""
    if not settings.cold_plates:
        return np.zeros(scenario.S, dtype=int)
    if settings.cold_plate_count is not None:
        return np.full(scenario.S, int(settings.cold_plate_count))
    return np.array([sv.plates for sv in scenario.servers], dtype=int)


def fixed_cold_plate_ops(scenario: Scenario, plates: np.ndarray, flow_Lmin: float,
                         coolant_C: float) -> ColdPlateOps:
    """One flow setting and coolant temperature for every cold plate and period."""
    scenario.cold_plate.level_of(flow_Lmin)
    has = (np.asarray(plates) > 0)[:, None]
    shape = (scenario.S, scenario.T)
    return ColdPlateOps(flow_Lmin=np.where(has, float(flow_Lmin), np.nan) * np.ones(shape),
                        coolant_C=np.where(has, float(coolant_C), np.nan) * np.ones(shape))


def priority_pairs(scenario: Scenario) -> list[tuple[int, int]]:
    """Ordered pairs (i, j) with O_ij = 1: i goes before j.

    Eq. (56): a time-sensitive i goes before a delay-tolerant j. Eq. (57): between two
    workloads of the same type and deadline, the earlier arrival goes first.

    O_ij is linked to the schedule as follows (the thesis prints no link, see
    docs/PROJECT_CONTEXT.md A6): if i and j run on the same server and j starts while i is
    waiting (A_i <= ST_j), then ST_i <= ST_j. Workloads on different servers do not
    compete, and j may start before i arrives.
    """
    out = []
    for wi in scenario.workloads:
        for wj in scenario.workloads:
            if wi.i == wj.i:
                continue
            if (wi.RT == 1 and wj.RT == 0) or (wi.RT == wj.RT and wi.D == wj.D and wi.A < wj.A):
                out.append((wi.i, wj.i))
    return out


def cop(constants: Constants, T_supply_C):
    """CRAC coefficient of performance at a supply temperature in degC (section 4.1)."""
    c0, c1, c2 = constants.COP_coeffs
    return c0 * T_supply_C ** 2 + c1 * T_supply_C + c2


@dataclass(frozen=True)
class Placement:
    """One workload's place in a schedule. ``server`` is None if it was not placed."""
    i: int
    server: int | None
    start: int | None
    end: int | None       # exclusive: start + periods run

    @property
    def periods_run(self) -> int | None:
        return None if self.start is None else self.end - self.start


@dataclass(frozen=True)
class Schedule:
    placements: tuple[Placement, ...]   # sorted by i; placements[i - 1].i == i

    def server_of(self) -> dict[int, int | None]:
        return {p.i: p.server for p in self.placements}


def constant_supply_temp(scenario: Scenario, value_C: float) -> np.ndarray:
    """Supply temperature array of shape (Z, T) at one fixed value."""
    return np.full((scenario.Z, scenario.T), float(value_C))


@dataclass
class EvalResult:
    """Everything the shared evaluator recomputes from a schedule and supply temperatures.

    Shapes: (S, T) per server per period, (Z, T) per zone per period, (T,) per period,
    (I,) per workload. Unplaced workloads have NaN in the per-workload float arrays.
    """
    # per server per period
    cpu_used_cores: np.ndarray      # sum of C_req running, Eq. (12) left side
    mem_used_MB: np.ndarray         # Eq. (13) left side
    util: np.ndarray                # Eq. (16)
    P_IT_W: np.ndarray              # Eq. (20); equals Q_heat, Eq. (21)
    Q_cp_W: np.ndarray              # cold-plate heat removed (all plates of the server)
    P_cp_W: np.ndarray              # cold-plate pump power, Eq. (48)
    T_C: np.ndarray                 # temperature recursion
    headroom_C: np.ndarray          # Eq. (31)
    DPF: np.ndarray                 # Eq. (37), all three branches
    PC_eff_MHz: np.ndarray          # Eq. (38): N_cores * f_s * DPF
    # per zone per period
    T_supply_C: np.ndarray          # as given
    Q_crac_kW: np.ndarray           # heat removed = heat balance at equality
    COP: np.ndarray
    P_crac_kW: np.ndarray           # Q_crac / COP
    P_fan_kW: np.ndarray            # zero slot; airflow not modelled
    # per period
    E_IT_kWh: np.ndarray            # Eq. (3) summed over servers
    E_crac_kWh: np.ndarray          # Eq. (4) summed over zones
    E_cp_kWh: np.ndarray            # Eq. (5) summed over servers
    E_fan_kWh: np.ndarray           # zero slot
    E_total_kWh: np.ndarray         # Eq. (2)
    carbon_g: np.ndarray            # Eq. (6)
    # per workload
    wait_periods: np.ndarray        # start - A, Eq. (24)
    work_delivered_MHz_s: np.ndarray
    work_required_MHz_s: np.ndarray
    placed: np.ndarray              # bool
    starts_on_or_after_arrival: np.ndarray   # bool, Eq. (23)
    wait_within_allowed: np.ndarray          # bool, Eq. (28)
    on_time: np.ndarray                      # bool, Eq. (27): end <= D
    work_complete: np.ndarray                # bool, Eq. (39)
    compliant: np.ndarray                    # bool: all of the above
    min_DPF_run: np.ndarray                  # lowest DPF over the periods it runs, NaN if unplaced
    # totals
    total_carbon_g: float           # Eq. (7), (8)
    totals: dict = field(default_factory=dict)       # scalar totals, see evaluate()
    compliance: dict = field(default_factory=dict)   # every check, see evaluate()
    airflow_modelled: bool = False
    n_cold_plates: np.ndarray | None = None     # (S,)
    cp_flow_Lmin: np.ndarray | None = None      # (S, T), NaN without cold plates
    cp_coolant_C: np.ndarray | None = None      # (S, T), NaN without cold plates

    @property
    def valid(self) -> bool:
        """True when the schedule breaks no limit of the model."""
        return bool(self.compliance["all_ok"])


@dataclass
class SolveResult:
    """What the optimizer returns. Numbers here are the solver's own; the comparison
    uses the evaluator's numbers for the same schedule."""
    status: str                         # one of the STATUS_* values
    schedule: Schedule | None           # None when no schedule was found
    supply_temp_C: np.ndarray | None    # shape (Z, T)
    objective_g: float | None           # solver objective, grams CO2
    best_bound_g: float | None
    mip_gap: float | None               # relative gap reported by the solver
    solve_time_s: float
    n_variables: int
    n_binaries: int
    n_constraints: int
    relaxed: tuple[str, ...] = ()       # constraint families relaxed in this solve
    solver_status_text: str = ""        # raw HiGHS model status
    notes: list[str] = field(default_factory=list)
    model_values: dict = field(default_factory=dict)   # optional solver-side T, DPF, ...
    cold_plate_ops: ColdPlateOps | None = None          # None when cold plates are off

    @property
    def has_schedule(self) -> bool:
        return self.schedule is not None


In [ ]:
%%writefile {LIB}/scheduler/io.py
"""Scenario loader: reads scenarios/<name>/*.csv and params.json into the shared types.

``availability.csv`` (columns s, t, AV) is optional: server-periods it does not list are
available, Eq. (11).
"""

from __future__ import annotations

import csv
import dataclasses
import json
from pathlib import Path

import numpy as np

from scheduler.build import scenario_fingerprint
from scheduler.types import (
    ColdPlateSpec, Constants, Placement, RunSettings, Scenario, Schedule, Server, Workload,
)


def _read_rows(path: Path) -> list[dict[str, str]]:
    if not path.is_file():
        raise ValueError(f"missing scenario file: {path}")
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def _blank(v: str | None) -> bool:
    return v is None or v.strip() == ""


def _int(v: str) -> int:
    return int(round(float(v)))


def _opt_int(v: str | None) -> int | None:
    return None if _blank(v) else _int(v)


def _check_contiguous(values: list[int], what: str, path: Path) -> None:
    expected = list(range(1, len(values) + 1))
    if values != expected:
        raise ValueError(f"{path}: column '{what}' must be 1..{len(values)} in order, "
                         f"got {values[:5]}...")


def _scenario_dir(name: str, scenarios_dir: str | Path) -> Path:
    d = Path(scenarios_dir) / name
    if not d.is_dir():
        raise ValueError(f"scenario directory not found: {d}")
    return d


def load_scenario(name: str, scenarios_dir: str | Path = "scenarios") -> Scenario:
    d = _scenario_dir(name, scenarios_dir)
    with open(d / "params.json") as f:
        params = json.load(f)

    T = int(params["time"]["T"])
    dt = float(params["time"]["dt_seconds"])

    # servers
    srv_path = d / "servers.csv"
    srv_rows = _read_rows(srv_path)
    if not srv_rows:
        raise ValueError(f"{srv_path}: no servers")
    _check_contiguous([_int(r["s"]) for r in srv_rows], "s", srv_path)
    servers = tuple(
        Server(
            s=_int(r["s"]), name=r["server"], zone=_int(r["zone"]),
            f_MHz=float(r["f_s_MHz"]), C_cores=_int(r["C_s_cores"]), M_MB=float(r["M_s_MB"]),
            P_idle_W=float(r["P_idle_W"]), P_max_W=float(r["P_max_W"]), T0_C=float(r["T0_C"]),
            CT_cold_plate=_int(r["CT_s_cold_plate"]), th_a_C=float(r["th_a_C"]),
            th_R_C_per_W=float(r["th_R_C_per_W"]), th_alpha=float(r["th_alpha_5min"]),
            n_cold_plates=_opt_int(r.get("n_cold_plates")),
        )
        for r in srv_rows
    )
    for sv in servers:
        if sv.plates < 0:
            raise ValueError(f"{srv_path}: server {sv.s} has {sv.plates} cold plates")
    n_zones = max(sv.zone for sv in servers)
    for sv in servers:
        if not 1 <= sv.zone <= n_zones:
            raise ValueError(f"{srv_path}: server {sv.s} has zone {sv.zone}, "
                             f"outside 1..{n_zones}")

    # workloads
    wl_path = d / "workloads.csv"
    wl_rows = _read_rows(wl_path)
    _check_contiguous([_int(r["i"]) for r in wl_rows], "i", wl_path)
    workloads = tuple(
        Workload(
            i=_int(r["i"]), workload_id=r["workload_id"], flavor=r["flavor"],
            A=_int(r["A_i"]), D=_int(r["D_i"]), AD=_int(r["AD_i"]),
            ET_base=_int(r["ET_base_i"]), RT=_int(r["RT_i"]), B=_int(r["B_i"]),
            X0=_opt_int(r["X0_s"]), C_req=_int(r["C_req_i"]),
            M_req_MB=float(r["M_req_i_MB"]), W_MHz_s=float(r["W_i_MHz_s"]),
            usage_imputed=_int(r["usage_imputed"]) if not _blank(r.get("usage_imputed")) else 0,
        )
        for r in wl_rows
    )

    # carbon
    ci = _read_carbon(d / "carbon.csv", T)

    # availability, Eq. (11)
    av_path = d / "availability.csv"
    AV = _read_availability(av_path, len(servers), T) if av_path.is_file() else None

    # constants
    c = params["constants_as_printed"]
    constants = Constants(
        R_dvfs=c["R_dvfs"], F_throttle=c["F_throttle"], T_max_server_C=c["T_max_server_C"],
        H_min_C=c["H_min_C"], T_dvfs_C=c["T_dvfs_C"], T_throttle_C=c["T_throttle_C"],
        Q_crac_kW=c["Q_crac_kW"], COP_coeffs=tuple(float(x) for x in c["COP_crac_coeffs"]),
        T_crac_min_C=c["T_crac_min_C"], T_crac_max_C=c["T_crac_max_C"],
        T_max_zone_C=c["T_max_zone_C"],
    )
    if len(constants.COP_coeffs) != 3:
        raise ValueError(f"{d / 'params.json'}: COP_crac_coeffs must have 3 entries")

    cold_plate = _cold_plate_spec(c, params.get("cold_plate", {}))
    cold_plate.validate()

    sc = Scenario(name=name, T=T, dt_seconds=dt, workloads=workloads, servers=servers,
                  n_zones=n_zones, CI_g_per_kWh=ci, constants=constants, cold_plate=cold_plate,
                  AV=AV, fingerprint=scenario_fingerprint(d),
                  built_with=params.get("built_with", {}).get("thermasched"))
    settings_dt = params.get("settings", {}).get("dt_seconds")
    if settings_dt is not None and float(settings_dt) != dt:
        raise ValueError(f"{d / 'params.json'}: time.dt_seconds={dt} does not match "
                         f"settings.dt_seconds={settings_dt}")
    return sc


def _read_carbon(path: Path, T: int) -> np.ndarray:
    rows = _read_rows(path)
    if len(rows) != T:
        raise ValueError(f"{path}: has {len(rows)} rows, the scenario has T={T}")
    _check_contiguous([_int(r["t"]) for r in rows], "t", path)
    return np.array([float(r["CI_t_g_per_kWh"]) for r in rows], dtype=float)


def _read_availability(path: Path, S: int, T: int) -> np.ndarray:
    AV = np.ones((S, T), dtype=int)
    for r in _read_rows(path):
        s, t, v = _int(r["s"]), _int(r["t"]), _int(r["AV"])
        if not (1 <= s <= S and 1 <= t <= T) or v not in (0, 1):
            raise ValueError(f"{path}: row s={s}, t={t}, AV={v} is outside 1..{S}, 1..{T}, {{0, 1}}")
        AV[s - 1, t - 1] = v
    return AV


def apply_overrides(scenario: Scenario, settings: RunSettings) -> Scenario:
    """Scenario switches of a run: zone map (S1 T1) and carbon intensity (S4 T3)."""
    sc = scenario
    if settings.zone_map is not None:
        zones = [int(z) for z in settings.zone_map]
        if len(zones) != sc.S:
            raise ValueError(f"zone_map has {len(zones)} entries, the scenario has {sc.S} servers")
        n_zones = max(zones)
        if sorted(set(zones)) != list(range(1, n_zones + 1)):
            raise ValueError(f"zone_map {zones}: every zone 1..{n_zones} must have a server")
        servers = tuple(dataclasses.replace(sv, zone=z) for sv, z in zip(sc.servers, zones))
        sc = dataclasses.replace(sc, servers=servers, n_zones=n_zones)
    if settings.carbon_file is not None:
        sc = dataclasses.replace(sc, CI_g_per_kWh=_read_carbon(Path(settings.carbon_file), sc.T))
    elif settings.ci_flat:
        sc = dataclasses.replace(sc, CI_g_per_kWh=np.full(sc.T, float(np.mean(sc.CI_g_per_kWh))))
    return sc


def _cold_plate_spec(c: dict, cp: dict) -> ColdPlateSpec:
    """Printed constants (kelvin and mW converted) plus the params.json "cold_plate" block.
    Anything missing keeps the ColdPlateSpec default."""
    kw: dict = {}
    if "Q_cp_kW" in c:
        kw["Q_max_per_plate_W"] = 1000.0 * float(c["Q_cp_kW"])
    if "T_cp_min_K" in c:
        kw["coolant_min_C"] = round(float(c["T_cp_min_K"]) - 273.15, 6)
    if "T_cp_max_K" in c:
        kw["coolant_max_C"] = round(float(c["T_cp_max_K"]) - 273.15, 6)
    if "FR_cp_min_Lmin" in c:
        kw["flow_min_Lmin"] = float(c["FR_cp_min_Lmin"])
    if "FR_cp_max_Lmin" in c:
        kw["flow_max_Lmin"] = float(c["FR_cp_max_Lmin"])
    if "P_cp_rated_as_printed_mW" in c:
        kw["pump_ref_W"] = float(c["P_cp_rated_as_printed_mW"]) / 1000.0
    if "FR_cp_rated_Lmin" in c:
        kw["pump_ref_flow_Lmin"] = float(c["FR_cp_rated_Lmin"])
    if "c_pw" in c:
        kw["c_pw_J_per_kgC"] = float(c["c_pw"])
    for key in ("flow_settings_Lmin", "R_C_per_W"):
        if key in cp:
            kw[key] = tuple(float(x) for x in cp[key])
    if "water_density_kg_per_L" in cp:
        kw["water_density_kg_per_L"] = float(cp["water_density_kg_per_L"])
    return ColdPlateSpec(**kw)


def load_reference_baseline(name: str, scenarios_dir: str | Path = "scenarios") -> Schedule:
    """Read scenarios/<name>/baseline_schedule.csv (columns i, s, start, end, late)."""
    path = _scenario_dir(name, scenarios_dir) / "baseline_schedule.csv"
    rows = _read_rows(path)
    _check_contiguous([_int(r["i"]) for r in rows], "i", path)
    placements = tuple(
        Placement(i=_int(r["i"]), server=_opt_int(r["s"]), start=_opt_int(r["start"]),
                  end=_opt_int(r["end"]))
        for r in rows
    )
    return Schedule(placements)


In [ ]:
%%writefile {LIB}/scheduler/baseline.py
"""Greedy, thermally blind baseline scheduler.

Workloads are taken in arrival order (the order of ``scenario.workloads``). Each one starts
at the earliest period at or after its arrival where some server has room for its cores
and memory (Eq. 12-13) for the whole run and is available (AV = 1, Eq. 11) throughout. Only
cores, memory and availability are checked: no temperature or deadline. A workload that fits
nowhere gets ``Placement(i, None, None, None)``.
"""

from __future__ import annotations

import math
import random

from scheduler.types import BASELINE_RULES, Placement, RunSettings, Scenario, Schedule


def periods_needed(workload, server, dt_seconds: float) -> int:
    """Run length of ``workload`` on ``server`` in periods: ceil(W / (C_req * f * dt)), at least 1."""
    return max(1, math.ceil(workload.W_MHz_s / (workload.C_req * server.f_MHz * dt_seconds)))


def build_baseline(scenario: Scenario, settings: RunSettings) -> Schedule:
    rule = settings.baseline_rule
    if rule not in BASELINE_RULES:
        raise ValueError(f"baseline_rule must be one of {BASELINE_RULES}, got {rule!r}")

    T, dt = scenario.T, scenario.dt_seconds
    servers = scenario.servers
    # used[s - 1][t] = cores in use in period t (index 0 unused, periods 1..T)
    used = [[0] * (T + 2) for _ in servers]
    mem = [[0.0] * (T + 2) for _ in servers]    # MB in use, the same way
    AV = scenario.availability()
    rng = random.Random(settings.seed)
    by_speed = sorted(servers, key=lambda sv: (-sv.f_MHz, sv.s))

    def has_room(sv, w, start: int) -> int | None:
        """Return the end (exclusive) of the run if ``w`` fits on ``sv`` from ``start``."""
        end = start + periods_needed(w, sv, dt)
        if end - 1 > T:
            return None
        u, m, av = used[sv.s - 1], mem[sv.s - 1], AV[sv.s - 1]
        if all(u[t] + w.C_req <= sv.C_cores and m[t] + w.M_req_MB <= sv.M_MB and av[t - 1]
               for t in range(start, end)):
            return end
        return None

    placements: list[Placement] = []
    for w in scenario.workloads:
        chosen = None  # (server, start, end)
        for start in range(max(1, w.A), T + 1):
            if rule == "least_loaded":
                order = sorted(servers, key=lambda sv: (used[sv.s - 1][start], sv.s))
            elif rule == "fastest_first":
                order = by_speed
            else:  # random: pick uniformly among servers with room
                order = servers
            fits = []
            for sv in order:
                end = has_room(sv, w, start)
                if end is not None:
                    fits.append((sv, end))
                    if rule != "random":
                        break
            if fits:
                sv, end = rng.choice(fits) if rule == "random" else fits[0]
                chosen = (sv, start, end)
                break
        if chosen is None:
            placements.append(Placement(w.i, None, None, None))
            continue
        sv, start, end = chosen
        u, m = used[sv.s - 1], mem[sv.s - 1]
        for t in range(start, end):
            u[t] += w.C_req
            m[t] += w.M_req_MB
        placements.append(Placement(w.i, sv.s, start, end))
    return Schedule(tuple(placements))


In [ ]:
%%writefile {LIB}/scheduler/build.py
"""Scenario builder: turns the cleaned tables in data/clean/ into a scenario folder.

``build_scenario`` writes servers.csv, workloads.csv, carbon.csv, baseline_schedule.csv and
params.json from a dict of scenario settings, the thesis constants and the cold-plate table.
``notebooks/data_prep.ipynb`` calls it for the scenarios of its Settings cell: the 6- and
10-server fleets on a light, normal and heavy day, and ``small``.

The settings' ``fleet`` table has one row ``{model, f_MHz, zone, cold_plate}`` per server, in
server order; the number of servers and zones follow from it. Each row gets a host of its
model (R840 or C6320), drawn by the settings' ``seed`` from the eligible hosts of the cleaned
table without regard to power (``draw_hosts``).

``derive_scenario`` builds a new scenario from an existing one's params.json with some values
changed, for ``notebooks/solver.ipynb``. It reads only data/clean/, not the raw datasets.

Both notebooks hold a copy of this file, so that each runs on its own (on Colab, say);
after editing it, run ``uv run python tools/sync_notebooks.py``.
"""

from __future__ import annotations

import csv
import hashlib
import json
import math
import random
import shutil
from pathlib import Path

from scheduler.version import VERSION


def read_csv_rows(path):
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def write_csv(path, records, fields=None):
    path.parent.mkdir(parents=True, exist_ok=True)
    fields = fields or list(records[0].keys())
    with open(path, "w", newline="") as f:
        w = csv.DictWriter(f, fieldnames=fields, extrasaction="ignore")
        w.writeheader()
        w.writerows(records)


# Server models of the cleaned table (column `model`) and the processor of each frequency
# tier, from the SPEC CPU results for those models: the frequencies are assumptions.
MODELS = ("R840", "C6320")
CPU_BY_MHZ = {2100: "Xeon Gold 6130", 2400: "Xeon Gold 6148", 2600: "Xeon E5-2690 v4"}
FLEET_ROW_KEYS = ("model", "f_MHz", "zone", "cold_plate")
# Columns of the cleaned servers.csv that a scenario's servers.csv takes over (as numbers).
HOST_VALUES = ("N_cores", "C_s_cores", "M_s_MB", "P_idle_W", "P_max_W", "P_fit_full_load_W", "T0_C",
               "th_a_C", "th_R_C_per_W", "th_alpha_5min")


def fleet_size(sc):
    """(n_servers, n_zones) of the settings' fleet table."""
    return len(sc["fleet"]), max(int(r["zone"]) for r in sc["fleet"])


def eligible_hosts(clean_dir):
    rows = read_csv_rows(Path(clean_dir) / "servers.csv")
    if rows and "model" not in rows[0]:
        raise ValueError("the cleaned servers.csv has no `model` column: clean it again to build a fleet")
    return [s for s in rows if s["eligible"] == "1"]


def host_draw_key(seed, host):
    # Seeded hash of the host name: a host's place in the draw depends only on the seed and
    # its own name, not on power, on the table's row order or on which other hosts are in it.
    return hashlib.sha256(f"{seed}:host:{host}".encode()).hexdigest()


def draw_hosts(sc, pool):
    """One host per fleet row, in server order. The eligible hosts of each model are put in
    seeded order (host_draw_key) and handed out in that order to the model's rows, in fleet
    order, so no host is drawn twice."""
    queue = {m: sorted((s for s in pool if s["model"] == m), key=lambda s: host_draw_key(sc["seed"], s["server"]))
             for m in MODELS}
    return [queue[r["model"]].pop(0) for r in sc["fleet"]]


def pick_servers(sc, clean_dir):
    out = []
    for i, (row, host) in enumerate(zip(sc["fleet"], draw_hosts(sc, eligible_hosts(clean_dir)))):
        s = {k: float(host[k]) for k in HOST_VALUES}
        f, cp = float(row["f_MHz"]), int(row["cold_plate"])
        out.append({
            "s": i + 1, "server": host["server"], "model": host["model"], "cpu": CPU_BY_MHZ.get(round(f), ""),
            "zone": int(row["zone"]),
            "f_s_MHz": f, "N_cores": int(s["N_cores"]), "C_s_cores": int(s["C_s_cores"]),
            "M_s_MB": round(s["M_s_MB"]), "PC_s_MHz": s["N_cores"] * f,
            "P_idle_W": round(s["P_idle_W"], 2), "P_max_W": round(s["P_max_W"], 2),
            "P_fit_full_load_W": round(s["P_fit_full_load_W"], 2),
            "T0_C": round(s["T0_C"], 2), "CT_s_cold_plate": cp,
            "n_cold_plates": cp,   # editable: plates on the server (0 = none)
            "th_a_C": round(s["th_a_C"], 3), "th_R_C_per_W": round(s["th_R_C_per_W"], 5),
            "th_alpha_5min": round(s["th_alpha_5min"], 5),
            # dynamic watts drawn per MHz of work delivered: lower is more efficient
            "W_per_MHz": round((s["P_max_W"] - s["P_idle_W"]) / (s["N_cores"] * f), 6),
        })
    return out


def is_time_sensitive(vm_id, seed, share):
    # Seeded hash of the VM id, uniform on [0, 1): depends only on this workload, so the
    # flag does not change when other workloads enter or leave the catalogue. Keyed on
    # the UFCG id because workload_id is a sort position in the catalogue and shifts with it.
    h = hashlib.sha256(f"{seed}:{vm_id}".encode()).digest()
    return int(int.from_bytes(h[:8], "big") / 2 ** 64 < share)


def pick_workloads(sc, fleet_cores, fleet_mem_mb, clean_dir):
    # Draw in seeded random order until peak requested vCPUs reach target_peak_load of the
    # fleet's cores, or peak requested memory that share of its memory, whichever comes first.
    # With top_up_by_repeats, a catalogue that runs out first is drawn again in the same
    # order: each copy keeps the workload's arrival time, work and flags, and its ID gets
    # "-r1", "-r2", ... (column `repeat`).
    T, rng = sc["periods"], random.Random(sc["seed"])
    per_day = 86400 // sc["dt_seconds"]
    start = sc["start_hour_local"] * 3600 // sc["dt_seconds"]
    cat = [w for w in read_csv_rows(Path(clean_dir) / "workloads.csv")
           if sc["min_et_periods"] <= int(w["ET_base_periods"]) <= sc["max_et_periods"]]
    rng.shuffle(cat)

    demand, mem, picked = [0] * (T + 2), [0] * (T + 2), []

    def full():
        return (max(demand) >= sc["target_peak_load"] * fleet_cores
                or max(mem) >= sc["target_peak_load"] * fleet_mem_mb)

    repeat = 0
    while True:
        n_before = len(picked)
        for w in cat:
            et, vcpu, mb = int(w["ET_base_periods"]), int(w["C_req_vcpu"]), int(w["M_req_MB"])
            a = (int(w["arrival_slot_of_day"]) - start) % per_day + 1
            rt = is_time_sensitive(w["ufcg_server_id"], sc["seed"], sc["rt_share"])
            slack = max(1, math.ceil(sc["rt_slack_frac"] * et)) if rt else \
                min(sc["dt_slack_max"], max(sc["dt_slack_min"], math.ceil(sc["dt_slack_mult"] * et)))
            d = min(a + et + slack, T + 1)          # D_i: first period by which it must be done
            if d - a - et < 1:
                continue                            # cannot finish inside the horizon
            for t in range(a, a + et):
                demand[t] += vcpu
                mem[t] += mb
            picked.append({
                "workload_id": w["workload_id"] + (f"-r{repeat}" if repeat else ""),
                "flavor": w["flavor_name"],
                "A_i": a, "D_i": d, "AD_i": d - a - et, "ET_base_i": et, "RT_i": rt, "B_i": 0,
                "C_req_i": vcpu, "M_req_i_MB": mb, "W_i_MHz_s": float(w["W_MHz_s"]),
                "usage_imputed": int(w["usage_imputed"]), "repeat": repeat,
            })
            if full():
                break
        if full() or len(picked) == n_before or not sc.get("top_up_by_repeats", False):
            break
        repeat += 1
    if not picked:
        raise ValueError("no workload fits: none can finish inside the horizon with these settings")
    picked.sort(key=lambda w: (w["A_i"], w["workload_id"]))
    for i, w in enumerate(picked, 1):
        w["i"] = i
    return picked, max(demand), max(mem), len(cat)


def carbon_profile(sc):
    # Annual mean with a synthetic daily shape: a midday dip and an evening peak.
    T, dt = sc["periods"], sc["dt_seconds"]
    hours = [(sc["start_hour_local"] + (t + 0.5) * dt / 3600) % 24 for t in range(T)]
    bump = lambda h, mu, sd: math.exp(-0.5 * (min(abs(h - mu), 24 - abs(h - mu)) / sd) ** 2)
    day = [bump(h / 12, 19, 2.0) - bump(h / 12, 12, 3.0) for h in range(288)]
    mean, span = sum(day) / 288, max(abs(x - sum(day) / 288) for x in day)
    shape = lambda h: (bump(h, 19, 2.0) - bump(h, 12, 3.0) - mean) / span
    return [{"t": t + 1, "hour_local": round(h, 3),
             "CI_t_g_per_kWh": round(sc["ci_mean_g_per_kwh"] * (1 + sc["ci_amplitude"] * shape(h)), 2)}
            for t, h in enumerate(hours)]


def baseline_schedule(servers, workloads, sc, unavailable=frozenset()):
    # Thermally unaware reference: each workload starts at arrival on the least-loaded
    # server that has room for its cores and memory, Eq. (12)-(13). `unavailable` holds
    # (server, period) pairs with AV = 0, Eq. (11).
    T, dt = sc["periods"], sc["dt_seconds"]
    used = {s["s"]: [0] * (T + 2) for s in servers}
    mem = {s["s"]: [0] * (T + 2) for s in servers}
    sched = []
    for w in workloads:
        placed = None
        for start in range(w["A_i"], T + 1):
            for s in sorted(servers, key=lambda s: (used[s["s"]][start], s["s"])):
                et = math.ceil(w["W_i_MHz_s"] / (w["C_req_i"] * s["f_s_MHz"] * dt))
                span = range(start, start + max(1, et))
                if span[-1] <= T and all(used[s["s"]][t] + w["C_req_i"] <= s["C_s_cores"]
                                         and mem[s["s"]][t] + w["M_req_i_MB"] <= s["M_s_MB"]
                                         and (s["s"], t) not in unavailable for t in span):
                    placed = (s, span)
                    break
            if placed:
                break
        if not placed:
            sched.append({"i": w["i"], "s": "", "start": "", "end": "", "late": 1})
            continue
        s, span = placed
        for t in span:
            used[s["s"]][t] += w["C_req_i"]
            mem[s["s"]][t] += w["M_req_i_MB"]
        w["X0_s"] = s["s"]
        sched.append({"i": w["i"], "s": s["s"], "start": span[0], "end": span[-1] + 1,
                      "late": int(span[-1] + 1 > w["D_i"])})
    return sched, used


def check_settings(sc, hosts=None):
    """The scenario settings must describe a fleet the builder can make. ``hosts``: the
    eligible hosts of the cleaned table, to check there are enough of each model."""
    old = sorted({"freq_mhz", "freq_split", "n_servers", "n_zones"} & set(sc))
    if old:
        raise ValueError(f"settings {old} are replaced by the fleet table: one row "
                         f"{{model, f_MHz, zone, cold_plate}} per server; n_servers and n_zones come from it")
    fleet = sc.get("fleet")
    if not isinstance(fleet, list) or not fleet:
        raise ValueError("fleet must be a non-empty list, one row {model, f_MHz, zone, cold_plate} per server")
    for i, r in enumerate(fleet, 1):
        if not isinstance(r, dict) or sorted(r) != sorted(FLEET_ROW_KEYS):
            raise ValueError(f"fleet row {i} must have exactly the keys {list(FLEET_ROW_KEYS)}, got {r}")
        if r["model"] not in MODELS:
            raise ValueError(f"fleet row {i}: model '{r['model']}' is not one of {list(MODELS)}")
        if not (isinstance(r["f_MHz"], (int, float)) and r["f_MHz"] > 0):
            raise ValueError(f"fleet row {i}: f_MHz = {r['f_MHz']} must be a positive number")
        if type(r["zone"]) is not int or r["zone"] < 1:
            raise ValueError(f"fleet row {i}: zone = {r['zone']} must be a whole number of at least 1")
        if type(r["cold_plate"]) is not int or r["cold_plate"] not in (0, 1):
            raise ValueError(f"fleet row {i}: cold_plate = {r['cold_plate']} must be 0 or 1")
    zones = sorted({r["zone"] for r in fleet})
    if zones != list(range(1, len(zones) + 1)):
        raise ValueError(f"fleet zones {zones}: every zone from 1 to {zones[-1]} must have a server")
    if hosts is not None:
        for m in MODELS:
            want, have = sum(r["model"] == m for r in fleet), sum(s["model"] == m for s in hosts)
            if want > have:
                raise ValueError(f"the fleet has {want} {m} servers, but only {have} {m} hosts "
                                 f"in the cleaned table are eligible")
    if sc["periods"] < 1 or sc["min_et_periods"] > sc["max_et_periods"]:
        raise ValueError("periods must be at least 1 and min_et_periods at most max_et_periods")


def assumed_server_values(servers):
    """The server values that are assumptions rather than UniMelb measurements."""
    tiers = sorted({(s["model"], s["f_s_MHz"], s["cpu"]) for s in servers}, key=lambda t: (t[1], t[0]))
    out = ["f_s_MHz: the frequency of a SPEC CPU result for the server model (" + "; ".join(
        f"{m} {f:g} MHz, {cpu or 'no SPEC processor'}" for m, f, cpu in tiers) + ")"]
    if any(s["model"] == "C6320" for s in servers):
        out += ["N_cores, C_s_cores of C6320: the hardware threads UniMelb reports before the "
                "cloud doubled its vCPU slots in March 2019",
                "inlet of C6320 (thermal fit): UniMelb has none; the R840 room's mean measured "
                "inlet at the same timestamp"]
    return out


def build_scenario(name, sc, const, cold_plate, *, clean_dir, scenarios_dir, f_ref_mhz,
                   slot_seconds=300, extra=None):
    """Write scenarios_dir/name/ from scenario settings ``sc``, the constants and the
    cold-plate table. ``extra`` is added to params.json. Returns (servers, diagnostics)."""
    assert sc["dt_seconds"] == slot_seconds, "period length must match the UFCG slot"
    check_settings(sc, eligible_hosts(clean_dir))
    out = Path(scenarios_dir) / name
    T, dt = sc["periods"], sc["dt_seconds"]

    servers = pick_servers(sc, clean_dir)
    fleet = sum(s["C_s_cores"] for s in servers)
    fleet_mem = sum(s["M_s_MB"] for s in servers)
    workloads, peak, peak_mem, n_cat = pick_workloads(sc, fleet, fleet_mem, clean_dir)
    carbon = carbon_profile(sc)
    sched, used = baseline_schedule(servers, workloads, sc)

    # Baseline IT energy and carbon by Eq. (16), (20), (3), (6); cooling is not included.
    ci = [c["CI_t_g_per_kWh"] for c in carbon]
    kwh = [sum(s["P_idle_W"] + (s["P_max_W"] - s["P_idle_W"]) * used[s["s"]][t] / s["C_s_cores"]
               for s in servers) * dt / 3.6e6 for t in range(1, T + 1)]
    cap_T = const["T_max_server_C"] - const["H_min_C"]

    def u_ceiling(s, inlet):
        # Highest steady utilization that keeps the server at or under T_max - H_min.
        p = (cap_T - inlet - s["th_a_C"]) / s["th_R_C_per_W"]
        return round(max(0.0, min(1.0, (p - s["P_idle_W"]) / (s["P_max_W"] - s["P_idle_W"]))), 3)

    for s in servers:
        s["U_ceiling_inlet_min"] = u_ceiling(s, const["T_crac_min_C"])
        s["U_ceiling_inlet_max"] = u_ceiling(s, const["T_crac_max_C"])

    write_csv(out / "servers.csv", servers)
    write_csv(out / "workloads.csv", workloads, ["i", "workload_id", "flavor", "A_i", "D_i", "AD_i", "ET_base_i",
                                                 "RT_i", "B_i", "X0_s", "C_req_i", "M_req_i_MB", "W_i_MHz_s",
                                                 "usage_imputed", "repeat"])
    write_csv(out / "carbon.csv", carbon)
    write_csv(out / "baseline_schedule.csv", sched)

    top_f = max(float(r["f_MHz"]) for r in sc["fleet"])
    top_cores = sum(s["C_s_cores"] for s in servers if s["f_s_MHz"] == top_f)
    diag = {
        "I_workloads": len(workloads), "S_servers": len(servers), "Z_zones": fleet_size(sc)[1], "T_periods": T,
        # Y_ist can only be 1 between arrival and deadline, so this is the real count
        "Y_ist_binaries_inside_windows": sum(w["D_i"] - w["A_i"] for w in workloads) * len(servers),
        "catalogue_size_within_ET_limits": n_cat,
        "fleet_cores": fleet,
        "peak_requested_vcpus": peak,
        "peak_load_share_of_fleet": round(peak / fleet, 3),
        "peak_load_share_of_fastest_tier": round(peak / top_cores, 3),
        "fleet_memory_MB": fleet_mem,
        "peak_requested_memory_MB": peak_mem,
        "peak_memory_share_of_fleet": round(peak_mem / fleet_mem, 3),
        "time_sensitive_workloads": sum(w["RT_i"] for w in workloads),
        "workloads_with_imputed_usage": sum(w["usage_imputed"] for w in workloads),
        "repeated_workloads": sum(w["repeat"] > 0 for w in workloads),
        "median_slack_periods_delay_tolerant": sorted(w["AD_i"] for w in workloads if not w["RT_i"])[
            sum(1 for w in workloads if not w["RT_i"]) // 2] if any(not w["RT_i"] for w in workloads) else None,
        "W_per_MHz_spread_best_to_worst": round(max(s["W_per_MHz"] for s in servers) / min(s["W_per_MHz"] for s in servers), 2),
        "servers_thermally_capped_at_coldest_inlet": sum(s["U_ceiling_inlet_min"] < 1 for s in servers),
        "servers_thermally_capped_at_warmest_inlet": sum(s["U_ceiling_inlet_max"] < 1 for s in servers),
        "CI_min_max_g_per_kWh": [min(ci), max(ci)],
        "baseline_IT_energy_kWh": round(sum(kwh), 2),
        # idle draw is paid whatever the schedule is: the model has no server on/off variable
        "baseline_idle_share_of_IT_energy": round(sum(s["P_idle_W"] for s in servers) * T * dt / 3.6e6 / sum(kwh), 3),
        "baseline_IT_carbon_kg": round(sum(k * c for k, c in zip(kwh, ci)) / 1000, 2),
        "baseline_unplaced_or_late": sum(x["late"] for x in sched),
    }
    params = {
        "scenario": name, "settings": sc, "constants_as_printed": const,
        "cold_plate": cold_plate,
        "time": {"dt_seconds": dt, "T": T, "period_1_local_hour": sc["start_hour_local"],
                 "convention": "A_i = first period it may run; D_i = period by which it must be finished "
                               "(CT_i = ST_i + ET_i <= D_i); AD_i = D_i - A_i - ET_base_i, Eq. (22) as an equality"},
        "work_rule": "ET_is = W_i / (C_req_i * f_s * DPF * dt): a workload is served by its own requested "
                     "cores, not the whole server. ET_base_i uses f_ref.",
        "f_ref_MHz": f_ref_mhz,
        "thermal_fit": "steady state T_cpu = T_inlet + th_a_C + th_R_C_per_W * P_IT; "
                       "per period T[t] = alpha*T[t-1] + (1-alpha)*steady, alpha = th_alpha_5min. "
                       "Fitted from UniMelb; not part of the thesis formulation.",
        "assumed_not_from_data": ["RT_i", "D_i", "zone", "CT_s_cold_plate", "n_cold_plates",
                                  *assumed_server_values(servers),
                                  "daily shape of CI_t", "AV_st = 1 for all s, t", "B_i = 0"],
        "diagnostics": diag,
        **(extra or {}),
        "built_with": {"thermasched": VERSION},
    }
    with open(out / "params.json", "w") as f:
        json.dump(params, f, indent=2)
    return servers, diag


def build_running_variant(name, start_from, running_arrivals, outage, *, scenarios_dir):
    """Write scenarios_dir/name/ as scenario ``start_from`` with some workloads already running
    (B_i = 1 for those arriving in periods ``running_arrivals`` = [first, last], Eq. 29) and
    one server down (``outage`` = {"server": s, "periods": [first, last]}, AV = 0, Eq. 11).
    The baseline schedule is rebuilt by the same least-loaded rule, skipping the downtime."""
    src, out = Path(scenarios_dir) / start_from, Path(scenarios_dir) / name
    with open(src / "params.json") as f:
        params = json.load(f)
    T, dt = params["time"]["T"], params["time"]["dt_seconds"]
    server_rows = read_csv_rows(src / "servers.csv")
    rows = read_csv_rows(src / "workloads.csv")
    servers = [{"s": int(r["s"]), "f_s_MHz": float(r["f_s_MHz"]), "C_s_cores": int(r["C_s_cores"]),
                "M_s_MB": int(r["M_s_MB"])} for r in server_rows]
    workloads = [{"i": int(r["i"]), "A_i": int(r["A_i"]), "D_i": int(r["D_i"]),
                  "C_req_i": int(r["C_req_i"]), "M_req_i_MB": int(r["M_req_i_MB"]),
                  "W_i_MHz_s": float(r["W_i_MHz_s"])} for r in rows]

    first, last = running_arrivals
    running = [r for r in rows if first <= int(r["A_i"]) <= last]
    for r in running:
        r["B_i"] = 1
    down_s, (t0, t1) = outage["server"], outage["periods"]
    if not 1 <= down_s <= len(servers) or not 1 <= t0 <= t1 <= T:
        raise ValueError(f"outage {outage} is outside the {len(servers)} servers or {T} periods")
    down = {(down_s, t) for t in range(t0, t1 + 1)}
    sched, _ = baseline_schedule(servers, workloads, {"periods": T, "dt_seconds": dt}, down)
    for r, w in zip(rows, workloads):
        r["X0_s"] = w.get("X0_s", "")

    out.mkdir(parents=True, exist_ok=True)
    for fname in ("servers.csv", "carbon.csv"):
        shutil.copyfile(src / fname, out / fname)
    write_csv(out / "workloads.csv", rows)
    write_csv(out / "baseline_schedule.csv", sched)
    write_csv(out / "availability.csv", [{"s": s, "t": t, "AV": 0} for s, t in sorted(down)])

    params.pop("variant_of", None)
    params.pop("built_with", None)
    params["scenario"] = name
    params["assumed_not_from_data"] = [
        {"AV_st = 1 for all s, t": "AV_st (availability.csv)", "B_i = 0": "B_i (first arrivals)"}.get(a, a)
        for a in params["assumed_not_from_data"]]
    params["derived_from"] = {
        "scenario": start_from,
        "changes": [
            f"B_i = 1 for the {len(running)} workloads arriving in periods {first}-{last} (already "
            f"running at the start, so they keep their server, Eq. 29)",
            f"availability.csv: server {down_s} has AV = 0 in periods {t0}-{t1} "
            f"({(t1 - t0 + 1) * dt / 3600:g} h of maintenance, Eq. 11)",
            "baseline_schedule.csv and X0_s rebuilt with the least-loaded rule, which skips "
            "unavailable periods",
        ],
        "purpose": "exercise Eq. (11) and (29), which are never binding in the other scenarios",
    }
    params["built_with"] = {"thermasched": VERSION}
    with open(out / "params.json", "w") as f:
        json.dump(params, f, indent=2)
    return sched


# A scenario's inputs, in the order they enter its fingerprint.
FINGERPRINT_FILES = ("params.json", "servers.csv", "workloads.csv", "carbon.csv",
                     "baseline_schedule.csv", "availability.csv")


def scenario_fingerprint(folder):
    """Short hash of a scenario folder's inputs: equal fingerprints mean the same scenario.
    params.json is compared by content without ``built_with``, and line endings are
    ignored, so a rebuild with a newer release or on another system still matches.
    None if the folder has no params.json."""
    folder = Path(folder)
    if not (folder / "params.json").is_file():
        return None
    h = hashlib.sha256()
    for fname in FINGERPRINT_FILES:
        path = folder / fname
        if not path.is_file():
            continue
        if fname == "params.json":
            params = json.loads(path.read_text())
            params.pop("built_with", None)
            data = json.dumps(params, sort_keys=True).encode()
        else:
            data = path.read_bytes().replace(b"\r\n", b"\n")
        h.update(fname.encode() + b"\0" + data + b"\0")
    return h.hexdigest()[:12]


# --- Only notebooks/solver.ipynb uses what follows: new scenarios from existing ones.
# (tools/sync_notebooks.py copies the code above this line into notebooks/data_prep.ipynb.)

# Written by data_prep.ipynb from its Settings cell.
# derive_scenario never overwrites them.
SHIPPED_SCENARIOS = ("servers10_light", "servers10", "servers10_heavy", "servers10_light_swing",
                     "servers10_swing", "servers10_heavy_swing",
                     "servers6_light", "servers6", "servers6_heavy", "servers6_light_swing",
                     "servers6_swing", "servers6_heavy_swing", "small", "small_running")


def _changed(base: dict, changes: dict | None, what: str) -> dict:
    changes = dict(changes or {})
    unknown = sorted(set(changes) - set(base))
    if unknown:
        raise ValueError(f"unknown {what}: {unknown}. Known: {sorted(base)}")
    return {k: v for k, v in changes.items() if v != base[k]}


def derive_scenario(name, start_from, settings=None, constants=None, cold_plate=None, *,
                    scenarios_dir="scenarios", clean_dir="data/clean"):
    """Build scenario ``name`` from scenario ``start_from``: its settings, constants and
    cold-plate table, with the given values changed. Each dict holds only the keys to change.

    Refuses to write over the shipped scenarios or over a folder this function did not
    write. Returns a dict with the starting and new params and the values that changed.
    """
    scenarios_dir = Path(scenarios_dir)
    if name in SHIPPED_SCENARIOS or name == start_from:
        raise ValueError(f"'{name}' is a shipped scenario or the starting one; choose a new name")
    out = scenarios_dir / name
    if out.exists() and not _built_here(out):
        raise ValueError(f"{out} exists and was not built by derive_scenario; choose a new name")
    start_path = scenarios_dir / start_from / "params.json"
    if not start_path.is_file():
        raise ValueError(f"scenario to start from not found: {start_path}")
    with open(start_path) as f:
        start = json.load(f)
    if "derived_from" in start:
        raise ValueError(f"'{start_from}' was edited by hand ({start['derived_from']['scenario']} plus "
                         f"running workloads or downtime), which the builder does not reproduce; "
                         f"start from '{start['derived_from']['scenario']}'")

    changed = {
        "settings": _changed(start["settings"], settings, "scenario setting"),
        "constants": _changed(start["constants_as_printed"], constants, "constant"),
        "cold_plate": _changed(start["cold_plate"], cold_plate, "cold-plate value"),
    }
    if "dt_seconds" in changed["settings"]:
        raise ValueError("dt_seconds is the slot length of the cleaned data and cannot be changed here")
    sc = start["settings"] | changed["settings"]
    const = start["constants_as_printed"] | changed["constants"]
    cp = start["cold_plate"] | changed["cold_plate"]
    built_from = {"scenario": start_from, "changed": {k: v for k, v in changed.items() if v}}

    build_scenario(name, sc, const, cp, clean_dir=clean_dir, scenarios_dir=scenarios_dir,
                   f_ref_mhz=start["f_ref_MHz"], slot_seconds=start["settings"]["dt_seconds"],
                   extra={"built_from": built_from})
    with open(out / "params.json") as f:
        new = json.load(f)
    return {"start": start, "params": new, "changed": built_from["changed"]}


def _built_here(d: Path) -> bool:
    try:
        with open(d / "params.json") as f:
            return "built_from" in json.load(f)
    except (OSError, ValueError):
        return False


In [ ]:
%%writefile {LIB}/scheduler/optimize.py
"""MILP optimizer (HiGHS) for thermal-aware workload scheduling.

Equation numbers refer to docs/PROJECT_CONTEXT.md section 2. This module builds the
constraints itself and shares no code with evaluate.py (the independent referee) or
baseline.py.

Model in one paragraph
----------------------
Each workload i runs as one unbroken block of periods on one server. ``Y[i,s,t]`` says it
runs on s in t. ``U[i,s,t]`` ("started by t") is a monotone 0/1 step that replaces the
start indicators ``S[i,s,t] = U[i,s,t] - U[i,s,t-1]``; ``X[i,s]`` is the last step of U.
Server power, the temperature recursion, a linear upper bound on the performance factor
DPF, and the work credit ``w`` follow the approved thesis equations. The supply
temperature of every zone and period is chosen from whole degrees by a one-hot choice so
the COP is a lookup. With cold plates on, each cold-plate server picks one flow setting per
period (one-hot, so resistance and pump power are lookups) and a coolant temperature; the
heat removed is linear in the temperature difference split across the settings. Priority
(56)-(57) orders the start times ``ST[i]`` of workloads that share a server, and servers
take no work in periods where AV = 0 (11). All limits are hard.

Units: P_IT in W, Q_crac / P_crac / P_fan in kW, dt in seconds, energy in kWh, CI in
g/kWh, carbon in g.
"""

from __future__ import annotations

import dataclasses
import math
import os
import tempfile
import time
from collections import defaultdict

import numpy as np

import highspy

from .types import (
    RELAX_COOLING, RELAX_CAPACITY, RELAX_DEADLINES, RELAX_FAMILIES, RELAX_PRIORITY, RELAX_THERMAL,
    STATUS_INFEASIBLE, STATUS_NO_SOLUTION, STATUS_OPTIMAL, STATUS_TIME_LIMIT,
    ColdPlateOps, Placement, RunSettings, Scenario, Schedule, SolveResult, cold_plate_counts, cop,
    priority_pairs,
)

_INF = highspy.kHighsInf
_EPS = 1e-9
_IPM_BINARIES = 30_000   # above this many binaries the MIP's LP relaxations use interior point


# ---------------------------------------------------------------------------
# Small LP container filled in bulk and handed to HiGHS in one call
# ---------------------------------------------------------------------------

class _LP:
    def __init__(self) -> None:
        self.lb: list[float] = []
        self.ub: list[float] = []
        self.cost: list[float] = []
        self.integer: list[bool] = []
        self.r_lo: list[float] = []
        self.r_hi: list[float] = []
        self.r_len: list[int] = []
        self.r_idx: list[int] = []
        self.r_val: list[float] = []

    def col(self, lb: float, ub: float, cost: float = 0.0, integer: bool = False) -> int:
        self.lb.append(lb)
        self.ub.append(ub)
        self.cost.append(cost)
        self.integer.append(integer)
        return len(self.lb) - 1

    def row(self, terms, lo: float, hi: float) -> None:
        """Add ``lo <= sum coef*col <= hi``; duplicate columns are merged."""
        acc: dict[int, float] = {}
        for c, v in terms:
            if c is not None:
                acc[c] = acc.get(c, 0.0) + v
        items = [(c, v) for c, v in acc.items() if abs(v) > 1e-13]
        self.r_lo.append(lo)
        self.r_hi.append(hi)
        self.r_len.append(len(items))
        for c, v in items:
            self.r_idx.append(c)
            self.r_val.append(v)

    @property
    def n_cols(self) -> int:
        return len(self.lb)

    @property
    def n_rows(self) -> int:
        return len(self.r_lo)

    def to_highs(self, log: bool = False) -> highspy.Highs:
        h = highspy.Highs()
        h.setOptionValue("log_to_console", bool(log))   # before passModel: no banner
        lp = highspy.HighsLp()
        lp.num_col_ = self.n_cols
        lp.num_row_ = self.n_rows
        lp.col_cost_ = np.asarray(self.cost, dtype=np.float64)
        lp.col_lower_ = np.asarray(self.lb, dtype=np.float64)
        lp.col_upper_ = np.asarray(self.ub, dtype=np.float64)
        lp.row_lower_ = np.asarray(self.r_lo, dtype=np.float64)
        lp.row_upper_ = np.asarray(self.r_hi, dtype=np.float64)
        lp.a_matrix_.format_ = highspy.MatrixFormat.kRowwise
        start = np.zeros(self.n_rows + 1, dtype=np.int32)
        np.cumsum(np.asarray(self.r_len, dtype=np.int64), out=start[1:])
        lp.a_matrix_.start_ = start
        lp.a_matrix_.index_ = np.asarray(self.r_idx, dtype=np.int32)
        lp.a_matrix_.value_ = np.asarray(self.r_val, dtype=np.float64)
        integ = np.asarray(self.integer, dtype=bool)
        lp.integrality_ = [highspy.HighsVarType.kInteger if b else highspy.HighsVarType.kContinuous
                           for b in integ]
        h.passModel(lp)
        return h


# ---------------------------------------------------------------------------
# Cold plates. The temperature, heat-balance and objective code reference the columns
# Q_cp[s,t] (W heat removed by all plates of s) and P_cp[s,t] (W pump power).
# ---------------------------------------------------------------------------

class _ColdPlateCols:
    """Per cold-plate server k and period t: one-hot flow choice b[k][t][j], the temperature
    difference split d[k][t][j] across the settings, and the coolant temperature cool[k][t].
    With the settings fixed (cp_mode "fixed") b and cool are fixed columns, and neg[k][t]
    (coolant above T_prev) with the switch on[k][t] lets the drive fall to zero."""

    def __init__(self):
        self.b: dict[int, list[list[int]]] = {}
        self.d: dict[int, list[list[int]]] = {}
        self.cool: dict[int, list[int]] = {}
        self.neg: dict[int, list[int]] = {}
        self.on: dict[int, list[int]] = {}
        self.fixed: tuple[int, float] | None = None    # (flow level, coolant degC)


def _add_cold_plate_terms(lp: _LP, scenario: Scenario, plates: np.ndarray, d_max: float,
                          ci_w_to_g: np.ndarray, fixed: tuple[int, float] | None = None):
    """Q_cp[s,t] and P_cp[s,t] columns, plus the choice columns of servers with cold plates.

    Servers without cold plates get Q_cp = P_cp = 0. P_cp enters the objective with the same
    weight as IT power (W -> g per period). ``fixed`` = (flow level, coolant) holds every
    plate at one setting. The rows are added by _add_cold_plate_rows.
    """
    S, T = scenario.S, scenario.T
    cp = scenario.cold_plate
    pump = cp.pump_W()
    nlev = len(cp.flow_settings_Lmin)
    cols = _ColdPlateCols()
    cols.fixed = fixed
    Q_cp, P_cp = [], []
    for k in range(S):
        n = int(plates[k])
        Q_cp.append([lp.col(0.0, n * cp.Q_max_per_plate_W) for _ in range(T)])
        P_cp.append([lp.col(n * pump.min(), n * pump.max(), cost=float(ci_w_to_g[t]))
                     for t in range(T)])
        if n == 0:
            continue
        if fixed is None:
            cols.b[k] = [[lp.col(0.0, 1.0, integer=True) for _ in range(nlev)] for _ in range(T)]
            cols.cool[k] = [lp.col(cp.coolant_min_C, cp.coolant_max_C) for _ in range(T)]
        else:
            lv, cool = fixed
            cols.b[k] = [[lp.col(float(j == lv), float(j == lv)) for j in range(nlev)]
                         for _ in range(T)]
            cols.cool[k] = [lp.col(cool, cool) for _ in range(T)]
            cols.on[k] = [lp.col(0.0, 1.0, integer=True) for _ in range(T)]
            cols.neg[k] = [lp.col(0.0, _INF) for _ in range(T)]
        cols.d[k] = [[lp.col(0.0, d_max) for _ in range(nlev)] for _ in range(T)]
    return Q_cp, P_cp, cols


def _add_cold_plate_rows(lp: _LP, scenario: Scenario, plates: np.ndarray, d_max: float,
                         cols: _ColdPlateCols, Q_cp, P_cp, P_IT, Tcol,
                         neg_max: dict[int, float] | None = None) -> None:
    """For server k with n plates in period t, with T_prev = T[k,t-1] (T0 in period 1):

        sum_j b_j = 1                              one flow setting
        sum_j d_j = T_prev - coolant               d_j <= d_max * b_j, d_j >= 0
        Q_cp <= n * sum_j G_j * d_j                heat through the plates, "<=" as agreed
        Q_cp <= P_IT                               no more than the server's heat
        Q_cp <= n * Q_max                          column bound, Eq. (55)
        P_cp  = n * sum_j pump_j * b_j             Eq. (48) at the chosen setting

    With the settings fixed the coolant cannot follow T_prev down, so the drive is
    max(0, T_prev - coolant): sum_j d_j - neg = T_prev - coolant, sum_j d_j <= d_max * on,
    neg <= neg_max[k] * (1 - on).
    """
    cp = scenario.cold_plate
    G = cp.conductance_W_per_C()
    pump = cp.pump_W()
    for k, bk in cols.b.items():
        n = int(plates[k])
        sv = scenario.servers[k]
        for t in range(scenario.T):
            b, d, cool = bk[t], cols.d[k][t], cols.cool[k][t]
            lp.row([(c_, 1.0) for c_ in b], 1.0, 1.0)
            terms = [(c_, 1.0) for c_ in d] + [(cool, 1.0)]
            if cols.fixed is not None:
                on, neg = cols.on[k][t], cols.neg[k][t]
                terms.append((neg, -1.0))
                lp.row([(c_, 1.0) for c_ in d] + [(on, -d_max)], -_INF, 0.0)
                lp.row([(neg, 1.0), (on, neg_max[k])], -_INF, neg_max[k])
            if t == 0:
                lp.row(terms, sv.T0_C, sv.T0_C)
            else:
                lp.row(terms + [(Tcol[k][t - 1], -1.0)], 0.0, 0.0)
            for dj, bj in zip(d, b):
                lp.row([(dj, 1.0), (bj, -d_max)], -_INF, 0.0)
            lp.row([(Q_cp[k][t], 1.0)] + [(dj, -n * float(g)) for dj, g in zip(d, G)], -_INF, 0.0)
            lp.row([(Q_cp[k][t], 1.0), (P_IT[k][t], -1.0)], -_INF, 0.0)
            lp.row([(P_cp[k][t], 1.0)] + [(bj, -n * float(pw)) for bj, pw in zip(b, pump)], 0.0, 0.0)


def _add_fan_terms(lp: _LP, scenario: Scenario, ci_kw_to_g: np.ndarray):
    """P_fan[z,t] (kW): fixed at zero; enters the objective (kW -> g per period)."""
    Z, T = scenario.Z, scenario.T
    return [[lp.col(0.0, 0.0, cost=float(ci_kw_to_g[t])) for t in range(T)] for _ in range(Z)]


# ---------------------------------------------------------------------------
# Model construction
# ---------------------------------------------------------------------------

@dataclasses.dataclass(frozen=True)
class WindowExtras:
    """What a per-tick window (scheduler/pertick.py) adds to the model of its scenario.

    ``priority_pairs``: the pairs O_ij = 1 to enforce, in the window's workload numbers,
    replacing priority_pairs(scenario) (the window's arrivals are shifted, so the pairs come
    from the whole day). ``forbidden``: per workload, servers it may not use. ``supply_history``:
    shape (Z, h), the supply temperatures of the h periods just before the window (the last
    column is the period right before it), so that the ramp limits (35)-(36) hold across the
    window's start."""
    priority_pairs: tuple[tuple[int, int], ...] | None = None
    forbidden: dict[int, frozenset[int]] | None = None
    supply_history: np.ndarray | None = None


class _Window:
    """Start and occupancy window of one (workload, server) pair: starts a..b, occupied
    periods ya..yb, run length n_min..n_max periods."""
    __slots__ = ("a", "b", "ya", "yb", "n_min", "n_max")

    def __init__(self, a, b, ya, yb, n_min, n_max):
        self.a, self.b, self.ya, self.yb, self.n_min, self.n_max = a, b, ya, yb, n_min, n_max


class _Built:
    """The assembled model plus the index maps needed to read a solution."""


def _levels(scenario: Scenario) -> list[int]:
    c = scenario.constants
    lo = math.ceil(c.T_crac_min_C - _EPS)
    hi = math.floor(c.T_crac_max_C + _EPS)
    levels = [k for k in range(lo, hi + 1) if k <= c.T_max_zone_C + _EPS]
    if not levels:
        raise ValueError("no whole-degree supply temperature lies in the CRAC range and "
                         "below the zone limit")
    return levels


def _allowed_servers(scenario: Scenario, original_server: dict[int, int],
                     settings: RunSettings,
                     forbidden: dict[int, frozenset[int]] | None = None) -> dict[int, list[int]]:
    zone_of = {sv.s: sv.zone for sv in scenario.servers}
    out: dict[int, list[int]] = {}
    for w in scenario.workloads:
        if w.i not in original_server:
            raise ValueError(f"original_server has no entry for workload {w.i}")
        o = original_server[w.i]
        if o not in zone_of:
            raise ValueError(f"original_server[{w.i}] = {o} is not a server of the scenario")
        if w.B == 1 or settings.reassignment == "none":
            out[w.i] = [o]
        elif settings.reassignment == "same_zone":
            out[w.i] = [sv.s for sv in scenario.servers if sv.zone == zone_of[o]]
        else:
            out[w.i] = [sv.s for sv in scenario.servers]
        if forbidden and w.i in forbidden:
            out[w.i] = [s for s in out[w.i] if s not in forbidden[w.i]]
    return out


def _build(scenario: Scenario, original_server: dict[int, int], settings: RunSettings,
           relax: tuple[str, ...], feasibility_only: bool, ignore_temperature: bool = False,
           extras: WindowExtras | None = None):
    """``ignore_temperature`` leaves server temperature out: no temperature recursion, no
    thermal cap, DPF = 1. Pass 1 of the thermal-blind placement uses it to choose servers.
    ``extras``: see WindowExtras."""
    extras = extras or WindowExtras()
    settings.validate()
    for fam in relax:
        if fam not in RELAX_FAMILIES:
            raise ValueError(f"unknown relaxation family {fam!r}")
    c = scenario.constants
    S, Z, T, I = scenario.S, scenario.Z, scenario.T, scenario.I
    dt = scenario.dt_seconds
    dt_h = scenario.dt_hours
    cap = settings.thermal_cap_C(scenario)
    if cap > c.T_throttle_C + _EPS:
        raise ValueError(
            f"thermal cap {cap:g} degC (T_max - H_min) exceeds T_throttle {c.T_throttle_C:g} degC: "
            "the linear DPF bound is only valid below T_throttle")

    relax_dead = RELAX_DEADLINES in relax
    relax_therm = RELAX_THERMAL in relax
    relax_cap = RELAX_CAPACITY in relax
    relax_cool = RELAX_COOLING in relax
    notes: list[str] = []

    fixed_mode = settings.supply_temp_mode == "fixed"
    if feasibility_only and not fixed_mode:
        # Feasibility does not depend on the supply schedule: the lowest level gives the lowest
        # temperatures, the same Q_crac cap, and trivially satisfies the ramps and zone limit.
        fixed_mode = True
        feas_supply = float(_levels(scenario)[0])
        settings = dataclasses.replace(settings, supply_temp_mode="fixed",
                                       fixed_supply_temp_C=feas_supply)
    if fixed_mode:
        v = float(settings.optimized_fixed_supply_temp_C())
        if not (c.T_crac_min_C - _EPS <= v <= c.T_crac_max_C + _EPS):
            raise ValueError(f"fixed supply temperature {v:g} degC is outside the CRAC range "
                             f"[{c.T_crac_min_C:g}, {c.T_crac_max_C:g}]")
        if v > c.T_max_zone_C + _EPS:
            raise ValueError(f"fixed supply temperature {v:g} degC exceeds the zone limit "
                             f"{c.T_max_zone_C:g}")
        fixed_value = v
        levels: list[int] = []
    else:
        fixed_value = None
        levels = _levels(scenario)
    cop_levels = [float(cop(c, k)) for k in levels]

    # Highest temperature the T columns may take; the linear DPF bound holds up to T_throttle.
    cap_eff = c.T_throttle_C if relax_therm else cap
    derating = cap_eff > c.T_dvfs_C + _EPS and not ignore_temperature   # DPF can fall below 1
    slope = c.R_dvfs / (c.T_throttle_C - c.T_dvfs_C)
    dpf_cap = min(1.0, 1.0 - slope * (cap_eff - c.T_dvfs_C)) if derating else 1.0

    servers = scenario.servers
    zone_of = np.array([sv.zone for sv in servers], dtype=int)
    allowed = _allowed_servers(scenario, original_server, settings, extras.forbidden)
    CI = np.asarray(scenario.CI_g_per_kWh, dtype=float)
    if CI.shape != (T,):
        raise ValueError("CI_g_per_kWh must have shape (T,)")

    # ---- windows per (i, s) ------------------------------------------------
    win: dict[tuple[int, int], _Window] = {}
    empty_workloads: list[int] = []
    for w in scenario.workloads:
        D = T + 1 if relax_dead else min(w.D, T + 1)
        last_start_by_wait = T + 1 if relax_dead else w.A + w.AD
        for s in allowed[w.i]:
            sv = servers[s - 1]
            per_full = w.C_req * sv.f_MHz * dt          # work of one full-speed period
            if w.W_MHz_s <= 0:
                n_min = n_max = 1
            else:
                n_min = max(1, math.ceil(w.W_MHz_s / per_full - _EPS))
                n_max = (T if dpf_cap <= 1e-6 else
                         max(n_min, math.ceil(w.W_MHz_s / (per_full * dpf_cap) - _EPS)))
            a = max(w.A, 1)
            b = min(last_start_by_wait, D - n_min)
            if b < a:
                continue
            ya = a
            yb = min(D - 1, b + n_max - 1)
            win[(w.i, s)] = _Window(a, b, ya, yb, n_min, n_max)
        if not any((w.i, s) in win for s in allowed[w.i]):
            empty_workloads.append(w.i)

    built = _Built()
    built.empty_workloads = empty_workloads
    built.notes = notes
    built.win = win
    built.levels = levels
    built.fixed_value = fixed_value
    built.derating = derating
    built.relax = relax
    built.feasibility_only = feasibility_only
    built.ignore_temperature = ignore_temperature
    if empty_workloads:
        return built

    lp = _LP()
    ci_w_to_g = CI * dt_h / 1000.0      # g per W held for one period
    ci_kw_to_g = CI * dt_h              # g per kW held for one period
    if feasibility_only:
        ci_w_to_g = np.zeros(T)
        ci_kw_to_g = np.zeros(T)

    # ---- scheduling columns ------------------------------------------------
    # X[i,s]: assigned; U[i,s,t]: started by t (t in [a, b-1]; U at b is X);
    # E[i,s,t]: tail period (beyond the n_min core periods) is on.
    X: dict[tuple[int, int], int] = {}
    U: dict[tuple[int, int], dict[int, int]] = {}
    E: dict[tuple[int, int], dict[int, int]] = {}
    for w in scenario.workloads:
        for s in allowed[w.i]:
            wd = win.get((w.i, s))
            if wd is None:
                continue
            X[(w.i, s)] = lp.col(0.0, 1.0, integer=True)
            U[(w.i, s)] = {t: lp.col(0.0, 1.0, integer=True) for t in range(wd.a, wd.b)}
            E[(w.i, s)] = {}
            if wd.n_max > wd.n_min:
                for t in range(wd.a + wd.n_min, wd.yb + 1):
                    E[(w.i, s)][t] = lp.col(0.0, 1.0, integer=True)
    built.X, built.U, built.E = X, U, E

    def ustep(i: int, s: int, t: int):
        """Column of 'started by t' for (i, s), or None when it is the constant 0."""
        wd = win[(i, s)]
        if t < wd.a:
            return None
        if t >= wd.b:
            return X[(i, s)]
        return U[(i, s)][t]

    # Occupancy of (i,s) in period t = core + tail = U[t] - U[t-n_min] + E[t]
    by_st: dict[tuple[int, int], list[tuple[int, list]]] = defaultdict(list)
    for (i, s), wd in win.items():
        emap = E[(i, s)]
        for t in range(wd.a, wd.yb + 1):
            terms = []
            if t <= wd.b + wd.n_min - 1:
                terms += [(ustep(i, s, t), 1.0), (ustep(i, s, t - wd.n_min), -1.0)]
            if t in emap:
                terms.append((emap[t], 1.0))
            by_st[(s, t)].append((i, terms))

    # Upper bound of IT power per server: all workloads that could ever run there.
    max_cores: dict[int, float] = defaultdict(float)
    for (i, s) in win:
        max_cores[s] += scenario.workloads[i - 1].C_req
    P_ub = []
    for sv in servers:
        if relax_cap:
            ub = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * max_cores[sv.s] / sv.C_cores
        else:
            ub = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * min(1.0, max_cores[sv.s] / sv.C_cores)
        P_ub.append(ub)

    # ---- system columns ----------------------------------------------------
    P_IT = [[lp.col(sv.P_idle_W, P_ub[sv.s - 1], cost=float(ci_w_to_g[t])) for t in range(T)]
            for sv in servers]
    plates = cold_plate_counts(scenario, settings)
    cp_d_max = max([cap_eff] + [sv.T0_C for sv in servers]) - scenario.cold_plate.coolant_min_C
    for sv in servers:
        if plates[sv.s - 1] > 0 and sv.T0_C < scenario.cold_plate.coolant_min_C:
            raise ValueError(f"server {sv.s}: T0 {sv.T0_C:g} C is below the lowest coolant "
                             f"temperature {scenario.cold_plate.coolant_min_C:g} C")
    cp_fixed = None
    cp_neg_max: dict[int, float] = {}
    if settings.cp_mode == "fixed" and plates.any():
        flow, cool = settings.optimized_fixed_cp()
        cps = scenario.cold_plate
        if not cps.coolant_min_C - _EPS <= cool <= cps.coolant_max_C + _EPS:
            raise ValueError(f"fixed coolant temperature {cool:g} degC is outside "
                             f"[{cps.coolant_min_C:g}, {cps.coolant_max_C:g}], Eq. (49)")
        cp_fixed = (cps.level_of(flow), cool)
        # Lowest temperature a server can reach: the steady state with all heat removed.
        sup_min = fixed_value if fixed_mode else float(levels[0])
        for sv in servers:
            cp_neg_max[sv.s - 1] = max(1.0, cool - min(sv.T0_C, sup_min + sv.th_a_C) + 1.0)
    Q_cp, P_cp, cp_cols = _add_cold_plate_terms(lp, scenario, plates, cp_d_max, ci_w_to_g,
                                                cp_fixed)
    P_fan = _add_fan_terms(lp, scenario, ci_kw_to_g)
    T_ub = _INF if ignore_temperature else cap_eff
    Tcol = [[lp.col(-_INF, T_ub) for t in range(T)] for _ in servers]
    # Loss L = 1 - DPF (only where some workload has tail periods, i.e. where DPF matters),
    # and its running sum CL, per server and period.
    Lmax = min(1.0, 1.0 - dpf_cap)
    tail_servers = {s for (i, s), wd in win.items() if wd.n_max > wd.n_min}
    Lcol: dict[int, list[int]] = {}
    CLcol: dict[int, list[int]] = {}
    if derating:
        for s in tail_servers:
            Lcol[s] = [lp.col(0.0, Lmax) for t in range(T)]
            CLcol[s] = [lp.col(0.0, T * Lmax) for t in range(T)]

    # Q_crac upper bound: Eq. (54) as a bound (also the big-M of the level split)
    sum_pub_kw = [sum(P_ub[sv.s - 1] for sv in servers if sv.zone == z) / 1000.0
                  for z in range(1, Z + 1)]
    Qmax = [sum_pub_kw[z - 1] if relax_cool else min(c.Q_crac_kW, sum_pub_kw[z - 1])
            for z in range(1, Z + 1)]
    Qcrac = [[None] * T for _ in range(Z)]
    Tsup = [[None] * T for _ in range(Z)]
    choice: dict[tuple[int, int], list[int]] = {}
    qcols: dict[tuple[int, int], list[int]] = {}
    for z in range(1, Z + 1):
        for t in range(T):
            if fixed_mode:
                Tsup[z - 1][t] = lp.col(fixed_value, fixed_value)
                cost = float(ci_kw_to_g[t] / cop(c, fixed_value))
                Qcrac[z - 1][t] = lp.col(0.0, Qmax[z - 1], cost=cost)
            else:
                Tsup[z - 1][t] = lp.col(levels[0], levels[-1])
                Qcrac[z - 1][t] = lp.col(0.0, Qmax[z - 1])
                choice[(z, t)] = [lp.col(0.0, 1.0, integer=True) for _ in levels]
                qcols[(z, t)] = [lp.col(0.0, Qmax[z - 1], cost=float(ci_kw_to_g[t] / cp))
                                 for cp in cop_levels]

    # ---- workload rows -----------------------------------------------------
    for w in scenario.workloads:
        i = w.i
        lp.row([(X[(i, s)], 1.0) for s in allowed[i] if (i, s) in X], 1.0, 1.0)          # (9)
        work_terms = []
        for s in allowed[i]:
            wd = win.get((i, s))
            if wd is None:
                continue
            Ucols = U[(i, s)]
            for t in range(wd.a + 1, wd.b):                      # monotone started-by
                lp.row([(Ucols[t], 1.0), (Ucols[t - 1], -1.0)], 0.0, _INF)
            if wd.b > wd.a:
                lp.row([(X[(i, s)], 1.0), (Ucols[wd.b - 1], -1.0)], 0.0, _INF)
            emap = E[(i, s)]
            for t, e in emap.items():
                # tail period t is on only inside the tail region of the actual start
                lp.row([(e, 1.0), (ustep(i, s, t - wd.n_min), -1.0), (ustep(i, s, t - wd.n_max), 1.0)],
                       -_INF, 0.0)
                # ... and only as an unbroken extension of the core block
                lp.row([(e, 1.0), (emap.get(t - 1), -1.0), (ustep(i, s, t - wd.n_min), -1.0),
                        (ustep(i, s, t - wd.n_min - 1), 1.0)], -_INF, 0.0)
            if w.W_MHz_s <= 0:
                continue
            coef = w.C_req * servers[s - 1].f_MHz * dt / w.W_MHz_s
            if wd.n_max == wd.n_min:
                # n_max == n_min: any core block delivers at least W even at the worst DPF
                work_terms.append((X[(i, s)], 1.0))
                continue
            work_terms.append((X[(i, s)], coef * wd.n_min))
            if not derating:
                work_terms += [(e, coef) for e in emap.values()]
                continue
            # derating: core credit n_min - (loss over the core block of the chosen start)
            M = wd.n_min * Lmax
            cl = lp.col(0.0, M)
            work_terms.append((cl, -coef))
            CL = CLcol[s]
            for tau in range(wd.a, wd.b + 1):
                lp.row([(cl, 1.0), (CL[tau + wd.n_min - 2], -1.0),
                        (CL[tau - 2] if tau >= 2 else None, 1.0),
                        (ustep(i, s, tau), -M), (ustep(i, s, tau - 1), M)], -M, _INF)
            # tail credit: wt <= E, wt <= 1 - L
            for t, e in emap.items():
                wt = lp.col(0.0, 1.0)
                lp.row([(wt, 1.0), (e, -1.0)], -_INF, 0.0)
                lp.row([(wt, 1.0), (Lcol[s][t - 1], 1.0)], -_INF, 1.0)
                work_terms.append((wt, coef))
        if w.W_MHz_s > 0:
            lp.row(work_terms, 1.0, _INF)                                               # (39)
    if derating:
        for s in tail_servers:
            sv = servers[s - 1]
            for t in range(1, T + 1):
                # L >= slope * (T - T_dvfs): the linear piece of Eq. (37)
                lp.row([(Lcol[s][t - 1], 1.0), (Tcol[s - 1][t - 1], -slope)], -slope * c.T_dvfs_C, _INF)
                lp.row([(CLcol[s][t - 1], 1.0), (CLcol[s][t - 2] if t >= 2 else None, -1.0),
                        (Lcol[s][t - 1], -1.0)], 0.0, 0.0)

    # ---- priority, Eq. (56)-(57) -------------------------------------------
    # ST[i] = sum_s (b * X[i,s] - sum_{t<b} U[i,s,t]), the start period. For a pair (i, j)
    # with O_ij = 1 and a shared server s:
    #   ST_i - ST_j <= M * (2 - X_is - X_js) + M * started_j(A_i - 1)
    # so i starts no later than j when both run on s and j had not started when i arrived.
    ST: dict[int, int] = {}

    def st_col(i: int) -> int:
        if i not in ST:
            spans = [win[(i, s)] for s in allowed[i] if (i, s) in win]
            col = lp.col(float(min(wd.a for wd in spans)), float(max(wd.b for wd in spans)))
            terms = [(col, 1.0)]
            for s in allowed[i]:
                wd = win.get((i, s))
                if wd is None:
                    continue
                terms.append((X[(i, s)], -float(wd.b)))
                terms += [(uc, 1.0) for uc in U[(i, s)].values()]
            lp.row(terms, 0.0, 0.0)
            ST[i] = col
        return ST[i]

    n_priority_rows = 0
    if settings.priority and RELAX_PRIORITY not in relax:
        pairs = priority_pairs(scenario) if extras.priority_pairs is None else extras.priority_pairs
        for i, j in pairs:
            A_i = scenario.workloads[i - 1].A
            shared = [s for s in allowed[i] if (i, s) in win and (j, s) in win]
            if not shared:
                continue
            M = (max(win[(i, s)].b for s in allowed[i] if (i, s) in win)
                 - min(win[(j, s)].a for s in allowed[j] if (j, s) in win))
            if M <= 0:
                continue                      # i can never start after j
            started_j = [(ustep(j, s, A_i - 1), -float(M)) for s in allowed[j] if (j, s) in win]
            for s in shared:
                wi, wj = win[(i, s)], win[(j, s)]
                if wj.b < A_i or wi.b <= max(wj.a, A_i):
                    continue                  # no inversion possible on s
                lp.row([(st_col(i), 1.0), (st_col(j), -1.0), (X[(i, s)], float(M)),
                        (X[(j, s)], float(M))] + started_j, -_INF, 2.0 * M)
                n_priority_rows += 1

    # ---- per server, per period -------------------------------------------
    AV = scenario.availability()
    for sv in servers:
        k = sv.s - 1
        dP = sv.P_max_W - sv.P_idle_W
        alpha, R, a_s = sv.th_alpha, sv.th_R_C_per_W, sv.th_a_C
        z = sv.zone
        for t in range(1, T + 1):
            items = by_st.get((sv.s, t), [])
            if not AV[k, t - 1]:
                for _, occ in items:                                                   # (11)
                    lp.row(occ, 0.0, 0.0)
            # (16), (20), (21): P_IT = P_idle + dP * sum C_req * occupancy / C_s
            terms = [(P_IT[k][t - 1], 1.0)]
            cpu: list = []
            mem: list = []
            cpu_sum = mem_sum = 0.0
            for i, occ in items:
                wl = scenario.workloads[i - 1]
                terms += [(col, -dP * wl.C_req / sv.C_cores * v) for col, v in occ]
                cpu += [(col, wl.C_req / sv.C_cores * v) for col, v in occ]
                mem += [(col, wl.M_req_MB / sv.M_MB * v) for col, v in occ]
                cpu_sum += wl.C_req / sv.C_cores
                mem_sum += wl.M_req_MB / sv.M_MB
            lp.row(terms, sv.P_idle_W, sv.P_idle_W)
            if not relax_cap:
                if cpu_sum > 1.0 + 1e-12:                                              # (12)
                    lp.row(cpu, -_INF, 1.0)
                if mem_sum > 1.0 + 1e-12:                                              # (13)
                    lp.row(mem, -_INF, 1.0)
            if ignore_temperature:
                continue
            # temperature recursion (approved)
            terms = [(Tcol[k][t - 1], 1.0), (Tsup[z - 1][t - 1], -(1 - alpha)),
                     (P_IT[k][t - 1], -(1 - alpha) * R), (Q_cp[k][t - 1], (1 - alpha) * R)]
            rhs = (1 - alpha) * a_s
            if t == 1:
                rhs += alpha * sv.T0_C
            else:
                terms.append((Tcol[k][t - 2], -alpha))
            lp.row(terms, rhs, rhs)

    _add_cold_plate_rows(lp, scenario, plates, cp_d_max, cp_cols, Q_cp, P_cp, P_IT, Tcol,
                         cp_neg_max)

    # ---- cooling zones -----------------------------------------------------
    members = {z: [sv for sv in servers if sv.zone == z] for z in range(1, Z + 1)}
    for z in range(1, Z + 1):
        for t in range(1, T + 1):
            # heat balance: Q_crac >= sum (P_IT - Q_cp) / 1000   (A3)
            terms = [(Qcrac[z - 1][t - 1], 1.0)]
            for sv in members[z]:
                terms.append((P_IT[sv.s - 1][t - 1], -1e-3))
                terms.append((Q_cp[sv.s - 1][t - 1], 1e-3))
            lp.row(terms, 0.0, _INF)
            if not fixed_mode:
                ch = choice[(z, t - 1)]
                lp.row([(col, 1.0) for col in ch], 1.0, 1.0)
                lp.row([(Tsup[z - 1][t - 1], 1.0)] + [(col, -float(lv)) for col, lv in zip(ch, levels)],
                       0.0, 0.0)
                qs = qcols[(z, t - 1)]
                lp.row([(Qcrac[z - 1][t - 1], 1.0)] + [(q, -1.0) for q in qs], 0.0, 0.0)
                for q, col in zip(qs, ch):
                    lp.row([(q, 1.0), (col, -Qmax[z - 1])], -_INF, 0.0)
        if not fixed_mode:
            for lag, lim in ((c.ramp_short_lag, c.ramp_short_C), (c.ramp_long_lag, c.ramp_long_C)):
                for t in range(lag + 1, T + 1):                                         # (35), (36)
                    lp.row([(Tsup[z - 1][t - 1], 1.0), (Tsup[z - 1][t - 1 - lag], -1.0)], -lim, lim)
                # across the window's start: period t against period t - lag before it
                hist = extras.supply_history
                h = 0 if hist is None else hist.shape[1]
                for t in range(1, min(lag, T) + 1):
                    back = lag - t + 1               # periods before the window's start
                    if back <= h:
                        v = float(hist[z - 1, h - back])
                        lp.row([(Tsup[z - 1][t - 1], 1.0)], v - lim, v + lim)

    # ---- objective: P_IT, P_cp, P_fan and CRAC columns already carry CI * energy factors
    built.lp = lp
    built.P_IT, built.Q_cp, built.P_cp, built.P_fan = P_IT, Q_cp, P_cp, P_fan
    built.Tcol, built.Qcrac, built.Tsup = Tcol, Qcrac, Tsup
    built.slope = slope
    built.choice = choice
    built.qcols = qcols
    built.cop_levels = cop_levels
    built.fixed_cop = None if fixed_value is None else float(cop(c, fixed_value))
    built.dpf_cap = dpf_cap
    built.plates, built.cp_cols = plates, cp_cols
    built.n_priority_rows = n_priority_rows
    built.n_unavailable = int((AV == 0).sum())
    return built


def _model_notes(scenario: Scenario, settings: RunSettings, built: _Built) -> list[str]:
    n = [
        "Y[i,s,t] is eliminated: a workload occupies n_min core periods after its start, "
        "U[t] - U[t-n_min], plus optional tail binaries E[i,s,t] only where n_max > n_min, because "
        "the Y-based model made the root LP of 'consolidated6' unsolvable (500k rows).",
        "Start indicators S are replaced by cumulative 'started by t' steps U[i,s,t] (S = U[t] - "
        "U[t-1], X[i,s] is the last step), so one unbroken block needs no window sums.",
        "The tail E[t] <= U[t-n_min] - U[t-n_max] and E[t] <= E[t-1] + U[t-n_min] - U[t-n_min-1] keep "
        "the run unbroken and within n_min..n_max periods; Y <= X (10) is then implied and Eq. (9) is kept.",
        "n_min = ceil(W/(C_req*f_s*dt)) is the shortest run since DPF <= 1, and starts with t + n_min > D "
        "are dropped.",
        "n_max = ceil(W/(C_req*f_s*dt*DPF_cap)) caps the run: valid for the optimum because trimming the "
        "tail of a run never raises power, heat or temperature, so every other limit stays met.",
        "DPF enters only through the loss L = 1 - DPF >= slope*(T - T_dvfs), L >= 0 (the linear piece of "
        "Eq. 37, valid below T_throttle, so a cap above T_throttle is rejected), and only on servers where "
        "some pair has n_max > n_min; higher DPF never hurts, so no binaries are needed.",
        "Work (39) uses core credit n_min*X - coreLoss, where coreLoss >= CL[tau+n_min-1] - CL[tau-1] "
        "- M(1 - sigma_tau) over the running sum CL of L, plus tail credits wt <= E, wt <= 1 - L; for a pair "
        "with n_max == n_min any core block delivers at least W, so its term is exactly X.",
        "The work row is divided by W_i (right side 1), and with the cap at or below T_dvfs there are no "
        "loss, tail or work-credit columns at all.",
        "Server utilisation and heat are substituted into P_IT[s,t] = P_idle + (P_max-P_idle)*"
        "sum C_req*occupancy/C_s (Eq. 16, 20, 21); P_IT is bounded by P_max (or by the cores of the "
        "workloads that can ever run there) as a column bound.",
        "Eq. (54) Q_crac <= 116 kW and the big-M of the supply-level split are one column bound "
        "Qmax = min(116, sum of the zone's maximum IT power in kW); the heat balance sets Q_crac >= "
        "heat and the cost increases in Q_crac, so it sits at equality in any optimum.",
        "CRAC power is Q_crac / COP (divide), COP looked up per whole-degree level; the printed "
        "Eq. (46) multiplies.",
        "Fan power P_fan is a named column fixed at zero (_add_fan_terms) that already appears in "
        "the objective.",
        "Idle energy is in the objective through the P_IT columns (lower bound P_idle), so "
        "objective_g equals total carbon with no separate offset.",
        "The wait limit (28) is applied to all workloads, which includes (25). Workloads with "
        "B_i = 1 keep their original server (29).",
    ]
    if settings.priority and RELAX_PRIORITY not in built.relax:
        n.append(
            f"Priority (56)-(57): O_ij = 1 for a time-sensitive i and a delay-tolerant j, and for the "
            f"earlier arrival of two same-type, same-deadline workloads. It is linked to the start "
            f"times as: on a shared server, j may not start while i waits (A_i <= ST_j implies "
            f"ST_i <= ST_j), through ST_i = sum_s (b X - sum U) and big-M rows "
            f"({getattr(built, 'n_priority_rows', 0)} pair-server rows after dropping those that "
            f"cannot bind). EDF within a type is not in the equations and is not modelled.")
    else:
        n.append("Priority (56)-(57) is off for this run: start order is not constrained.")
    n_off = getattr(built, "n_unavailable", 0)
    n.append(f"Availability (11): {n_off} server-period(s) with AV = 0 take no work; their "
             "servers stay on at idle power." if n_off else
             "Availability (11): every server is available in every period (AV = 1).")
    plates = getattr(built, "plates", cold_plate_counts(scenario, settings))
    if plates.any():
        cp = scenario.cold_plate
        n.append(
            f"Cold plates: {int(plates.sum())} on {int((plates > 0).sum())} server(s). "
            f"Each picks one of the flow settings {list(cp.flow_settings_Lmin)} L/min per period "
            "(one-hot), so its resistance and the cubic pump power of Eq. (48) are lookups. "
            "T_prev - coolant is split into d_j <= d_max*b_j, so Q_cp <= n*sum G_j*d_j is linear "
            "(G_j = min(1/R_j, flow_j*c_pw)); Q_cp <= P_IT and Q_cp <= n*0.3 kW. T_prev is the "
            "previous period's temperature, and d_j >= 0 keeps the coolant at or below it.")
        if settings.cp_mode == "fixed":
            flow, cool = settings.optimized_fixed_cp()
            n.append(f"Cold plates fixed (cp_mode fixed): every plate runs at {flow:g} L/min and "
                     f"{cool:g} C coolant in every period. The drive is max(0, T_prev - coolant), "
                     "with one on/off binary per server and period.")
    else:
        n.append("Cold plates are off: Q_cp and P_cp are columns fixed at zero.")
    if getattr(built, "ignore_temperature", False):
        n.append("Server temperature ignored (pass 1 of the thermal-blind placement): no temperature "
                 "recursion, no thermal cap T <= T_max - H_min and DPF = 1, so temperature plays no part "
                 "in choosing servers. Cold plates may remove up to min(P_IT, n*0.3 kW) whatever the "
                 "server temperature.")
    if built.feasibility_only:
        n.append("Feasibility-only solve (diagnosis): zero objective and the supply fixed at the lowest "
                 "whole degree, which gives the same feasibility as optimising it (lowest temperatures, "
                 "same Q_crac cap, no ramp).")
    if RELAX_THERMAL in built.relax:
        n.append("RELAX_THERMAL: the cap T <= T_max - H_min is replaced by T <= T_throttle, the limit "
                 "of validity of the linear DPF bound, with DPF >= 0.")
    if RELAX_CAPACITY in built.relax:
        n.append("RELAX_CAPACITY: CPU and memory rows are dropped; P_IT is bounded by the cores of the "
                 "workloads that can run on the server.")
    if RELAX_COOLING in built.relax:
        n.append("RELAX_COOLING: Q_crac <= 116 dropped; the column bound is the zone's maximum IT power.")
    if RELAX_DEADLINES in built.relax:
        n.append("RELAX_DEADLINES: starts and runs may extend over A_i..T (arrival still enforced).")
    return n


# ---------------------------------------------------------------------------
# Solving
# ---------------------------------------------------------------------------

def _infeasible_result(built: _Built, relax, t0: float, text: str) -> SolveResult:
    return SolveResult(
        status=STATUS_INFEASIBLE, schedule=None, supply_temp_C=None, objective_g=None,
        best_bound_g=None, mip_gap=None, solve_time_s=time.perf_counter() - t0,
        n_variables=0, n_binaries=0, n_constraints=0, relaxed=tuple(relax),
        solver_status_text=text, notes=list(built.notes))


def _fill_warm_start(scenario: Scenario, built: _Built,
                     runs: dict[int, tuple[int, int, int]]) -> int:
    """Place every workload the starting schedule leaves open, by the baseline rule: in
    workload order, at the earliest start of its window where a server it may use has room
    for its cores and memory (Eq. 12-13) and is available (Eq. 11) for the whole run, least
    loaded server first. The run is n_max periods (the longest, so it delivers its work at
    any DPF), cut at the window's end. Temperature, cooling and priority are not checked: a
    start that breaks them is rejected by HiGHS as before. Adds to ``runs`` (i -> server,
    start, length) and returns how many workloads it placed."""
    T = scenario.T
    servers = scenario.servers
    AV = scenario.availability()
    cores = np.zeros((scenario.S, T + 2))
    mem = np.zeros((scenario.S, T + 2))
    wl = {w.i: w for w in scenario.workloads}
    for i, (s, st, m) in runs.items():
        cores[s - 1, st:st + m] += wl[i].C_req
        mem[s - 1, st:st + m] += wl[i].M_req_MB
    by_i: dict[int, list[int]] = {}
    for (i, s) in built.win:
        by_i.setdefault(i, []).append(s)
    n = 0
    for w in scenario.workloads:
        if w.i in runs or w.i not in by_i:
            continue
        cands = sorted(by_i[w.i])
        starts = sorted({t for s in cands for t in range(built.win[(w.i, s)].a,
                                                          built.win[(w.i, s)].b + 1)})
        for st in starts:
            fits = []
            for s in cands:
                wd = built.win[(w.i, s)]
                if not wd.a <= st <= wd.b:
                    continue
                m = min(wd.n_max, wd.yb - st + 1)
                sv = servers[s - 1]
                k = s - 1
                if (cores[k, st:st + m] + w.C_req <= sv.C_cores).all() \
                        and (mem[k, st:st + m] + w.M_req_MB <= sv.M_MB).all() \
                        and AV[k, st - 1:st - 1 + m].all():
                    fits.append((cores[k, st], s, m))
            if fits:
                _, s, m = min(fits)
                runs[w.i] = (s, st, m)
                cores[s - 1, st:st + m] += w.C_req
                mem[s - 1, st:st + m] += w.M_req_MB
                n += 1
                break
    return n


def _warm_start_values(scenario: Scenario, built: _Built, schedule: Schedule,
                       supply: np.ndarray | None, settings: RunSettings,
                       fill: bool = False) -> tuple[dict[int, float], int]:
    """Column values of the starting schedule, and how many workloads ``fill`` placed.

    With ``fill`` only the placements are given: the supply temperatures and cold-plate
    settings are left to HiGHS, which completes them for the placements. The previous plan's
    cooling (and the baseline's cold-plate flow) need not cool the workloads added to it,
    and a start with them fixed can be infeasible where the placements alone are not."""
    vals: dict[int, float] = {}
    runs: dict[int, tuple[int, int, int]] = {}      # i -> (server, start, run length)
    for p in schedule.placements:
        if p.server is None or p.start is None or p.end is None:
            continue
        wd = built.win.get((p.i, p.server))
        if wd is None or not (wd.a <= p.start <= wd.b):
            continue
        m = min(p.end - p.start, wd.n_max)       # a longer run is cut to n_max (never needed)
        if m < wd.n_min or p.start + m - 1 > wd.yb:
            continue
        runs[p.i] = (p.server, p.start, m)
    n_filled = _fill_warm_start(scenario, built, runs) if fill else 0
    for i, (server, start, m) in runs.items():
        wd = built.win[(i, server)]
        for s in range(1, scenario.S + 1):
            xc = built.X.get((i, s))
            if xc is None:
                continue
            on = s == server
            vals[xc] = 1.0 if on else 0.0
            for t, uc in built.U[(i, s)].items():
                vals[uc] = 1.0 if (on and t >= start) else 0.0
            for t, ec in built.E[(i, s)].items():
                vals[ec] = 1.0 if (on and start + wd.n_min <= t < start + m) else 0.0
    if fill:
        return vals, n_filled
    if built.cp_cols.b:
        lv = (built.cp_cols.fixed[0] if built.cp_cols.fixed is not None
              else scenario.cold_plate.level_of(settings.baseline_cp_flow_Lmin))
        for bk in built.cp_cols.b.values():
            for b in bk:
                for j, col in enumerate(b):
                    vals[col] = 1.0 if j == lv else 0.0
    if supply is not None and built.fixed_value is None:
        supply = np.asarray(supply, dtype=float)
        if supply.shape == (scenario.Z, scenario.T):
            for (z, t), cols in built.choice.items():
                v = supply[z - 1, t]
                k = int(round(v)) if abs(v - round(v)) < 1e-6 else None
                if k in built.levels:
                    for col, lv in zip(cols, built.levels):
                        vals[col] = 1.0 if lv == k else 0.0
    return vals, n_filled


def _solve_impl(scenario: Scenario, original_server: dict[int, int], settings: RunSettings, *,
                relax: tuple[str, ...], warm_start: Schedule | None,
                warm_start_supply_temp_C: np.ndarray | None, time_limit_s: float | None,
                log: bool, feasibility_only: bool,
                ignore_temperature: bool = False,
                extras: WindowExtras | None = None,
                warm_start_fill: bool = False) -> SolveResult:
    """See solve. With ``warm_start_fill`` the solution is also polished (_polish)."""
    relax = tuple(relax)
    t_build0 = time.perf_counter()
    built = _build(scenario, original_server, settings, relax, feasibility_only,
                   ignore_temperature, extras)
    notes = _model_notes(scenario, settings, built)
    built.notes[:] = notes
    if built.empty_workloads:
        res = _infeasible_result(built, relax, t_build0,
                                 "no feasible server/start window for workload(s) "
                                 + ", ".join(map(str, built.empty_workloads[:10])))
        res.notes.append("Detected before building the model: some workload has no server and start "
                         "period that meets arrival, wait limit, deadline and minimum run length.")
        return res
    lp = built.lp
    h = lp.to_highs(log)
    build_time = time.perf_counter() - t_build0

    limit = settings.time_limit_s if time_limit_s is None else time_limit_s
    h.setOptionValue("time_limit", float(limit))
    h.setOptionValue("mip_rel_gap", 1.0 if feasibility_only else float(settings.mip_rel_gap))
    lp_solver = "choose"
    if sum(lp.integer) > _IPM_BINARIES:
        lp_solver = "ipm"
        if h.setOptionValue("mip_lp_solver", "ipm") != highspy.HighsStatus.kOk:
            raise RuntimeError("HiGHS rejected mip_lp_solver = ipm")
        notes.append(f"More than {_IPM_BINARIES} binaries: the MIP's LP relaxations use interior point "
                     "(mip_lp_solver = ipm) because dual and primal simplex do not finish the root LP "
                     "on large models.")
    h.setOptionValue("log_to_console", bool(log))
    log_path = None
    n_ws_set = n_ws_filled = 0
    if warm_start is not None and not feasibility_only:
        vals, n_ws_filled = _warm_start_values(scenario, built, warm_start,
                                               warm_start_supply_temp_C, settings,
                                               fill=warm_start_fill)
        if vals:
            idx = np.array(sorted(vals), dtype=np.int32)
            h.setSolution(len(idx), idx, np.array([vals[k] for k in idx], dtype=np.float64))
            n_ws_set = len(idx)
            fd, log_path = tempfile.mkstemp(suffix=".log")
            os.close(fd)
            h.setOptionValue("log_file", log_path)

    t0 = time.perf_counter()
    h.run()
    solve_time = time.perf_counter() - t0
    ws_text = ""
    if log_path is not None:
        try:
            with open(log_path) as f:
                ws_text = f.read()
        finally:
            os.unlink(log_path)

    info = h.getInfo()
    ms = h.getModelStatus()
    text = h.modelStatusToString(ms)
    MS = highspy.HighsModelStatus
    has_sol = info.primal_solution_status == 2
    if ms == MS.kInfeasible or ms == MS.kUnboundedOrInfeasible:
        status = STATUS_INFEASIBLE
    elif ms == MS.kOptimal:
        status = STATUS_OPTIMAL
    elif has_sol:
        status = STATUS_TIME_LIMIT
    else:
        status = STATUS_NO_SOLUTION

    nv, nr = lp.n_cols, lp.n_rows
    nb = int(sum(lp.integer))
    out = SolveResult(status=status, schedule=None, supply_temp_C=None, objective_g=None,
                      best_bound_g=None, mip_gap=None, solve_time_s=solve_time,
                      n_variables=nv, n_binaries=nb, n_constraints=nr, relaxed=relax,
                      solver_status_text=text, notes=notes)
    out.model_values["build_time_s"] = build_time
    out.model_values["mip_lp_solver"] = lp_solver
    if n_ws_set:
        out.model_values["warm_start_columns_set"] = n_ws_set
        out.model_values["warm_start_filled"] = n_ws_filled
        out.model_values["warm_start_log"] = "\n".join(
            ln for ln in ws_text.splitlines() if "MIP start" in ln or "solution" in ln.lower())[:2000]
    if status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT) and has_sol:
        x = np.asarray(h.getSolution().col_value)
        out.objective_g = None if feasibility_only else float(info.objective_function_value)
        out.best_bound_g = None if feasibility_only else float(info.mip_dual_bound)
        out.mip_gap = None if feasibility_only else float(info.mip_gap)
        if warm_start_fill and not feasibility_only:
            x = _polish(h, lp, x, out)
        out.schedule = _extract_schedule(scenario, built, x)
        out.supply_temp_C = _extract_supply(scenario, built, x)
        S, Z, T = scenario.S, scenario.Z, scenario.T
        out.model_values["T_C"] = np.array([[x[c_] for c_ in row] for row in built.Tcol])
        Tm = out.model_values["T_C"]
        c = scenario.constants
        out.model_values["DPF"] = np.clip(1.0 - built.slope * (Tm - c.T_dvfs_C), 0.0, 1.0)
        out.model_values["Q_crac_kW"] = np.array([[x[c_] for c_ in row] for row in built.Qcrac])
        out.model_values["P_IT_W"] = np.array([[x[c_] for c_ in row] for row in built.P_IT])
        out.model_values["Q_cp_W"] = np.array([[x[c_] for c_ in row] for row in built.Q_cp])
        out.model_values["P_cp_W"] = np.array([[x[c_] for c_ in row] for row in built.P_cp])
        out.model_values["P_fan_kW"] = np.array([[x[c_] for c_ in row] for row in built.P_fan])
        if built.fixed_cop is not None:
            out.model_values["P_crac_kW"] = out.model_values["Q_crac_kW"] / built.fixed_cop
        else:
            pc = np.zeros((Z, T))
            for (z, t), qs in built.qcols.items():
                pc[z - 1, t] = sum(x[q] / cp_ for q, cp_ in zip(qs, built.cop_levels))
            out.model_values["P_crac_kW"] = pc
        out.cold_plate_ops = _extract_cold_plate_ops(scenario, built, x)
    elif status == STATUS_INFEASIBLE:
        pass
    return out


POLISH_TIME_LIMIT_S = 30.0


def _polish(h: highspy.Highs, lp, x: np.ndarray, out: SolveResult) -> np.ndarray:
    """Re-solve with every binary fixed at the solution's value, so the continuous columns
    (CRAC heat, coolant temperature, ...) take their cheapest values for those decisions. A
    solution HiGHS accepted within the gap can carry slack there, most often when it kept a
    completed warm start, and then the solver's carbon exceeds the evaluator's. The decisions,
    the bound and the status stay; the objective and gap are updated. Returns the polished
    column values, or ``x`` if the re-solve does not give a solution at least as good."""
    idx = np.flatnonzero(np.asarray(lp.integer, dtype=bool)).astype(np.int32)
    t0 = time.perf_counter()
    if len(idx):
        v = np.round(x[idx]).astype(np.float64)
        h.changeColsBounds(len(idx), idx, v, v)
    h.setOptionValue("time_limit", POLISH_TIME_LIMIT_S)
    h.run()
    out.model_values["polish_s"] = time.perf_counter() - t0
    info = h.getInfo()
    if info.primal_solution_status != 2:
        return x
    obj = float(info.objective_function_value)
    if obj > out.objective_g + 1e-9 * max(1.0, abs(out.objective_g)):
        return x
    out.model_values["polish_saved_g"] = out.objective_g - obj
    out.objective_g = obj
    if out.best_bound_g is not None and math.isfinite(out.best_bound_g):
        out.mip_gap = abs(obj - out.best_bound_g) / max(abs(obj), 1e-9)
    return np.asarray(h.getSolution().col_value)


def _extract_schedule(scenario: Scenario, built: _Built, x: np.ndarray) -> Schedule:
    chosen: dict[int, tuple[int, int, int]] = {}
    for (i, s), xc in built.X.items():
        if x[xc] <= 0.5:
            continue
        wd = built.win[(i, s)]
        # start = first period at which the started-by step is 1 (the last step is X itself)
        steps = sorted(t for t, uc in built.U[(i, s)].items() if x[uc] > 0.5)
        start = steps[0] if steps else wd.b
        tail = sum(1 for ec in built.E[(i, s)].values() if x[ec] > 0.5)
        chosen[i] = (s, start, start + wd.n_min + tail)
    pl = []
    for w in scenario.workloads:
        if w.i in chosen:
            s, st, en = chosen[w.i]
            pl.append(Placement(i=w.i, server=s, start=st, end=en))
        else:
            pl.append(Placement(i=w.i, server=None, start=None, end=None))
    return Schedule(placements=tuple(pl))


def _extract_supply(scenario: Scenario, built: _Built, x: np.ndarray) -> np.ndarray:
    Z, T = scenario.Z, scenario.T
    if built.fixed_value is not None:
        return np.full((Z, T), float(built.fixed_value))
    out = np.zeros((Z, T))
    for (z, t), cols in built.choice.items():
        k = int(np.argmax([x[col] for col in cols]))
        out[z - 1, t] = built.levels[k]
    return out


def _extract_cold_plate_ops(scenario: Scenario, built: _Built, x: np.ndarray) -> ColdPlateOps | None:
    if not built.cp_cols.b:
        return None
    flows = scenario.cold_plate.flow_settings_Lmin
    flow = np.full((scenario.S, scenario.T), np.nan)
    cool = np.full((scenario.S, scenario.T), np.nan)
    for k, bk in built.cp_cols.b.items():
        for t, b in enumerate(bk):
            flow[k, t] = flows[int(np.argmax([x[c_] for c_ in b]))]
            cool[k, t] = x[built.cp_cols.cool[k][t]]
    cp = scenario.cold_plate
    cool = np.clip(cool, cp.coolant_min_C, cp.coolant_max_C)    # solver round-off only
    return ColdPlateOps(flow_Lmin=flow, coolant_C=cool)


def solve(scenario: Scenario, original_server: dict[int, int], settings: RunSettings, *,
          relax: tuple[str, ...] = (), warm_start: Schedule | None = None,
          warm_start_supply_temp_C: np.ndarray | None = None,
          time_limit_s: float | None = None, log: bool = False,
          ignore_temperature: bool = False, extras: WindowExtras | None = None,
          warm_start_fill: bool = False) -> SolveResult:
    """Build and solve the MILP. See the module docstring for the model.

    ``ignore_temperature`` solves it without server temperature (no recursion, no thermal
    cap, DPF = 1); its schedule is not checked against the thermal limits. ``extras``: what a
    per-tick window adds, see WindowExtras. ``warm_start_fill``: workloads the warm start
    leaves open are placed by the baseline rule first (see _fill_warm_start)."""
    return _solve_impl(scenario, original_server, settings, relax=relax, warm_start=warm_start,
                       warm_start_supply_temp_C=warm_start_supply_temp_C,
                       time_limit_s=time_limit_s, log=log, feasibility_only=False,
                       ignore_temperature=ignore_temperature, extras=extras,
                       warm_start_fill=warm_start_fill)


def diagnose_infeasibility(scenario: Scenario, original_server: dict[int, int],
                           settings: RunSettings, *,
                           time_limit_s: float = 120.0,
                           extras: WindowExtras | None = None) -> dict[str, str]:
    """Re-solve with one constraint family relaxed at a time (first feasible point is enough).

    Returns {family: "feasible" | "infeasible" | "unknown (time limit)"} for all families.
    ``time_limit_s`` applies to each family separately.
    """
    settings.validate()
    out: dict[str, str] = {}
    for fam in RELAX_FAMILIES:
        if fam == RELAX_PRIORITY and not settings.priority:
            continue
        res = _solve_impl(scenario, original_server, settings, relax=(fam,), warm_start=None,
                          warm_start_supply_temp_C=None, time_limit_s=time_limit_s, log=False,
                          feasibility_only=True, extras=extras)
        if res.status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT):
            out[fam] = "feasible"
        elif res.status == STATUS_INFEASIBLE:
            out[fam] = "infeasible"
        else:
            out[fam] = "unknown (time limit)"
    return out


In [ ]:
%%writefile {LIB}/scheduler/evaluate.py
"""Shared evaluator: the referee that scores any schedule with the same code.

Given a Scenario, a Schedule, supply temperatures and (with cold plates on) the cold-plate
flow and coolant temperatures, it recomputes, from scratch, utilisation, IT power,
cold-plate heat and pump power, server temperature, DPF, work delivered, CRAC heat /
power, energy, carbon, and every compliance check. A schedule is never rejected; every
limit is measured and reported. Equation numbers refer to docs/PROJECT_CONTEXT.md.

This module must not import from baseline.py or optimize.py.
"""

from __future__ import annotations

import numpy as np

from .types import (
    ColdPlateOps, EvalResult, RunSettings, Scenario, Schedule, cold_plate_counts, cop,
    fixed_cold_plate_ops, priority_pairs,
)

TOL = 1e-6
W_TO_KWH_FACTOR = 1.0 / 3.6e6   # W * seconds -> kWh


# ---------------------------------------------------------------------------
# Slot that is not defined yet
# ---------------------------------------------------------------------------

def fan_power_kW(scenario: Scenario, Q_crac_kW: np.ndarray) -> np.ndarray:
    """Fan slot: returns P_fan[z, t] in kW. Airflow is not modelled, so zero."""
    return np.zeros_like(Q_crac_kW)


# ---------------------------------------------------------------------------
# Model pieces
# ---------------------------------------------------------------------------

def dpf(T_C, constants):
    """Eq. (37), all three branches as printed. Works on scalars or arrays."""
    T = np.asarray(T_C, dtype=float)
    c = constants
    middle = 1.0 - c.R_dvfs * (T - c.T_dvfs_C) / (c.T_throttle_C - c.T_dvfs_C)
    out = np.where(T < c.T_dvfs_C, 1.0, np.where(T < c.T_throttle_C, middle, c.F_throttle))
    return out if out.ndim else float(out)


def temperature_recursion(scenario: Scenario, supply_temp_C: np.ndarray,
                          P_IT_W: np.ndarray, Q_cp_W: np.ndarray) -> np.ndarray:
    """T[s,t] = alpha*T[s,t-1] + (1-alpha)*(Tsup[zone,t] + a + R*(P_IT - Q_cp))."""
    S, T = scenario.S, scenario.T
    temp = np.zeros((S, T))
    for sv in scenario.servers:
        k = sv.s - 1
        prev = sv.T0_C
        for t in range(T):
            steady = (supply_temp_C[sv.zone - 1, t] + sv.th_a_C
                      + sv.th_R_C_per_W * (P_IT_W[k, t] - Q_cp_W[k, t]))
            prev = sv.th_alpha * prev + (1.0 - sv.th_alpha) * steady
            temp[k, t] = prev
    return temp


def thermal_with_cold_plates(scenario: Scenario, supply_temp_C: np.ndarray, P_IT_W: np.ndarray,
                             plates: np.ndarray, ops: ColdPlateOps):
    """Temperature recursion with cold plates. Returns (T, Q_cp, P_cp), each (S, T) in degC / W.

    Per server and period, with n plates at flow setting k and coolant temperature c:
    Q_cp = min(n * Q_max, P_IT, n * G_k * max(0, T[s,t-1] - c)), the most heat physics and
    the limits allow, using the previous period's temperature (T0 in period 1). Pump power
    is n * pump_k. Servers without cold plates have Q_cp = P_cp = 0.
    """
    cp = scenario.cold_plate
    G = cp.conductance_W_per_C()
    pump = cp.pump_W()
    S, T = scenario.S, scenario.T
    temp = np.zeros((S, T))
    Q_cp = np.zeros((S, T))
    P_cp = np.zeros((S, T))
    for sv in scenario.servers:
        k = sv.s - 1
        n = int(plates[k])
        prev = sv.T0_C
        for t in range(T):
            if n > 0:
                lv = cp.level_of(float(ops.flow_Lmin[k, t]))
                drive = max(0.0, prev - float(ops.coolant_C[k, t]))
                Q_cp[k, t] = min(n * cp.Q_max_per_plate_W, P_IT_W[k, t], n * G[lv] * drive)
                P_cp[k, t] = n * pump[lv]
            steady = (supply_temp_C[sv.zone - 1, t] + sv.th_a_C
                      + sv.th_R_C_per_W * (P_IT_W[k, t] - Q_cp[k, t]))
            prev = sv.th_alpha * prev + (1.0 - sv.th_alpha) * steady
            temp[k, t] = prev
    return temp, Q_cp, P_cp


def _placement_arrays(scenario: Scenario, schedule: Schedule):
    """Per workload: placed flag, server (1-based), start, end (ints; 0 if unplaced)."""
    I, S = scenario.I, scenario.S
    placed = np.zeros(I, dtype=bool)
    srv = np.zeros(I, dtype=int)
    start = np.zeros(I, dtype=int)
    end = np.zeros(I, dtype=int)
    by_i = {p.i: p for p in schedule.placements}
    for w in scenario.workloads:
        p = by_i.get(w.i)
        if p is None or p.server is None or p.start is None or p.end is None:
            continue
        if not (1 <= p.server <= S):
            continue    # a server that does not exist is treated as not placed
        placed[w.i - 1] = True
        srv[w.i - 1], start[w.i - 1], end[w.i - 1] = p.server, p.start, p.end
    return placed, srv, start, end


def priority_violations(scenario: Scenario, placed, srv, start) -> list[tuple[int, int]]:
    """Pairs (i, j) of priority_pairs where j started on i's server while i was waiting:
    A_i <= ST_j < ST_i, Eq. (56)-(57)."""
    out = []
    for i, j in priority_pairs(scenario):
        a, b = i - 1, j - 1
        if not (placed[a] and placed[b]) or srv[a] != srv[b]:
            continue
        if scenario.workloads[a].A <= start[b] < start[a]:
            out.append((i, j))
    return out


def _f(x) -> float:
    return float(x)


def _none_if_nan(x):
    x = float(x)
    return None if np.isnan(x) else x


# ---------------------------------------------------------------------------
# Main entry point
# ---------------------------------------------------------------------------

def evaluate(scenario: Scenario, schedule: Schedule, supply_temp_C: np.ndarray,
             settings: RunSettings, cold_plate_ops: ColdPlateOps | None = None) -> EvalResult:
    """Score a schedule. ``cold_plate_ops`` gives the cold-plate flow and coolant temperature
    per server and period; None runs every cold plate at the baseline's fixed settings."""
    settings.validate()
    S, Z, T, I = scenario.S, scenario.Z, scenario.T, scenario.I
    c = scenario.constants
    dt = scenario.dt_seconds
    supply = np.asarray(supply_temp_C, dtype=float)
    if supply.shape != (Z, T):
        raise ValueError(f"supply_temp_C must have shape (Z, T) = {(Z, T)}, got {supply.shape}")
    cap = settings.thermal_cap_C(scenario)

    placed, srv, start, end = _placement_arrays(scenario, schedule)

    # Which periods does each placed workload occupy inside the horizon (0-based)?
    cpu_used = np.zeros((S, T))
    mem_used = np.zeros((S, T))
    run_idx: list[np.ndarray] = []
    for w in scenario.workloads:
        k = w.i - 1
        if not placed[k]:
            run_idx.append(np.zeros(0, dtype=int))
            continue
        lo = max(start[k], 1)
        hi = min(end[k] - 1, T)           # last occupied period (1-based), clipped
        idx = np.arange(lo - 1, hi) if hi >= lo else np.zeros(0, dtype=int)
        run_idx.append(idx)
        cpu_used[srv[k] - 1, idx] += w.C_req
        mem_used[srv[k] - 1, idx] += w.M_req_MB

    C_cores = np.array([sv.C_cores for sv in scenario.servers], dtype=float)
    M_MB = np.array([sv.M_MB for sv in scenario.servers], dtype=float)
    P_idle = np.array([sv.P_idle_W for sv in scenario.servers])
    P_max = np.array([sv.P_max_W for sv in scenario.servers])
    f_MHz = np.array([sv.f_MHz for sv in scenario.servers])
    zone_of = np.array([sv.zone for sv in scenario.servers], dtype=int)

    util = cpu_used / C_cores[:, None]                                    # Eq. (16)
    P_IT = P_idle[:, None] + (P_max - P_idle)[:, None] * util            # Eq. (20), (21)
    plates = cold_plate_counts(scenario, settings)
    has_cp = plates > 0
    if has_cp.any():
        ops = cold_plate_ops or fixed_cold_plate_ops(
            scenario, plates, settings.baseline_cp_flow_Lmin, settings.baseline_cp_coolant_C)
        for name in ("flow_Lmin", "coolant_C"):
            if np.shape(getattr(ops, name)) != (S, T):
                raise ValueError(f"cold_plate_ops.{name} must have shape (S, T) = {(S, T)}")
        temp, Q_cp, P_cp = thermal_with_cold_plates(scenario, supply, P_IT, plates, ops)
        cp_flow = np.where(has_cp[:, None], ops.flow_Lmin, np.nan)
        cp_coolant = np.where(has_cp[:, None], ops.coolant_C, np.nan)
    else:
        Q_cp, P_cp = np.zeros_like(P_IT), np.zeros_like(P_IT)
        temp = temperature_recursion(scenario, supply, P_IT, Q_cp)
        cp_flow = cp_coolant = np.full((S, T), np.nan)
    headroom = c.T_max_server_C - temp                                   # Eq. (31)
    DPF = dpf(temp, c)                                                   # Eq. (37)
    PC_eff = C_cores[:, None] * f_MHz[:, None] * DPF                     # Eq. (38)

    # Zones: heat balance at equality, divide by COP.
    Q_crac = np.zeros((Z, T))
    for z in range(1, Z + 1):
        members = zone_of == z
        Q_crac[z - 1] = (P_IT[members] - Q_cp[members]).sum(axis=0) / 1000.0
    COP = cop(c, supply)
    P_crac = Q_crac / COP
    P_fan = fan_power_kW(scenario, Q_crac)

    # Energy and carbon per period.
    E_IT = P_IT.sum(axis=0) * dt * W_TO_KWH_FACTOR
    E_cp = P_cp.sum(axis=0) * dt * W_TO_KWH_FACTOR
    E_crac = P_crac.sum(axis=0) * dt / 3600.0
    E_fan = P_fan.sum(axis=0) * dt / 3600.0
    E_total = E_IT + E_crac + E_cp + E_fan
    carbon = scenario.CI_g_per_kWh * E_total
    total_carbon = float(carbon.sum())

    # Per workload.
    nan = float("nan")
    wait = np.full(I, nan)
    delivered = np.full(I, nan)
    required = np.array([w.W_MHz_s for w in scenario.workloads], dtype=float)
    starts_ok = np.zeros(I, dtype=bool)
    wait_ok = np.zeros(I, dtype=bool)
    on_time = np.zeros(I, dtype=bool)
    work_ok = np.zeros(I, dtype=bool)
    within_horizon = np.zeros(I, dtype=bool)
    min_dpf_run = np.full(I, nan)
    for w in scenario.workloads:
        k = w.i - 1
        if not placed[k]:
            continue
        wait[k] = start[k] - w.A
        starts_ok[k] = start[k] >= w.A
        wait_ok[k] = wait[k] <= w.AD + TOL
        on_time[k] = end[k] <= w.D
        idx = run_idx[k]
        delivered[k] = float(w.C_req * f_MHz[srv[k] - 1] * dt * DPF[srv[k] - 1, idx].sum())
        if len(idx):
            min_dpf_run[k] = float(DPF[srv[k] - 1, idx].min())
        work_ok[k] = delivered[k] >= w.W_MHz_s * (1.0 - 1e-6)
        within_horizon[k] = start[k] >= 1 and end[k] <= T + 1 and end[k] > start[k]
    compliant = placed & starts_ok & wait_ok & on_time & work_ok & within_horizon

    # ---- compliance checks ------------------------------------------------
    cpu_ok = bool(np.all(cpu_used <= C_cores[:, None] + TOL))
    mem_ok = bool(np.all(mem_used <= M_MB[:, None] + TOL))
    thermal_cap_ok = bool(np.all(temp <= cap + TOL))
    workloads_ok = bool(np.all(compliant))
    supply_range_ok = bool(np.all(supply >= c.T_crac_min_C - TOL)
                           and np.all(supply <= c.T_crac_max_C + TOL))
    zone_temp_ok = bool(np.all(supply <= c.T_max_zone_C + TOL))

    def max_ramp(lag: int) -> float:
        if T <= lag:
            return 0.0
        return float(np.abs(supply[:, lag:] - supply[:, :-lag]).max())

    ramp_short = max_ramp(c.ramp_short_lag)
    ramp_long = max_ramp(c.ramp_long_lag)
    ramp_ok = bool(ramp_short <= c.ramp_short_C + TOL and ramp_long <= c.ramp_long_C + TOL)
    crac_capacity_ok = bool(np.all(Q_crac <= c.Q_crac_kW + TOL))
    cps = scenario.cold_plate
    cp_coolant_ok = bool(np.all((cp_coolant[has_cp] >= cps.coolant_min_C - TOL)
                                & (cp_coolant[has_cp] <= cps.coolant_max_C + TOL)))
    cp_flow_ok = bool(np.all((cp_flow[has_cp] >= cps.flow_min_Lmin - TOL)
                             & (cp_flow[has_cp] <= cps.flow_max_Lmin + TOL)))
    AV = scenario.availability()
    unavailable_used = (AV == 0) & (cpu_used > 0)
    availability_ok = not bool(unavailable_used.any())                   # Eq. (11)
    prio_bad = priority_violations(scenario, placed, srv, start)
    priority_ok = not prio_bad                                           # Eq. (56)-(57)

    above_dvfs = temp > c.T_dvfs_C
    above_cap = temp > cap + TOL

    servers_detail = []
    for sv in scenario.servers:
        k = sv.s - 1
        servers_detail.append({
            "s": sv.s, "name": sv.name, "zone": sv.zone, "cold_plate": int(sv.CT_cold_plate),
            "n_cold_plates": int(plates[k]),
            "Q_cp_peak_W": _f(Q_cp[k].max()), "Q_cp_capacity_W": _f(plates[k] * cps.Q_max_per_plate_W),
            "cpu_peak_cores": _f(cpu_used[k].max()), "cpu_capacity_cores": _f(sv.C_cores),
            "max_utilization": _f(util[k].max()),
            "mem_peak_MB": _f(mem_used[k].max()), "mem_capacity_MB": _f(sv.M_MB),
            "hottest_T_C": _f(temp[k].max()), "min_headroom_C": _f(headroom[k].min()),
            "periods_above_dvfs": int(above_dvfs[k].sum()),
            "periods_above_cap": int(above_cap[k].sum()),
            "min_DPF": _f(DPF[k].min()), "min_PC_eff_MHz": _f(PC_eff[k].min()),
            "periods_unavailable": int((AV[k] == 0).sum()),
        })
    cold_plates_detail = []
    for sv in scenario.servers:
        k = sv.s - 1
        if not has_cp[k]:
            continue
        cold_plates_detail.append({
            "s": sv.s, "name": sv.name, "n_cold_plates": int(plates[k]),
            "coolant_min_C": _f(cp_coolant[k].min()), "coolant_max_C": _f(cp_coolant[k].max()),
            "coolant_allowed_min_C": _f(cps.coolant_min_C),
            "coolant_allowed_max_C": _f(cps.coolant_max_C),
            "flow_min_Lmin": _f(cp_flow[k].min()), "flow_max_Lmin": _f(cp_flow[k].max()),
            "flow_allowed_min_Lmin": _f(cps.flow_min_Lmin),
            "flow_allowed_max_Lmin": _f(cps.flow_max_Lmin),
            "Q_cp_peak_W": _f(Q_cp[k].max()),
            "Q_cp_capacity_W": _f(plates[k] * cps.Q_max_per_plate_W),
        })
    zones_detail = []
    for z in range(1, Z + 1):
        zones_detail.append({
            "z": z,
            "supply_min_C": _f(supply[z - 1].min()), "supply_max_C": _f(supply[z - 1].max()),
            "supply_allowed_min_C": _f(c.T_crac_min_C), "supply_allowed_max_C": _f(c.T_crac_max_C),
            "zone_limit_C": _f(c.T_max_zone_C),
            "max_ramp_short_C": (float(np.abs(supply[z - 1, c.ramp_short_lag:]
                                              - supply[z - 1, :-c.ramp_short_lag]).max())
                                 if T > c.ramp_short_lag else 0.0),
            "ramp_short_limit_C": _f(c.ramp_short_C),
            "max_ramp_long_C": (float(np.abs(supply[z - 1, c.ramp_long_lag:]
                                             - supply[z - 1, :-c.ramp_long_lag]).max())
                                if T > c.ramp_long_lag else 0.0),
            "ramp_long_limit_C": _f(c.ramp_long_C),
            "Q_crac_peak_kW": _f(Q_crac[z - 1].max()), "Q_crac_capacity_kW": _f(c.Q_crac_kW),
        })

    checks = {"cpu_ok": cpu_ok, "mem_ok": mem_ok, "thermal_cap_ok": thermal_cap_ok,
              "workloads_ok": workloads_ok, "supply_range_ok": supply_range_ok,
              "zone_temp_ok": zone_temp_ok, "ramp_ok": ramp_ok,
              "crac_capacity_ok": crac_capacity_ok, "cold_plate_coolant_ok": cp_coolant_ok,
              "cold_plate_flow_ok": cp_flow_ok, "availability_ok": availability_ok,
              "priority_ok": priority_ok}
    # Priority is measured either way but is only a limit when the run enforces it.
    binding = {k: v for k, v in checks.items() if settings.priority or k != "priority_ok"}

    violations: list[str] = []
    if not cpu_ok:
        bad = [d for d in servers_detail if d["cpu_peak_cores"] > d["cpu_capacity_cores"] + TOL]
        violations.append("CPU capacity exceeded on server(s) "
                          + ", ".join(str(d["s"]) for d in bad) + " (Eq. 12)")
    if not mem_ok:
        bad = [d for d in servers_detail if d["mem_peak_MB"] > d["mem_capacity_MB"] + TOL]
        violations.append("memory capacity exceeded on server(s) "
                          + ", ".join(str(d["s"]) for d in bad) + " (Eq. 13)")
    if not thermal_cap_ok:
        violations.append(f"server temperature above thermal cap {cap:g} C in "
                          f"{int(above_cap.sum())} server-periods (Eq. 31-33), "
                          f"hottest {temp.max():.2f} C")
    if not workloads_ok:
        n_unp = int((~placed).sum())
        parts = []
        if n_unp:
            parts.append(f"{n_unp} unplaced")
        for label, flag in (("starts before arrival", ~starts_ok), ("wait exceeds allowed delay", ~wait_ok),
                            ("finishes after deadline", ~on_time), ("work short", ~work_ok),
                            ("runs outside horizon", ~within_horizon)):
            n = int((placed & flag).sum())
            if n:
                parts.append(f"{n} {label}")
        violations.append(f"{int((~compliant).sum())} non-compliant workload(s): " + "; ".join(parts))
    if not supply_range_ok:
        violations.append(f"supply temperature outside [{c.T_crac_min_C:g}, {c.T_crac_max_C:g}] C (Eq. 42)")
    if not zone_temp_ok:
        violations.append(f"zone temperature above {c.T_max_zone_C:g} C (Eq. 34)")
    if not ramp_ok:
        violations.append(f"supply ramp too steep: short {ramp_short:.2f} C (limit {c.ramp_short_C:g}), "
                          f"long {ramp_long:.2f} C (limit {c.ramp_long_C:g}) (Eq. 35-36)")
    if not crac_capacity_ok:
        violations.append(f"CRAC heat above capacity {c.Q_crac_kW:g} kW, peak {Q_crac.max():.2f} kW (Eq. 54)")
    if not cp_coolant_ok:
        violations.append(f"cold-plate coolant temperature outside [{cps.coolant_min_C:g}, "
                          f"{cps.coolant_max_C:g}] C (Eq. 49)")
    if not cp_flow_ok:
        violations.append(f"cold-plate flow outside [{cps.flow_min_Lmin:g}, {cps.flow_max_Lmin:g}] "
                          f"L/min (Eq. 47)")
    if not availability_ok:
        bad = sorted({int(k) + 1 for k in np.nonzero(unavailable_used)[0]})
        violations.append(f"work runs on unavailable server-periods ({int(unavailable_used.sum())}, "
                          f"server(s) {', '.join(map(str, bad))}) (Eq. 11)")
    if not priority_ok and settings.priority:
        ex = ", ".join(f"{scenario.workloads[j - 1].workload_id} before "
                       f"{scenario.workloads[i - 1].workload_id}" for i, j in prio_bad[:3])
        violations.append(f"{len(prio_bad)} priority inversion(s): a lower-priority workload "
                          f"started on a server while a higher-priority one waited for it, "
                          f"e.g. {ex} (Eq. 56-57)")

    compliance = {
        "all_ok": all(binding.values()), **checks,
        "servers": servers_detail, "zones": zones_detail, "cold_plates": cold_plates_detail,
        "priority_violations": [list(x) for x in prio_bad],
        "thermal_cap_C": _f(cap), "violations": violations,
        "within_horizon": [bool(x) for x in within_horizon],
    }

    # ---- totals -----------------------------------------------------------
    rt = np.array([w.RT for w in scenario.workloads])
    placed_rt = placed & (rt == 1)
    placed_dt = placed & (rt == 0)

    def mean_max(mask):
        if not mask.any():
            return None, None
        return float(wait[mask].mean()), float(wait[mask].max())

    mean_rt, max_rt = mean_max(placed_rt)
    mean_dt, max_dt = mean_max(placed_dt)
    deadline = np.array([w.D for w in scenario.workloads])
    late = (~placed) | (placed & (end > deadline))
    k_hot, t_hot = np.unravel_index(int(np.argmax(temp)), temp.shape)
    totals = {
        "E_IT_kWh": _f(E_IT.sum()), "E_crac_kWh": _f(E_crac.sum()),
        "E_cp_kWh": _f(E_cp.sum()), "E_fan_kWh": _f(E_fan.sum()),
        "Q_cp_kWh": _f(Q_cp.sum() * dt * W_TO_KWH_FACTOR),
        "E_total_kWh": _f(E_total.sum()),
        "carbon_g": total_carbon, "carbon_kg": total_carbon / 1000.0,
        "n_workloads": int(I), "n_unplaced": int((~placed).sum()),
        "n_late": int(late.sum()),
        "n_wait_exceeded": int((placed & ~wait_ok).sum()),
        "n_work_short": int((placed & ~work_ok).sum()),
        "n_noncompliant": int((~compliant).sum()),
        "mean_wait_RT": mean_rt, "max_wait_RT": max_rt,
        "mean_wait_DT": mean_dt, "max_wait_DT": max_dt,
        "hottest_T_C": _f(temp.max()), "hottest_server": int(k_hot + 1),
        "hottest_period": int(t_hot + 1),
        "min_headroom_C": _f(headroom.min()),
        "periods_above_dvfs": int(above_dvfs.sum()),
        "periods_above_cap": int(above_cap.sum()),
        "min_DPF": _f(DPF.min()),
        "n_throttled_servers": int((DPF < 1.0 - 1e-12).any(axis=1).sum()),
        "mean_T_C": _f(temp.mean()),
        "mean_headroom_C": _f(headroom.mean()),
        "n_priority_violations": len(prio_bad),
        "E_crac_by_zone_kWh": [_f(P_crac[z].sum() * dt / 3600.0) for z in range(Z)],
    }

    return EvalResult(
        cpu_used_cores=cpu_used, mem_used_MB=mem_used, util=util, P_IT_W=P_IT,
        Q_cp_W=Q_cp, P_cp_W=P_cp, T_C=temp, headroom_C=headroom, DPF=DPF, PC_eff_MHz=PC_eff,
        T_supply_C=supply.copy(), Q_crac_kW=Q_crac, COP=COP, P_crac_kW=P_crac, P_fan_kW=P_fan,
        E_IT_kWh=E_IT, E_crac_kWh=E_crac, E_cp_kWh=E_cp, E_fan_kWh=E_fan,
        E_total_kWh=E_total, carbon_g=carbon,
        wait_periods=wait, work_delivered_MHz_s=delivered, work_required_MHz_s=required,
        placed=placed, starts_on_or_after_arrival=starts_ok, wait_within_allowed=wait_ok,
        on_time=on_time, work_complete=work_ok, compliant=compliant, min_DPF_run=min_dpf_run,
        total_carbon_g=total_carbon, totals=totals, compliance=compliance,
        airflow_modelled=False,
        n_cold_plates=plates, cp_flow_Lmin=cp_flow, cp_coolant_C=cp_coolant,
    )


In [ ]:
%%writefile {LIB}/scheduler/report.py
"""Result writer: CSV tables, summary.json and the printed comparison table.

Computes no model numbers of its own: everything comes from EvalResult, SolveResult
and the inputs (apart from differences and percentages between the two columns, and
a count of workloads whose server differs from the original one).
"""

from __future__ import annotations

import csv
import json
import math
import platform
import subprocess
from datetime import datetime, timezone
from importlib import metadata
from pathlib import Path

import numpy as np

from .types import EvalResult, RunSettings, Scenario, Schedule, SolveResult
from .version import SCHEMA_VERSION, VERSION

AIRFLOW_TEXT = "not modelled"
AIRFLOW_NOTE = "Airflow and fan power are not modelled (fan power and airflow are zero)."
NO_COLD_PLATE_NOTE = ("Cold plates are off (cold-plate heat removal and pump power are zero; "
                      "--no-cold-plates was set).")
# Flagged with the thesis group: explain in the writeup, revisit after the first prototype.
COLD_PLATE_FLAG = ("FLAGGED: cold-plate flow and coolant temperature carry almost no cost (pump "
                   "power is at most about 1 mW per plate, and no equation charges energy for "
                   "cooling the water), so the optimizer runs them wherever they remove the most "
                   "heat. Where the 0.3 kW limit or the server's own heat is what limits removal, "
                   "the reported flow and coolant values are arbitrary.")


def model_note(settings: RunSettings) -> str:
    return AIRFLOW_NOTE + " " + (COLD_PLATE_FLAG if settings.cold_plates else NO_COLD_PLATE_NOTE)


def _cold_plate_block(scenario: Scenario, settings: RunSettings, ev: EvalResult) -> dict:
    cp = scenario.cold_plate
    return to_jsonable({
        "enabled": bool(settings.cold_plates),
        "plates_per_server": ev.n_cold_plates,
        "count_source": ("not modelled" if not settings.cold_plates else
                         "servers.csv" if settings.cold_plate_count is None else
                         f"--cold-plate-count {settings.cold_plate_count}"),
        "flow_settings_Lmin": list(cp.flow_settings_Lmin),
        "R_C_per_W": list(cp.R_C_per_W),
        "conductance_W_per_C": cp.conductance_W_per_C(),
        "pump_W_per_plate": cp.pump_W(),
        "flow_limits_Lmin": [cp.flow_min_Lmin, cp.flow_max_Lmin],
        "coolant_limits_C": [cp.coolant_min_C, cp.coolant_max_C],
        "Q_max_per_plate_W": cp.Q_max_per_plate_W,
        "baseline_flow_Lmin": settings.baseline_cp_flow_Lmin,
        "baseline_coolant_C": settings.baseline_cp_coolant_C,
        "optimized_mode": settings.cp_mode,
        "optimized_fixed_flow_Lmin": settings.optimized_fixed_cp()[0] if settings.cp_mode == "fixed" else None,
        "optimized_fixed_coolant_C": settings.optimized_fixed_cp()[1] if settings.cp_mode == "fixed" else None,
        "flag": COLD_PLATE_FLAG if settings.cold_plates else None,
    })


# ---------------------------------------------------------------------------
# JSON / CSV helpers
# ---------------------------------------------------------------------------

def to_jsonable(x):
    """Convert numpy scalars/arrays and tuples to plain Python; NaN and inf -> None."""
    if isinstance(x, dict):
        return {str(k): to_jsonable(v) for k, v in x.items()}
    if isinstance(x, (list, tuple, set)):
        return [to_jsonable(v) for v in x]
    if isinstance(x, np.ndarray):
        return to_jsonable(x.tolist())
    if isinstance(x, np.bool_):
        return bool(x)
    if isinstance(x, np.integer):
        return int(x)
    if isinstance(x, (float, np.floating)):
        v = float(x)
        return None if (math.isnan(v) or math.isinf(v)) else v
    return x


def _cell(v):
    if v is None:
        return ""
    if isinstance(v, (bool, np.bool_)):
        return int(bool(v))
    if isinstance(v, (float, np.floating)):
        return "" if math.isnan(float(v)) else repr(float(v))
    if isinstance(v, np.integer):
        return int(v)
    return v


def _write_csv(path: Path, header: list[str], rows) -> None:
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(header)
        for row in rows:
            w.writerow([_cell(v) for v in row])


def _type_name(RT: int) -> str:
    return "time_sensitive" if RT == 1 else "delay_tolerant"


# ---------------------------------------------------------------------------
# Per-schedule tables
# ---------------------------------------------------------------------------

def _count_reassigned(schedule: Schedule, original_server: dict) -> int:
    n = 0
    for p in schedule.placements:
        if p.server is not None and p.server != original_server.get(p.i):
            n += 1
    return n


def _count_cross_zone(scenario: Scenario, schedule: Schedule, original_server: dict) -> int:
    zone = {sv.s: sv.zone for sv in scenario.servers}
    return sum(1 for p in schedule.placements
               if p.server is not None and original_server.get(p.i) is not None
               and zone[p.server] != zone[original_server[p.i]])


def _move_reason(scenario: Scenario, ev: EvalResult, w, orig: int, final: int,
                 start: int, end: int) -> str:
    """Post-hoc reading of one move, from the scored schedule. The solver only minimises
    carbon; these are the local conditions that favoured the new server."""
    so, sf = scenario.servers[orig - 1], scenario.servers[final - 1]
    idx = np.arange(max(start, 1) - 1, min(end - 1, scenario.T))
    out = []
    if len(idx) and (np.any(ev.cpu_used_cores[orig - 1, idx] + w.C_req > so.C_cores + 1e-9)
                     or np.any(ev.mem_used_MB[orig - 1, idx] + w.M_req_MB > so.M_MB + 1e-9)):
        out.append("original server full at the chosen start")
    if sf.f_MHz > so.f_MHz:
        out.append("faster server")

    def it_energy(sv) -> float:   # extra IT energy of this workload on sv, W x periods
        n = max(1, math.ceil(w.W_MHz_s / (w.C_req * sv.f_MHz * scenario.dt_seconds) - 1e-9))
        return (sv.P_max_W - sv.P_idle_W) * w.C_req / sv.C_cores * n
    if it_energy(sf) < it_energy(so) - 1e-9:
        out.append("less IT energy for its work")
    if len(idx) and ev.T_C[final - 1, idx].mean() < ev.T_C[orig - 1, idx].mean() - 1e-9:
        out.append("cooler server")
    if len(idx) and sf.zone != so.zone and \
            ev.COP[sf.zone - 1, idx].mean() > ev.COP[so.zone - 1, idx].mean() + 1e-9:
        out.append("zone with higher CRAC COP")
    return "; ".join(out) or "carbon trade-off across the fleet (no single local cause)"


def _write_reassignment(path: Path, scenario: Scenario, settings: RunSettings,
                        schedule: Schedule, ev: EvalResult, original_server: dict) -> None:
    """One row per workload: original and final server and zone, and why it moved (or not)."""
    by_i = {p.i: p for p in schedule.placements}
    zone = {sv.s: sv.zone for sv in scenario.servers}
    rows = []
    for w in scenario.workloads:
        p = by_i.get(w.i)
        orig = original_server.get(w.i)
        placed = bool(ev.placed[w.i - 1]) and p is not None
        final = p.server if placed else None
        moved = placed and orig is not None and final != orig
        if not placed:
            reason = "not placed"
        elif moved:
            reason = _move_reason(scenario, ev, w, orig, final, p.start, p.end)
        elif w.B == 1:
            reason = "running at the start (B_i = 1): stays on its server, Eq. (29)"
        elif settings.reassignment == "none":
            reason = "reassignment off"
        else:
            reason = "kept"
        rows.append([
            w.i, w.workload_id, _type_name(w.RT), w.B, orig,
            zone.get(orig) if orig is not None else None, final,
            zone.get(final) if final is not None else None, moved,
            bool(moved and zone[orig] != zone[final]), reason])
    _write_csv(path, ["i", "workload_id", "type", "B_running", "original_server", "original_zone",
                      "final_server", "final_zone", "reassigned", "cross_zone", "reason"], rows)


def _write_schedule_tables(folder: Path, scenario: Scenario, schedule: Schedule,
                           ev: EvalResult, original_server: dict) -> None:
    folder.mkdir(parents=True, exist_ok=True)
    by_i = {p.i: p for p in schedule.placements}

    rows = []
    for w in scenario.workloads:
        k = w.i - 1
        p = by_i.get(w.i)
        placed = bool(ev.placed[k])
        orig = original_server.get(w.i)
        server = p.server if (p is not None and placed) else None
        start = p.start if (p is not None and placed) else None
        end = p.end if (p is not None and placed) else None
        rows.append([
            w.i, w.workload_id, _type_name(w.RT), w.A, orig, server,
            (server != orig) if placed else None, start,
            (end - start) if placed else None, end,
            ev.wait_periods[k], w.AD, w.D,
            ev.work_required_MHz_s[k], ev.work_delivered_MHz_s[k],
            ev.compliant[k], ev.placed[k], ev.starts_on_or_after_arrival[k],
            ev.wait_within_allowed[k], ev.on_time[k], ev.work_complete[k], w.B,
        ])
    _write_csv(folder / "schedule.csv",
               ["i", "workload_id", "type", "arrival", "original_server", "server", "reassigned",
                "start", "periods_run", "finish", "wait", "allowed_delay", "deadline",
                "work_required_MHz_s", "work_delivered_MHz_s", "compliant", "placed",
                "starts_on_or_after_arrival", "wait_within_allowed", "on_time", "work_complete",
                "B_running"],
               rows)

    T = scenario.T
    _write_csv(folder / "server_period.csv",
               ["s", "t", "cpu_used_cores", "mem_used_MB", "utilization", "P_IT_W", "Q_cp_W",
                "cp_flow_Lmin", "cp_coolant_C", "P_cp_W",
                "temperature_C", "headroom_C", "DPF", "PC_eff_MHz"],
               ([s + 1, t + 1, ev.cpu_used_cores[s, t], ev.mem_used_MB[s, t], ev.util[s, t],
                 ev.P_IT_W[s, t], ev.Q_cp_W[s, t], ev.cp_flow_Lmin[s, t], ev.cp_coolant_C[s, t],
                 ev.P_cp_W[s, t], ev.T_C[s, t], ev.headroom_C[s, t],
                 ev.DPF[s, t], ev.PC_eff_MHz[s, t]]
                for s in range(scenario.S) for t in range(T)))
    _write_csv(folder / "zone_period.csv",
               ["z", "t", "supply_temp_C", "Q_crac_kW", "COP", "P_crac_kW", "P_fan_kW", "airflow"],
               ([z + 1, t + 1, ev.T_supply_C[z, t], ev.Q_crac_kW[z, t], ev.COP[z, t],
                 ev.P_crac_kW[z, t], ev.P_fan_kW[z, t], AIRFLOW_TEXT]
                for z in range(scenario.Z) for t in range(T)))
    _write_csv(folder / "period.csv",
               ["t", "E_IT_kWh", "E_crac_kWh", "E_cp_kWh", "E_fan_kWh", "E_total_kWh",
                "CI_g_per_kWh", "carbon_g"],
               ([t + 1, ev.E_IT_kWh[t], ev.E_crac_kWh[t], ev.E_cp_kWh[t], ev.E_fan_kWh[t],
                 ev.E_total_kWh[t], scenario.CI_g_per_kWh[t], ev.carbon_g[t]] for t in range(T)))

    comp = ev.compliance
    srv_cols = ["s", "name", "zone", "cold_plate", "n_cold_plates", "Q_cp_peak_W", "Q_cp_capacity_W",
                "cpu_peak_cores", "cpu_capacity_cores", "max_utilization",
                "mem_peak_MB", "mem_capacity_MB", "hottest_T_C", "min_headroom_C",
                "periods_above_dvfs", "periods_above_cap", "min_DPF", "min_PC_eff_MHz",
                "periods_unavailable"]
    _write_csv(folder / "compliance_servers.csv", srv_cols,
               ([d[c] for c in srv_cols] for d in comp["servers"]))
    zone_cols = ["z", "supply_min_C", "supply_max_C", "supply_allowed_min_C",
                 "supply_allowed_max_C", "zone_limit_C", "max_ramp_short_C", "ramp_short_limit_C",
                 "max_ramp_long_C", "ramp_long_limit_C", "Q_crac_peak_kW", "Q_crac_capacity_kW"]
    _write_csv(folder / "compliance_zones.csv", zone_cols,
               ([d[c] for c in zone_cols] for d in comp["zones"]))
    cp_cols = ["s", "name", "n_cold_plates", "coolant_min_C", "coolant_max_C",
               "coolant_allowed_min_C", "coolant_allowed_max_C", "flow_min_Lmin", "flow_max_Lmin",
               "flow_allowed_min_Lmin", "flow_allowed_max_Lmin", "Q_cp_peak_W", "Q_cp_capacity_W"]
    _write_csv(folder / "compliance_cold_plates.csv", cp_cols,
               ([d[c] for c in cp_cols] for d in comp.get("cold_plates", [])))
    horizon = comp.get("within_horizon", [None] * scenario.I)
    _write_csv(folder / "compliance_workloads.csv",
               ["i", "workload_id", "placed", "work_delivered_MHz_s", "work_required_MHz_s",
                "min_DPF", "starts_on_or_after_arrival", "wait_within_allowed", "on_time",
                "work_complete", "within_horizon", "compliant"],
               ([w.i, w.workload_id, ev.placed[w.i - 1], ev.work_delivered_MHz_s[w.i - 1],
                 ev.work_required_MHz_s[w.i - 1], ev.min_DPF_run[w.i - 1],
                 ev.starts_on_or_after_arrival[w.i - 1],
                 ev.wait_within_allowed[w.i - 1], ev.on_time[w.i - 1], ev.work_complete[w.i - 1],
                 horizon[w.i - 1], ev.compliant[w.i - 1]] for w in scenario.workloads))


def _write_comparison_period(path: Path, scenario: Scenario, base: EvalResult,
                             opt: EvalResult | None) -> None:
    comps = ["E_IT_kWh", "E_crac_kWh", "E_cp_kWh", "E_fan_kWh", "E_total_kWh", "carbon_g"]
    header = (["t", "CI_g_per_kWh"] + [f"baseline_{c}" for c in comps]
              + [f"optimized_{c}" for c in comps] + [f"diff_{c}" for c in comps])
    rows = []
    for t in range(scenario.T):
        b = [getattr(base, c)[t] for c in comps]
        if opt is None:
            o = [None] * len(comps)
            d = [None] * len(comps)
        else:
            o = [getattr(opt, c)[t] for c in comps]
            d = [ov - bv for ov, bv in zip(o, b)]
        rows.append([t + 1, scenario.CI_g_per_kWh[t]] + b + o + d)
    _write_csv(path, header, rows)


# ---------------------------------------------------------------------------
# Summary
# ---------------------------------------------------------------------------

# (key, label, unit) in table order. Values come from EvalResult.totals.
METRICS = [
    ("carbon_kg", "Total carbon", "kg"),
    ("E_IT_kWh", "Energy, IT", "kWh"),
    ("E_crac_kWh", "Energy, CRAC", "kWh"),
    ("E_cp_kWh", "Energy, cold plate", "kWh"),
    ("Q_cp_kWh", "Heat removed by cold plates", "kWh"),
    ("E_fan_kWh", "Energy, fan", "kWh"),
    ("E_total_kWh", "Energy, total", "kWh"),
    ("n_late", "Workloads late", ""),
    ("n_noncompliant", "Non-compliant workloads", ""),
    ("mean_wait_RT", "Mean wait, time-sensitive", "periods"),
    ("max_wait_RT", "Max wait, time-sensitive", "periods"),
    ("mean_wait_DT", "Mean wait, delay-tolerant", "periods"),
    ("max_wait_DT", "Max wait, delay-tolerant", "periods"),
    ("n_priority_violations", "Priority inversions", ""),
    ("hottest_T_C", "Hottest server temperature", "C"),
    ("mean_T_C", "Mean server temperature", "C"),
    ("min_headroom_C", "Minimum headroom", "C"),
    ("mean_headroom_C", "Mean headroom", "C"),
    ("periods_above_dvfs", "Server-periods above DVFS temperature", ""),
    ("periods_above_cap", "Server-periods above thermal cap", ""),
    ("n_throttled_servers", "Servers throttled (DPF < 1)", ""),
    ("n_reassigned", "Workloads reassigned", ""),
    ("n_cross_zone", "Reassigned to another zone", ""),
]
_SCHEDULE_METRICS = ("n_reassigned", "n_cross_zone")


def _metrics_block(scenario: Scenario, ev: EvalResult, schedule: Schedule,
                   original_server: dict) -> dict:
    t = ev.totals
    block = {k: t.get(k) for k, _, _ in METRICS if k not in _SCHEDULE_METRICS}
    block["n_reassigned"] = _count_reassigned(schedule, original_server)
    block["n_cross_zone"] = _count_cross_zone(scenario, schedule, original_server)
    block["hottest_server"] = t.get("hottest_server")
    block["hottest_period"] = t.get("hottest_period")
    block["n_unplaced"] = t.get("n_unplaced")
    block["min_DPF"] = t.get("min_DPF")
    return to_jsonable(block)


def _side(scenario: Scenario, ev: EvalResult, schedule: Schedule, original_server: dict) -> dict:
    return {
        "valid": bool(ev.valid),
        "violations": list(ev.compliance.get("violations", [])),
        "metrics": _metrics_block(scenario, ev, schedule, original_server),
        "totals": to_jsonable(ev.totals),
        "compliance_flags": to_jsonable({k: v for k, v in ev.compliance.items()
                                         if k.endswith("_ok") or k == "all_ok"}),
    }


def _diff(b, o):
    if b is None or o is None:
        return {"baseline": b, "optimized": o, "diff": None, "pct": None}
    d = o - b
    pct = None if b == 0 else 100.0 * d / b
    return {"baseline": b, "optimized": o, "diff": d, "pct": pct}


def _solver_block(sr: SolveResult | None) -> dict | None:
    if sr is None:
        return None
    return to_jsonable({
        "status": sr.status, "solver_status_text": sr.solver_status_text,
        "solve_time_s": sr.solve_time_s, "mip_gap": sr.mip_gap,
        "objective_g": sr.objective_g, "best_bound_g": sr.best_bound_g,
        "n_variables": sr.n_variables, "n_binaries": sr.n_binaries,
        "n_constraints": sr.n_constraints, "relaxed": list(sr.relaxed),
        "has_schedule": sr.has_schedule, "notes": list(sr.notes),
    })


def _git_commit() -> str | None:
    """Commit of the code that made the run, with "+dirty" for uncommitted changes; None
    outside a git checkout (the notebooks run a copy of the code)."""
    here = Path(__file__).resolve().parent
    try:
        commit = subprocess.run(["git", "-C", str(here), "rev-parse", "--short", "HEAD"],
                                capture_output=True, text=True, timeout=5)
        if commit.returncode != 0:
            return None
        dirty = subprocess.run(["git", "-C", str(here), "status", "--porcelain",
                                "--untracked-files=no", "--", "."],
                               capture_output=True, text=True, timeout=5).stdout.strip()
    except (OSError, subprocess.SubprocessError):
        return None
    return commit.stdout.strip() + ("+dirty" if dirty else "")


def _package_version(name: str) -> str | None:
    try:
        return metadata.version(name)
    except metadata.PackageNotFoundError:
        return None


def provenance(scenario: Scenario, made_with: str) -> dict:
    """What made a run: the release, the scenario build it was solved on, and the solver.
    Two runs are comparable when their scenario fingerprints are equal."""
    return {
        "schema_version": SCHEMA_VERSION,
        "thermasched_version": VERSION,
        "made_with": made_with,
        "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
        "git_commit": _git_commit(),
        "scenario_fingerprint": scenario.fingerprint,
        "scenario_built_with": scenario.built_with,
        "highspy": _package_version("highspy"),
        "python": platform.python_version(),
    }


def write_results(out_dir, scenario: Scenario, settings: RunSettings,
                  original_server: dict, baseline_schedule: Schedule, baseline_eval: EvalResult,
                  optimized_schedule: Schedule | None, optimized_eval: EvalResult | None,
                  solve_result: SolveResult | None, infeasibility: dict | None,
                  checks: dict, made_with: str = "command line",
                  per_tick: dict | None = None) -> dict:
    """``per_tick``: the per-tick block (scheduler/pertick.py) of a per-tick run."""
    out = Path(out_dir)
    out.mkdir(parents=True, exist_ok=True)

    _write_schedule_tables(out / "baseline", scenario, baseline_schedule, baseline_eval,
                           original_server)
    have_opt = optimized_schedule is not None and optimized_eval is not None
    if have_opt:
        _write_schedule_tables(out / "optimized", scenario, optimized_schedule, optimized_eval,
                               original_server)
        _write_reassignment(out / "optimized" / "reassignment.csv", scenario, settings,
                            optimized_schedule, optimized_eval, original_server)
    _write_comparison_period(out / "comparison_period.csv", scenario, baseline_eval,
                             optimized_eval if have_opt else None)

    base_side = _side(scenario, baseline_eval, baseline_schedule, original_server)
    opt_side = (_side(scenario, optimized_eval, optimized_schedule, original_server)
                if have_opt else None)
    comparison = {}
    for key, _, _ in METRICS:
        b = base_side["metrics"].get(key)
        o = opt_side["metrics"].get(key) if opt_side else None
        comparison[key] = _diff(b, o)

    summary = {
        "scenario": {"name": scenario.name, "I": scenario.I, "S": scenario.S, "Z": scenario.Z,
                     "T": scenario.T, "dt_seconds": scenario.dt_seconds,
                     "zones": [sv.zone for sv in scenario.servers],
                     "n_running": sum(w.B for w in scenario.workloads),
                     "n_unavailable_server_periods": int((scenario.availability() == 0).sum()),
                     "CI_mean_g_per_kWh": float(np.mean(scenario.CI_g_per_kWh))},
        "provenance": provenance(scenario, made_with),
        "settings": settings.as_dict(),
        "mode": "per_tick" if settings.per_tick else "whole_day",
        "per_tick": per_tick,
        "model_note": model_note(settings),
        "cold_plates": _cold_plate_block(scenario, settings, baseline_eval),
        "airflow_modelled": bool(baseline_eval.airflow_modelled),
        "thermal_cap_C": baseline_eval.compliance.get("thermal_cap_C"),
        "T_dvfs_C": scenario.constants.T_dvfs_C,
        "baseline": base_side,
        "optimized": opt_side,
        "has_optimized_schedule": have_opt,
        "comparison": comparison,
        "solver": _solver_block(solve_result),
        "infeasibility": infeasibility,
        "checks": checks,
    }
    summary = to_jsonable(summary)
    with open(out / "summary.json", "w") as fh:
        json.dump(summary, fh, indent=2, allow_nan=False)
    return summary


# ---------------------------------------------------------------------------
# Printed table
# ---------------------------------------------------------------------------

def _fmt(v, unit_key: str | None = None) -> str:
    if v is None:
        return "-"
    if isinstance(v, bool):
        return str(v)
    if isinstance(v, int):
        return str(v)
    a = abs(v)
    if a != 0 and a < 0.01:
        return f"{v:.2e}"
    return f"{v:,.2f}"


def _fmt_pct(p) -> str:
    return "-" if p is None else f"{p:+.1f}%"


def _fmt_diff(d) -> str:
    if d is None:
        return "-"
    if isinstance(d, int):
        return f"{d:+d}"
    return f"{d:+,.2f}" if (d == 0 or abs(d) >= 0.01) else f"{d:+.2e}"


def format_summary(summary: dict) -> str:
    sc = summary["scenario"]
    st = summary["settings"]
    lines = []
    lines.append(f"Scenario {sc['name']}: I={sc['I']} workloads, S={sc['S']} servers, "
                 f"Z={sc['Z']} zones, T={sc['T']} periods of {sc['dt_seconds']:g} s")
    lines.append(f"Run '{st.get('run_name')}': baseline={st.get('baseline_rule')} at "
                 f"{st.get('baseline_supply_temp_C')} C, reassignment={st.get('reassignment')}, "
                 f"placement={st.get('placement', 'thermal_aware')}, "
                 f"supply_temp_mode={st.get('supply_temp_mode')}, "
                 f"priority={'on' if st.get('priority', True) else 'off'}, "
                 f"thermal cap {summary.get('thermal_cap_C')} C")
    pv = summary.get("provenance") or {}
    if pv:
        lines.append(f"Made with ThermaSched {pv.get('thermasched_version')} ({pv.get('made_with')}"
                     + (f", commit {pv['git_commit']}" if pv.get("git_commit") else "")
                     + f"), scenario fingerprint {pv.get('scenario_fingerprint')}")
    extra = []
    if st.get("zone_map"):
        extra.append(f"zone map {st['zone_map']}")
    if st.get("carbon_file"):
        extra.append(f"carbon intensity from {st['carbon_file']}")
    if st.get("ci_flat"):
        extra.append(f"flat carbon intensity {sc.get('CI_mean_g_per_kWh', 0):.2f} g/kWh")
    if extra:
        lines.append("Scenario overrides: " + "; ".join(extra))
    cpb = summary.get("cold_plates") or {}
    if cpb.get("enabled"):
        lines.append(f"Cold plates per server {cpb.get('plates_per_server')} "
                     f"({cpb.get('count_source')}); flow settings {cpb.get('flow_settings_Lmin')} L/min; "
                     f"baseline fixed at {cpb.get('baseline_flow_Lmin')} L/min, "
                     f"{cpb.get('baseline_coolant_C')} C coolant"
                     + (f"; optimized fixed at {cpb.get('optimized_fixed_flow_Lmin')} L/min, "
                        f"{cpb.get('optimized_fixed_coolant_C')} C coolant"
                        if cpb.get("optimized_mode") == "fixed" else ""))
    lines.append("")

    have_opt = summary.get("has_optimized_schedule")
    rows = [("Metric", "Baseline", "Optimized", "Diff", "Diff %")]
    comp = summary["comparison"]
    for key, label, unit in METRICS:
        c = comp[key]
        name = f"{label} ({unit})" if unit else label
        if key == "periods_above_dvfs":
            name = f"Server-periods above {summary.get('T_dvfs_C'):g} C"
        b_txt = _fmt(c["baseline"])
        o_txt = _fmt(c["optimized"]) if have_opt else "-"
        if key == "hottest_T_C":
            b_txt += f" (server {summary['baseline']['metrics'].get('hottest_server')})"
            if have_opt:
                o_txt += f" (server {summary['optimized']['metrics'].get('hottest_server')})"
        rows.append((name, b_txt, o_txt,
                     _fmt_diff(c["diff"]) if have_opt else "-",
                     _fmt_pct(c["pct"]) if have_opt else "-"))
    rows.append(("Schedule valid (no limit broken)", str(summary["baseline"]["valid"]),
                 str(summary["optimized"]["valid"]) if have_opt else "-", "", ""))

    widths = [max(len(r[i]) for r in rows) for i in range(5)]
    for n, r in enumerate(rows):
        lines.append("  ".join(r[0].ljust(widths[0]) if i == 0 else r[i].rjust(widths[i])
                               for i in range(5)))
        if n == 0:
            lines.append("  ".join("-" * w for w in widths))

    for side, label in (("baseline", "Baseline"), ("optimized", "Optimized")):
        d = summary.get(side)
        if d and d["violations"]:
            lines.append("")
            lines.append(f"{label} violations:")
            lines.extend(f"  - {v}" for v in d["violations"])

    sv = summary.get("solver")
    lines.append("")
    if sv is None:
        lines.append("Solver: not run.")
    else:
        gap = sv.get("mip_gap")
        lines.append(f"Solver: status={sv['status']} ({sv.get('solver_status_text') or '-'}), "
                     f"time={sv['solve_time_s']:.1f} s, gap={'-' if gap is None else f'{gap:.4%}'}, "
                     f"objective={_fmt(sv.get('objective_g'))} g, bound={_fmt(sv.get('best_bound_g'))} g")
        lines.append(f"        {sv['n_variables']} variables, {sv['n_binaries']} binaries, "
                     f"{sv['n_constraints']} constraints")
        for note in sv.get("notes") or []:
            lines.append(f"        note: {note}")
    pt = summary.get("per_tick")
    if pt:
        slow = pt.get("slowest_tick_solve_s")
        lines.append(f"Per-tick re-planning: {pt['n_ticks']} ticks, {pt['n_ticks_optimal']} "
                     f"optimal, {pt['n_ticks_time_limit']} at the {pt['tick_time_limit_s']:g} s "
                     f"tick limit, slowest {'-' if slow is None else f'{slow:.1f}'} s "
                     f"(tick {pt.get('slowest_tick')}); log in {pt['ticks_file']}")
        if pt.get("stuck"):
            st = pt["stuck"]
            lines.append(f"STUCK at tick {st['tick']}: {st['limit_text']}. {st['detail']}.")
    if not have_opt:
        lines.append("")
        lines.append("NO VALID SCHEDULE WAS FOUND by the optimizer; only the baseline is shown.")
    inf = summary.get("infeasibility")
    if inf:
        lines.append("Infeasibility diagnosis (one constraint family relaxed at a time):")
        for fam, outcome in inf.items():
            lines.append(f"  {fam}: {outcome if outcome is not None else '-'}")
    lines.append("")
    lines.append("Note: " + summary["model_note"])
    return "\n".join(lines)


In [ ]:
%%writefile {LIB}/scheduler/pertick.py
"""Per-tick re-planning (docs/PLAN.md section 2, option 2A).

At every tick t (one period) the optimizer of optimize.py solves a window that runs from t to
the end of the horizon. The window holds:

- the workloads that have arrived (A_i <= t) and not started, with their arrival, deadline and
  allowed delay measured from t; workloads that arrive later are not in it;
- the workloads already running, on their server and with their start fixed (they occupy the
  window from its first period), counting only the work they have left;
- each server's temperature at the end of period t - 1 as T0 (Eq. 30);
- the supply temperatures of the last periods as fixed history, so the ramp limits (35)-(36)
  hold across ticks;
- the carbon intensity of every remaining period.

Only period t's decisions are kept: which workloads start at t and on which server, the supply
temperature of each zone and the cold-plate settings. The rest of the plan is offered to the
next tick as a starting solution and otherwise discarded; the workloads it does not place (those
that arrived since) are added to that start by the baseline rule, using only arrived workloads,
so every tick starts from a complete schedule. A tick that still has no schedule at the tick
time limit is solved again, up to the no-schedule limit in all. Period t is then played out with the
same physics as the evaluator: the server temperatures, and for every running workload the work
delivered at the DPF it actually ran at. A running workload ends once its work is done.

A tick with no valid schedule stops the run and names the limit; nothing is relaxed. The state
is saved after every tick in ``<run folder>/checkpoint/``, so a stopped run resumes where it
stopped and gives the same schedule.
"""

from __future__ import annotations

import csv
import dataclasses
import json
import math
import os
import shutil
import time
from dataclasses import dataclass, field, replace
from pathlib import Path

import numpy as np

from .evaluate import dpf
from .optimize import WindowExtras, _build, _solve_impl, diagnose_infeasibility, solve
from .types import (
    RELAX_CAPACITY, RELAX_COOLING, RELAX_DEADLINES, RELAX_PRIORITY, RELAX_THERMAL,
    STATUS_INFEASIBLE, STATUS_NO_SOLUTION, STATUS_OPTIMAL, STATUS_TIME_LIMIT, ColdPlateOps,
    Placement, RunSettings, Scenario, Schedule, SolveResult, cold_plate_counts, priority_pairs,
)

CHECKPOINT_DIR = "checkpoint"
STATE_FILE = "state.json"
TICKS_FILE = "ticks.csv"
PARTIAL_FILE = "partial_schedule.csv"
CHECKPOINT_FORMAT = 1
DONE_REL_TOL = 1e-7         # a running workload ends once it has delivered W * (1 - this)
BLIND_SHARE = 0.7           # thermal-blind placement: share of the tick limit for pass 1

OUTCOME_DONE = "done"
OUTCOME_STOPPED = "stopped"     # wall-time limit: resume by running the same command again
OUTCOME_STUCK = "stuck"         # a tick has no valid schedule

# Limits a stuck tick can name.
LIMIT_SUPPLY_RAMP = "supply_ramp"
LIMIT_TICK_TIME = "tick_time_limit"
LIMIT_COMBINATION = "several_limits_together"
LIMIT_MODEL = "model_input"
LIMIT_TEXT = {
    RELAX_DEADLINES: "deadlines and allowed waiting (Eq. 27-28)",
    RELAX_THERMAL: "thermal cap T <= T_max - H_min (Eq. 31-33)",
    RELAX_CAPACITY: "server CPU and memory capacity (Eq. 12-13)",
    RELAX_COOLING: "CRAC cooling capacity (Eq. 54)",
    RELAX_PRIORITY: "priority (Eq. 56-57)",
    LIMIT_SUPPLY_RAMP: "supply-air ramp limits across ticks (Eq. 35-36)",
    LIMIT_TICK_TIME: "no schedule found within the tick time limit",
    LIMIT_COMBINATION: "no single limit: only several relaxed together would allow a schedule",
    LIMIT_MODEL: "the window's inputs are outside what the model accepts",
}

TICK_COLUMNS = ["tick", "status", "solve_s", "mip_gap", "n_waiting", "n_started", "n_running",
                "n_arrived", "n_done", "objective_g", "kept_carbon_g", "n_binaries",
                "n_constraints", "tick_wall_s", "extended", "warm_start_filled"]


@dataclass
class PerTickOutcome:
    """What run_per_tick returns. ``schedule`` and the arrays are None unless status is done."""
    status: str
    ticks: list[dict]
    schedule: Schedule | None = None
    supply_temp_C: np.ndarray | None = None
    cold_plate_ops: ColdPlateOps | None = None
    result: SolveResult | None = None
    stuck: dict | None = None
    resumed_from_tick: int | None = None
    next_tick: int | None = None


@dataclass
class _State:
    """Everything the next tick needs; saved as JSON after every tick."""
    next_tick: int
    temp: list[float]                       # (S,) server temperature at the end of next_tick - 1
    supply: list[list[float]]               # (Z, next_tick - 1) kept supply temperatures
    cp_flow: list[list[float | None]]       # (S, next_tick - 1), None without cold plates
    cp_coolant: list[list[float | None]]
    server: dict[int, int] = field(default_factory=dict)
    start: dict[int, int] = field(default_factory=dict)
    end: dict[int, int] = field(default_factory=dict)
    delivered: dict[int, float] = field(default_factory=dict)
    plan: dict[int, list[int]] | None = None        # last window's plan, original periods
    plan_supply: list[list[float]] | None = None    # last window's supply, (Z, its T)
    ticks: list[dict] = field(default_factory=list)
    tick_status: dict[str, int] = field(default_factory=dict)
    notes: list[str] = field(default_factory=list)  # model notes of the first tick
    max_size: list[int] = field(default_factory=lambda: [0, 0, 0])

    def to_json(self) -> dict:
        d = dataclasses.asdict(self)
        for k in ("server", "start", "end", "delivered", "plan"):
            if d[k] is not None:
                d[k] = {str(i): v for i, v in d[k].items()}
        return d

    @classmethod
    def from_json(cls, d: dict) -> "_State":
        d = dict(d)
        for k in ("server", "start", "end", "delivered", "plan"):
            if d.get(k) is not None:
                d[k] = {int(i): v for i, v in d[k].items()}
        return cls(**d)


# ---------------------------------------------------------------------------
# Checkpoint
# ---------------------------------------------------------------------------

def _plain(x):
    """JSON round trip, so settings compare equal whether fresh or read back."""
    from .report import to_jsonable
    return json.loads(json.dumps(to_jsonable(x)))


def _write_json_atomic(path: Path, data: dict) -> None:
    tmp = path.with_suffix(path.suffix + ".tmp")
    with open(tmp, "w") as fh:
        json.dump(data, fh, allow_nan=False)
    os.replace(tmp, path)


def _save_checkpoint(folder: Path, settings: RunSettings, scenario: Scenario,
                     state: _State) -> None:
    folder.mkdir(parents=True, exist_ok=True)
    _write_json_atomic(folder / STATE_FILE, {
        "format": CHECKPOINT_FORMAT, "settings": _plain(settings.as_dict()),
        "scenario_fingerprint": scenario.fingerprint, "state": state.to_json()})


def _load_checkpoint(folder: Path, settings: RunSettings, scenario: Scenario) -> _State | None:
    path = folder / STATE_FILE
    if not path.is_file():
        return None
    with open(path) as fh:
        data = json.load(fh)
    if data.get("format") != CHECKPOINT_FORMAT:
        raise ValueError(f"{path}: checkpoint format {data.get('format')} is not "
                         f"{CHECKPOINT_FORMAT}; delete the folder to start again")
    if data.get("settings") != _plain(settings.as_dict()):
        theirs, ours = data.get("settings") or {}, _plain(settings.as_dict())
        diff = sorted(k for k in set(theirs) | set(ours) if theirs.get(k) != ours.get(k))
        raise ValueError(f"{path}: the checkpoint was made with other settings ({', '.join(diff)}); "
                         "run the same command, or delete the folder to start again")
    if data.get("scenario_fingerprint") != scenario.fingerprint:
        raise ValueError(f"{path}: the checkpoint was made on another build of scenario "
                         f"{scenario.name}; delete the folder to start again")
    return _State.from_json(data["state"])


def _write_ticks(path: Path, ticks: list[dict]) -> None:
    from .report import _cell
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(TICK_COLUMNS)
        for r in ticks:
            w.writerow([_cell(r.get(c)) for c in TICK_COLUMNS])


# ---------------------------------------------------------------------------
# One tick
# ---------------------------------------------------------------------------

class _Tick:
    """The window of tick t: its scenario and the maps between window and day numbers."""

    def __init__(self, scenario: Scenario, original: dict[int, int], state: _State, t: int,
                 pairs_by_i: dict[int, list[int]], pairs: list[tuple[int, int]],
                 settings: RunSettings):
        self.t = t
        running = sorted(i for i in state.start if i not in state.end)
        waiting = sorted(w.i for w in scenario.workloads
                         if w.A <= t and w.i not in state.start)
        self.running, self.waiting = running, waiting
        self.ids = sorted(running + waiting)              # window workload k is ids[k - 1]
        self.k_of = {i: k for k, i in enumerate(self.ids, start=1)}
        run_set = set(running)
        wl, orig = [], {}
        for k, i in enumerate(self.ids, start=1):
            w = scenario.workloads[i - 1]
            if i in run_set:
                # running: occupies the window from its first period, on its server, with the
                # work it has left; no wait limit applies any more
                wl.append(replace(w, i=k, A=1, AD=0, D=w.D - t + 1, B=1,
                                  W_MHz_s=w.W_MHz_s - state.delivered[i]))
                orig[k] = state.server[i]
            else:
                # waiting: the latest start A + AD stays where it was
                wl.append(replace(w, i=k, A=1, AD=w.A + w.AD - t, D=w.D - t + 1))
                orig[k] = original[i]
        self.original = orig
        servers = tuple(replace(sv, T0_C=float(state.temp[sv.s - 1])) for sv in scenario.servers)
        av = None if scenario.AV is None else np.asarray(scenario.AV)[:, t - 1:]
        self.scenario = replace(
            scenario, name=f"{scenario.name}@tick{t}", T=scenario.T - t + 1,
            workloads=tuple(wl), servers=servers,
            CI_g_per_kWh=np.asarray(scenario.CI_g_per_kWh, dtype=float)[t - 1:], AV=av,
            fingerprint=None)

        # Priority: pairs between waiting workloads keep their whole-day order. A waiting i may
        # not use the server of a started j that started while i was waiting (A_i <= ST_j),
        # or j would have overtaken it there.
        wait_set = set(waiting)
        win_pairs = tuple((self.k_of[i], self.k_of[j]) for i, j in pairs
                          if i in wait_set and j in wait_set)
        forbidden: dict[int, frozenset[int]] = {}
        if settings.priority:
            for i in waiting:
                A_i = scenario.workloads[i - 1].A
                bad = {state.server[j] for j in pairs_by_i.get(i, ())
                       if j in state.start and A_i <= state.start[j]}
                if bad:
                    forbidden[self.k_of[i]] = frozenset(bad)
        hist = None
        if state.supply and len(state.supply[0]):
            c = scenario.constants
            h = min(max(c.ramp_short_lag, c.ramp_long_lag), len(state.supply[0]))
            hist = np.asarray(state.supply, dtype=float)[:, -h:]
        self.extras = WindowExtras(priority_pairs=win_pairs, forbidden=forbidden or None,
                                   supply_history=hist)

    def warm_start(self, state: _State):
        """The previous tick's plan in window numbers; workloads it does not place (new
        arrivals, and at tick 1 every workload) are left open here and placed by the baseline
        rule in the solver (optimize._fill_warm_start)."""
        if state.plan is None:
            return Schedule(()), None
        run_set = set(self.running)
        pl = []
        for k, i in enumerate(self.ids, start=1):
            p = state.plan.get(i)
            if p is None:
                pl.append(Placement(k, None, None, None))
                continue
            s, st, en = p
            st = 1 if i in run_set else st - self.t + 1
            en = en - self.t + 1
            pl.append(Placement(k, s, st, en) if st >= 1 and en > st
                      else Placement(k, None, None, None))
        sup = None
        if state.plan_supply is not None:
            arr = np.asarray(state.plan_supply, dtype=float)
            if arr.shape[1] == self.scenario.T + 1:
                sup = arr[:, 1:]
        return Schedule(tuple(pl)), sup


def _solve_tick(tick: _Tick, settings: RunSettings, warm, warm_supply, log: bool) -> SolveResult:
    """Solve tick t within tick_time_limit_s. If HiGHS stops there with no schedule at all,
    solve again from the same start for the rest of tick_no_schedule_limit_s; the tick's
    solve_s counts both."""
    res = _solve_tick_once(tick, settings, warm, warm_supply, log, settings.tick_time_limit_s)
    rest = settings.tick_no_schedule_limit_s - res.solve_time_s
    if res.status != STATUS_NO_SOLUTION or rest < 1.0:
        return res
    res2 = _solve_tick_once(tick, settings, warm, warm_supply, log, rest)
    res2.solve_time_s += res.solve_time_s
    res2.model_values["extended"] = True
    return res2


def _solve_tick_once(tick: _Tick, settings: RunSettings, warm, warm_supply, log: bool,
                     limit: float) -> SolveResult:
    sc = tick.scenario
    if settings.placement != "thermal_blind":
        return solve(sc, tick.original, settings, warm_start=warm,
                     warm_start_supply_temp_C=warm_supply, time_limit_s=limit, log=log,
                     extras=tick.extras, warm_start_fill=True)
    # thermal-blind: servers chosen with temperature left out, then the full model on them
    pass1 = solve(sc, tick.original, settings, warm_start=warm,
                  warm_start_supply_temp_C=warm_supply, time_limit_s=BLIND_SHARE * limit,
                  log=log, ignore_temperature=True, extras=tick.extras, warm_start_fill=True)
    if not pass1.has_schedule:
        return pass1
    servers = pass1.schedule.server_of()
    extras = replace(tick.extras, forbidden=None)        # pass 1 already kept to them
    pass2 = solve(sc, servers, replace(settings, reassignment="none"),
                  warm_start=pass1.schedule, warm_start_supply_temp_C=pass1.supply_temp_C,
                  time_limit_s=max(1.0, limit - pass1.solve_time_s), log=log, extras=extras)
    pass2.solve_time_s += pass1.solve_time_s
    return pass2


def _gap(res: SolveResult) -> float | None:
    """The tick's relative MIP gap; None when unknown. HiGHS gives inf when a tick stops at its
    time limit with a schedule (e.g. the warm start) but no bound yet."""
    if res.mip_gap is None or not math.isfinite(res.mip_gap):
        return None
    return float(res.mip_gap)


def _no_bound(r: dict) -> bool:
    """A tick that stopped at its time limit before HiGHS found a bound: its gap is unknown."""
    return r["status"] == STATUS_TIME_LIMIT and r.get("mip_gap") is None


def _kept_carbon_g(tick: _Tick, res: SolveResult) -> float | None:
    """Carbon of period t by the solver's own numbers (the objective's period-t terms)."""
    mv = res.model_values
    if "P_crac_kW" not in mv:
        return None
    kw = (mv["P_IT_W"][:, 0].sum() + mv["P_cp_W"][:, 0].sum()) / 1000.0 \
        + mv["P_crac_kW"][:, 0].sum() + mv["P_fan_kW"][:, 0].sum()
    return float(tick.scenario.CI_g_per_kWh[0] * tick.scenario.dt_hours * kw)


def _diagnose(tick: _Tick, settings: RunSettings, res: SolveResult | None, error: str | None,
              time_limit_s: float) -> dict:
    """Which limit leaves tick t with no valid schedule."""
    ids = tick.ids
    out: dict = {"tick": tick.t, "status": None if res is None else res.status,
                 "solver_status_text": error if res is None else res.solver_status_text,
                 "n_waiting": len(tick.waiting), "n_running": len(tick.running)}
    if res is None:
        limits, detail = [LIMIT_MODEL], error
    elif res.status == STATUS_NO_SOLUTION:
        limits = [LIMIT_TICK_TIME]
        detail = (f"HiGHS found no schedule within the tick time limit "
                  f"({settings.tick_time_limit_s:g} s, then up to "
                  f"{settings.tick_no_schedule_limit_s:g} s in all); whether one exists is "
                  "not known")
    else:
        built = _build(tick.scenario, tick.original, settings, (), False,
                       extras=tick.extras)
        if built.empty_workloads:
            free = _build(tick.scenario, tick.original, settings, (), False,
                          extras=replace(tick.extras, forbidden=None))
            who = [ids[k - 1] for k in built.empty_workloads]
            out["workloads"] = [tick_wid(tick, k) for k in built.empty_workloads]
            if not free.empty_workloads:
                limits = [RELAX_PRIORITY]
                detail = (f"workload(s) {who} may use no server: on every server they may use, "
                          "a lower-priority workload started while they waited")
            else:
                limits = [RELAX_DEADLINES]
                detail = (f"workload(s) {who} have no server and start left that meets their "
                          "allowed delay and deadline")
        else:
            base = _solve_impl(tick.scenario, tick.original, settings, relax=(), warm_start=None,
                               warm_start_supply_temp_C=None, time_limit_s=time_limit_s,
                               log=False, feasibility_only=True, extras=tick.extras)
            if base.status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT):
                limits = [LIMIT_SUPPLY_RAMP]
                detail = ("a schedule exists with the supply temperature free of its recent "
                          "history, so the ramp limits from the previous ticks block it")
            else:
                fams = diagnose_infeasibility(tick.scenario, tick.original, settings,
                                              time_limit_s=time_limit_s, extras=tick.extras)
                out["relaxed_one_at_a_time"] = fams
                freed = [f for f, v in fams.items() if v == "feasible"]
                limits = freed or [LIMIT_COMBINATION]
                detail = ("relaxing " + " or ".join(LIMIT_TEXT[f] for f in freed)
                          + " alone gives a schedule") if freed else LIMIT_TEXT[LIMIT_COMBINATION]
    out["limits"] = limits
    out["limit_text"] = "; ".join(LIMIT_TEXT[x] for x in limits)
    out["detail"] = detail
    return out


def tick_wid(tick: _Tick, k: int) -> str:
    return tick.scenario.workloads[k - 1].workload_id


def _play_period(scenario: Scenario, plates: np.ndarray, state: _State, t: int,
                 supply_t: np.ndarray, flow_t: np.ndarray, cool_t: np.ndarray) -> None:
    """Period t with the kept decisions, by the evaluator's physics: temperatures, work
    delivered at the DPF actually reached, and the end of every workload whose work is done."""
    S = scenario.S
    running = sorted(i for i in state.start if i not in state.end)
    cpu = np.zeros(S)
    for i in running:
        cpu[state.server[i] - 1] += scenario.workloads[i - 1].C_req
    cp = scenario.cold_plate
    G = cp.conductance_W_per_C()
    temp = np.zeros(S)
    for sv in scenario.servers:
        k = sv.s - 1
        P = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * (cpu[k] / sv.C_cores)
        Q = 0.0
        n = int(plates[k])
        if n > 0:
            lv = cp.level_of(float(flow_t[k]))
            drive = max(0.0, state.temp[k] - float(cool_t[k]))
            Q = min(n * cp.Q_max_per_plate_W, P, n * G[lv] * drive)
        steady = supply_t[sv.zone - 1] + sv.th_a_C + sv.th_R_C_per_W * (P - Q)
        temp[k] = sv.th_alpha * state.temp[k] + (1.0 - sv.th_alpha) * steady
    D = dpf(temp, scenario.constants)
    dt = scenario.dt_seconds
    for i in running:
        w = scenario.workloads[i - 1]
        sv = scenario.servers[state.server[i] - 1]
        state.delivered[i] += w.C_req * sv.f_MHz * dt * float(D[sv.s - 1])
        if state.delivered[i] >= w.W_MHz_s * (1.0 - DONE_REL_TOL):
            state.end[i] = t + 1
    state.temp = [float(x) for x in temp]


# ---------------------------------------------------------------------------
# The loop
# ---------------------------------------------------------------------------

def _initial_state(scenario: Scenario) -> _State:
    S, Z = scenario.S, scenario.Z
    return _State(next_tick=1, temp=[float(sv.T0_C) for sv in scenario.servers],
                  supply=[[] for _ in range(Z)], cp_flow=[[] for _ in range(S)],
                  cp_coolant=[[] for _ in range(S)])


def _aggregate(scenario: Scenario, state: _State) -> SolveResult:
    ticks = state.ticks
    n = len(ticks)
    n_opt = state.tick_status.get(STATUS_OPTIMAL, 0)
    n_tl = state.tick_status.get(STATUS_TIME_LIMIT, 0)
    gaps = [r["mip_gap"] for r in ticks if r.get("mip_gap") is not None]
    kept = [r["kept_carbon_g"] for r in ticks]
    slow = max(ticks, key=lambda r: r["solve_s"]) if ticks else {"solve_s": 0.0, "tick": None}
    n_nb = sum(1 for r in ticks if _no_bound(r))
    no_bound = (f" {n_nb} of these stopped before a bound was found, so their gap is unknown "
                "and not in the largest gap." if n_nb else "")
    notes = list(state.notes) + [
        "Per-tick re-planning: the model above is solved once per tick over a window from the "
        "tick to the end of the horizon, holding only the workloads that have arrived; running "
        "workloads keep their server and start and count only the work they have left; T0 is "
        "each server's temperature at the end of the previous tick (Eq. 30), and the supply "
        "temperatures of the previous ticks bound the ramps (35)-(36). Only the tick's own "
        "decisions are kept.",
        "Each tick starts from the previous tick's plan, with the workloads that arrived since "
        "placed by the baseline rule (arrival order, earliest start with room for cores and "
        "memory, least loaded server first); the solver may change any of it. A tick with no "
        "schedule at the tick time limit is solved again until the no-schedule limit.",
        f"{n} ticks: {n_opt} solved to the gap, {n_tl} stopped at the tick time limit "
        f"(slowest {slow['solve_s']:.1f} s, tick {slow['tick']}). "
        "The objective is the sum of each tick's own carbon for its period; the gap shown is "
        "the largest of any tick." + no_bound,
    ]
    return SolveResult(
        status=STATUS_OPTIMAL if n_tl == 0 else STATUS_TIME_LIMIT, schedule=None,
        supply_temp_C=None,
        objective_g=None if any(k is None for k in kept) else float(sum(kept)),
        best_bound_g=None, mip_gap=max(gaps) if gaps else None,
        solve_time_s=float(sum(r["solve_s"] for r in ticks)),
        n_variables=state.max_size[0], n_binaries=state.max_size[1],
        n_constraints=state.max_size[2],
        solver_status_text=f"per tick: {n_opt} of {n} optimal, {n_tl} at the tick time limit",
        notes=notes)


def per_tick_block(outcome: PerTickOutcome, settings: RunSettings) -> dict:
    """The "per_tick" block of summary.json."""
    ticks = outcome.ticks
    slow = max(ticks, key=lambda r: r["solve_s"]) if ticks else None
    return {
        "outcome": outcome.status,
        "n_ticks": len(ticks),
        "tick_time_limit_s": settings.tick_time_limit_s,
        "tick_no_schedule_limit_s": settings.tick_no_schedule_limit_s,
        # ticks with no schedule at tick_time_limit_s that were solved again
        "n_ticks_extended": sum(1 for r in ticks if r.get("extended")),
        "n_ticks_optimal": sum(1 for r in ticks if r["status"] == STATUS_OPTIMAL),
        "n_ticks_time_limit": sum(1 for r in ticks if r["status"] == STATUS_TIME_LIMIT),
        # of those, stopped before a bound was found: gap unknown, left out of max_mip_gap
        "n_ticks_no_bound": sum(1 for r in ticks if _no_bound(r)),
        "total_solve_s": float(sum(r["solve_s"] for r in ticks)),
        "slowest_tick": None if slow is None else slow["tick"],
        "slowest_tick_solve_s": None if slow is None else slow["solve_s"],
        "max_mip_gap": max((r["mip_gap"] for r in ticks if r.get("mip_gap") is not None),
                           default=None),
        "resumed_from_tick": outcome.resumed_from_tick,
        "stuck": outcome.stuck,
        "ticks_file": TICKS_FILE,
    }


def stuck_result(outcome: PerTickOutcome, scenario: Scenario) -> SolveResult:
    """The solver result of a run stuck at a tick: no schedule, the tick and the limit."""
    st = outcome.stuck
    return SolveResult(
        status=st["status"] or STATUS_INFEASIBLE, schedule=None, supply_temp_C=None,
        objective_g=None, best_bound_g=None, mip_gap=None,
        solve_time_s=float(sum(r["solve_s"] for r in outcome.ticks)),
        n_variables=0, n_binaries=0, n_constraints=0,
        solver_status_text=f"stuck at tick {st['tick']}: {st['limit_text']}",
        notes=[f"Per-tick re-planning stopped at tick {st['tick']} of {scenario.T}: no valid "
               f"schedule. Limit: {st['limit_text']}. {st['detail']}."])


def run_per_tick(scenario: Scenario, original: dict[int, int], settings: RunSettings,
                 run_dir: str | Path, *, warm_start: bool = True, log: bool = False,
                 wall_time_limit_s: float | None = None, started_at: float | None = None,
                 diagnosis_time_limit_s: float = 120.0, quiet: bool = True,
                 on_tick=None, stop_after_ticks: int | None = None) -> PerTickOutcome:
    """Run (or resume) the per-tick loop of one run. ``original``: the baseline's server of
    every workload (reassignment none / same_zone and B_i = 1 refer to it).

    ``wall_time_limit_s``: after a tick, stop with the checkpoint saved once this much time has
    passed since ``started_at`` (time.monotonic(); default: now). ``on_tick(t, tick, result,
    state)`` is called after every tick (tests). ``stop_after_ticks`` stops like the wall-time
    limit after that many ticks of this call (tests)."""
    t_begin = time.monotonic() if started_at is None else started_at
    run_dir = Path(run_dir)
    ckpt = run_dir / CHECKPOINT_DIR
    state = _load_checkpoint(ckpt, settings, scenario)
    resumed = None
    if state is None:
        state = _initial_state(scenario)
        # a summary left by an earlier run of this name would make a stopped run look done
        (run_dir / "summary.json").unlink(missing_ok=True)
    else:
        resumed = state.next_tick
        if not quiet:
            print(f"Resuming from the checkpoint at tick {resumed}.")
    run_dir.mkdir(parents=True, exist_ok=True)
    plates = cold_plate_counts(scenario, settings)
    pairs = priority_pairs(scenario) if settings.priority else []
    pairs_by_i: dict[int, list[int]] = {}
    for i, j in pairs:
        pairs_by_i.setdefault(i, []).append(j)
    T = scenario.T
    done_here = 0

    while state.next_tick <= T:
        t = state.next_tick
        t0 = time.perf_counter()
        tick = _Tick(scenario, original, state, t, pairs_by_i, pairs, settings)
        warm, warm_sup = tick.warm_start(state) if warm_start else (None, None)
        res, error = None, None
        try:
            res = _solve_tick(tick, settings, warm, warm_sup, log)
        except ValueError as e:
            error = str(e)
        if res is None or not res.has_schedule:
            stuck = _diagnose(tick, settings, res, error, diagnosis_time_limit_s)
            row = {"tick": t, "status": STATUS_INFEASIBLE if res is None else res.status,
                   "solve_s": 0.0 if res is None else res.solve_time_s, "mip_gap": None,
                   "n_waiting": len(tick.waiting), "n_started": 0,
                   "n_running": len(tick.running),
                   "n_arrived": sum(1 for w in scenario.workloads if w.A == t),
                   "n_done": len(state.end), "objective_g": None, "kept_carbon_g": None,
                   "n_binaries": None if res is None else res.n_binaries,
                   "n_constraints": None if res is None else res.n_constraints,
                   "tick_wall_s": time.perf_counter() - t0,
                   "extended": bool(res is not None and res.model_values.get("extended")),
                   "warm_start_filled": None if res is None
                   else res.model_values.get("warm_start_filled")}
            ticks = state.ticks + [row]
            _write_ticks(run_dir / TICKS_FILE, ticks)
            _write_partial(run_dir / PARTIAL_FILE, scenario, state)
            shutil.rmtree(ckpt, ignore_errors=True)
            if not quiet:
                print(f"Tick {t}: no valid schedule. Limit: {stuck['limit_text']}. {stuck['detail']}")
            return PerTickOutcome(status=OUTCOME_STUCK, ticks=ticks, stuck=stuck,
                                  resumed_from_tick=resumed, next_tick=t)

        # keep period t
        pl = res.schedule.placements
        started_now = []
        for k, i in enumerate(tick.ids, start=1):
            p = pl[k - 1]
            if i in state.start:
                if p.server != state.server[i] or p.start != 1:
                    raise RuntimeError(f"tick {t}: running workload {i} moved in the window "
                                       f"({p.server}, {p.start})")
            elif p.start == 1:
                state.server[i], state.start[i] = int(p.server), t
                state.delivered[i] = 0.0
                started_now.append(i)
            elif p.start is None or p.start < 1:
                raise RuntimeError(f"tick {t}: workload {i} has no start in the window")
        supply_t = np.asarray(res.supply_temp_C, dtype=float)[:, 0]
        S = scenario.S
        if res.cold_plate_ops is not None:
            flow_t = np.asarray(res.cold_plate_ops.flow_Lmin)[:, 0]
            cool_t = np.asarray(res.cold_plate_ops.coolant_C)[:, 0]
        else:
            flow_t = cool_t = np.full(S, np.nan)
        for z in range(scenario.Z):
            state.supply[z].append(float(supply_t[z]))
        for k in range(S):
            state.cp_flow[k].append(None if math.isnan(flow_t[k]) else float(flow_t[k]))
            state.cp_coolant[k].append(None if math.isnan(cool_t[k]) else float(cool_t[k]))
        state.plan = {i: [p.server, p.start + t - 1, p.end + t - 1]
                      for i, p in zip(tick.ids, pl) if p.server is not None}
        state.plan_supply = np.asarray(res.supply_temp_C, dtype=float).tolist()
        n_running = len(tick.running) + len(started_now)
        _play_period(scenario, plates, state, t, supply_t, flow_t, cool_t)

        if t == 1:
            state.notes = list(res.notes)
        state.tick_status[res.status] = state.tick_status.get(res.status, 0) + 1
        state.max_size = [max(state.max_size[0], res.n_variables),
                          max(state.max_size[1], res.n_binaries),
                          max(state.max_size[2], res.n_constraints)]
        state.ticks.append({
            "tick": t, "status": res.status, "solve_s": float(res.solve_time_s),
            "mip_gap": _gap(res),
            "n_waiting": len(tick.waiting), "n_started": len(started_now),
            "n_running": n_running,
            "n_arrived": sum(1 for w in scenario.workloads if w.A == t),
            "n_done": len(state.end),
            "objective_g": None if res.objective_g is None else float(res.objective_g),
            "kept_carbon_g": _kept_carbon_g(tick, res),
            "n_binaries": res.n_binaries, "n_constraints": res.n_constraints,
            "tick_wall_s": time.perf_counter() - t0,
            "extended": bool(res.model_values.get("extended")),
            "warm_start_filled": res.model_values.get("warm_start_filled")})
        state.next_tick = t + 1
        done_here += 1
        if state.next_tick <= T:
            _save_checkpoint(ckpt, settings, scenario, state)
        _write_ticks(run_dir / TICKS_FILE, state.ticks)
        if on_tick is not None:
            on_tick(t, tick, res, state)
        if not quiet:
            r = state.ticks[-1]
            gap = "-" if r["mip_gap"] is None else f"{r['mip_gap']:.2%}"
            print(f"tick {t}/{T}: {r['status']}, {r['solve_s']:.2f} s, gap {gap}, "
                  f"{r['n_waiting']} waiting, {r['n_started']} started, "
                  f"{r['n_running']} running", flush=True)
        if state.next_tick <= T and (
                (wall_time_limit_s is not None
                 and time.monotonic() - t_begin >= wall_time_limit_s)
                or (stop_after_ticks is not None and done_here >= stop_after_ticks)):
            return PerTickOutcome(status=OUTCOME_STOPPED, ticks=list(state.ticks),
                                  resumed_from_tick=resumed, next_tick=state.next_tick)

    # every tick done
    missing = [w.i for w in scenario.workloads if w.i not in state.end]
    if missing:
        raise RuntimeError(f"per-tick run ended with workloads not finished: {missing[:10]}")
    schedule = Schedule(tuple(Placement(w.i, state.server[w.i], state.start[w.i], state.end[w.i])
                              for w in scenario.workloads))
    supply = np.asarray(state.supply, dtype=float)
    ops = None
    if plates.any():
        f = np.array([[np.nan if v is None else v for v in row] for row in state.cp_flow])
        c = np.array([[np.nan if v is None else v for v in row] for row in state.cp_coolant])
        ops = ColdPlateOps(flow_Lmin=f, coolant_C=c)
    result = _aggregate(scenario, state)
    result.schedule, result.supply_temp_C, result.cold_plate_ops = schedule, supply, ops
    shutil.rmtree(ckpt, ignore_errors=True)
    return PerTickOutcome(status=OUTCOME_DONE, ticks=list(state.ticks), schedule=schedule,
                          supply_temp_C=supply, cold_plate_ops=ops, result=result,
                          resumed_from_tick=resumed, next_tick=None)


def _write_partial(path: Path, scenario: Scenario, state: _State) -> None:
    """Starts kept before a stuck tick (end blank for workloads still running)."""
    with open(path, "w", newline="") as fh:
        w = csv.writer(fh)
        w.writerow(["i", "workload_id", "server", "start", "end"])
        for i in sorted(state.start):
            w.writerow([i, scenario.workloads[i - 1].workload_id, state.server[i],
                        state.start[i], state.end.get(i, "")])


In [ ]:
%%writefile {LIB}/scheduler/run.py
"""Command line: build the baseline, solve the optimized schedule, compare the two.

    uv run python -m scheduler.run --scenario small
    uv run python -m scheduler.run --scenario small --baseline-rule fastest_first --baseline-supply-temp 27
    uv run python -m scheduler.run --scenario small --no-cold-plates
    uv run python -m scheduler.run --scenario small --cold-plate-count 2
    uv run python -m scheduler.run --scenario servers6 --zone-map 1,1,1,2,2,2
    uv run python -m scheduler.run --scenario servers6 --cp-mode fixed
    uv run python -m scheduler.run --scenario servers6 --ci-flat
    uv run python -m scheduler.run --scenario small --thermal-blind
    uv run python -m scheduler.run --scenario small --per-tick --tick-time-limit 30

Both schedules are scored by the shared evaluator; only its numbers are compared.

--per-tick re-plans at every period with only the workloads that have arrived and keeps that
period's decisions (scheduler/pertick.py). It saves its state after every tick in
results/<scenario>/<run-name>/checkpoint/; running the same command again resumes from it.

Exit codes: 0 done and every check passed; 1 done but a check failed; 2 a switch that is not
defined yet; 3 stopped by --wall-time-limit (run the same command again to resume); 4 a tick
had no valid schedule (the tick and the limit are in summary.json and ticks.csv).
"""

from __future__ import annotations

import argparse
import sys
import time
import traceback
from dataclasses import replace
from pathlib import Path

from scheduler.baseline import build_baseline
from scheduler.evaluate import evaluate
from scheduler.io import apply_overrides, load_reference_baseline, load_scenario
from scheduler.optimize import diagnose_infeasibility, solve
from scheduler.pertick import (
    OUTCOME_DONE, OUTCOME_STOPPED, OUTCOME_STUCK, per_tick_block, run_per_tick, stuck_result,
)
from scheduler.report import format_summary, write_results
from scheduler.types import (
    BASELINE_RULES, COLD_PLATE_MODES, REASSIGNMENT_MODES, STATUS_INFEASIBLE, SUPPLY_TEMP_MODES,
    NotDefinedYet, RunSettings, constant_supply_temp,
)

OBJECTIVE_REL_TOL = 1e-6    # optimizer objective against the evaluator's carbon
NO_WORSE_REL_TOL = 1e-9     # optimized carbon against the baseline's
STAGE_SHARE = 0.2           # share of the time limit for the fixed-server first stage
BLIND_SHARE = 0.7           # thermal-blind placement: share of the time limit for pass 1
EXIT_CHECK_FAILED = 1
EXIT_NOT_DEFINED = 2
EXIT_WALL_TIME = 3          # per-tick run stopped by --wall-time-limit; run again to resume
EXIT_STUCK = 4              # per-tick run: a tick has no valid schedule
EXIT_ERROR = 5              # the run crashed, e.g. a checkpoint made by another command


def parse_args(argv=None) -> tuple[RunSettings, argparse.Namespace]:
    d = RunSettings()
    p = argparse.ArgumentParser(prog="scheduler.run", description=__doc__,
                                formatter_class=argparse.RawDescriptionHelpFormatter)
    p.add_argument("--scenario", default=d.scenario)
    p.add_argument("--run-name", default=None,
                   help="folder under results/<scenario>/ (default: built from the switches)")
    p.add_argument("--baseline-rule", choices=BASELINE_RULES, default=d.baseline_rule)
    p.add_argument("--seed", type=int, default=d.seed, help="for the random baseline rule")
    p.add_argument("--baseline-supply-temp", type=float, default=d.baseline_supply_temp_C,
                   help="fixed supply air temperature of the baseline, degC")
    p.add_argument("--reassignment", choices=REASSIGNMENT_MODES, default=d.reassignment)
    p.add_argument("--thermal-blind", action="store_true",
                   help="choose servers with server temperature left out, then set start times "
                        "and cooling on those servers with every thermal limit (two passes)")
    p.add_argument("--supply-temp-mode", choices=SUPPLY_TEMP_MODES, default=d.supply_temp_mode)
    p.add_argument("--fixed-supply-temp", type=float, default=None,
                   help="supply temperature when --supply-temp-mode fixed "
                        "(default: the baseline's)")
    p.add_argument("--h-min", type=float, default=None,
                   help="override the minimum thermal headroom H_min, degC")
    p.add_argument("--time-limit", type=float, default=d.time_limit_s, help="seconds")
    p.add_argument("--mip-gap", type=float, default=d.mip_rel_gap, help="relative gap")
    p.add_argument("--cold-plates", action=argparse.BooleanOptionalAction, default=d.cold_plates,
                   help="model cold plates (on by default); plates per server from servers.csv "
                        "(n_cold_plates, else CT_s_cold_plate). --no-cold-plates turns them off")
    p.add_argument("--cold-plate-count", type=int, default=None,
                   help="put this many cold plates on every server (0 disables them); "
                        "needs cold plates on")
    p.add_argument("--baseline-cp-flow", type=float, default=d.baseline_cp_flow_Lmin,
                   help="fixed cold-plate flow of the baseline, L/min (one of the flow settings)")
    p.add_argument("--baseline-cp-coolant", type=float, default=d.baseline_cp_coolant_C,
                   help="fixed cold-plate coolant temperature of the baseline, degC")
    p.add_argument("--cp-mode", choices=COLD_PLATE_MODES, default=d.cp_mode,
                   help="optimize: the optimized run picks flow and coolant per server and period; "
                        "fixed: it holds them at --fixed-cp-flow / --fixed-cp-coolant")
    p.add_argument("--fixed-cp-flow", type=float, default=None,
                   help="cold-plate flow when --cp-mode fixed, L/min (default: the baseline's)")
    p.add_argument("--fixed-cp-coolant", type=float, default=None,
                   help="coolant temperature when --cp-mode fixed, degC (default: the baseline's)")
    p.add_argument("--no-priority", action="store_true",
                   help="drop the priority rules (56)-(57) from the optimized run")
    p.add_argument("--zone-map", default=None,
                   help="zone of each server in order, e.g. 1,1,1,2,2,2 (replaces servers.csv)")
    ci = p.add_mutually_exclusive_group()
    ci.add_argument("--carbon-file", default=None,
                    help="CSV with columns t, CI_t_g_per_kWh replacing the scenario's carbon.csv")
    ci.add_argument("--ci-flat", action="store_true",
                    help="hold carbon intensity at the scenario's mean in every period")
    p.add_argument("--fan-power", action="store_true", help="not defined yet")
    p.add_argument("--per-tick", action="store_true",
                   help="re-plan at every period with only the workloads that have arrived and "
                        "keep that period's decisions (instead of one plan for the whole day)")
    p.add_argument("--tick-time-limit", type=float, default=d.tick_time_limit_s,
                   help="per-tick: HiGHS time limit of each tick's solve, seconds")
    p.add_argument("--tick-no-schedule-limit", type=float, default=d.tick_no_schedule_limit_s,
                   help="per-tick: a tick with no schedule at --tick-time-limit is solved "
                        "again until this many seconds in all; equal to it: no second solve")
    p.add_argument("--wall-time-limit", type=float, default=None,
                   help="per-tick: after a tick, if this many seconds have passed, save the "
                        "checkpoint and exit with code 3; the same command resumes the run")
    p.add_argument("--baseline-only", action="store_true", help="skip the optimized run")
    p.add_argument("--no-warm-start", action="store_true",
                   help="do not offer the solver a starting schedule")
    p.add_argument("--diagnosis-time-limit", type=float, default=120.0,
                   help="seconds per relaxed re-solve when no valid schedule exists")
    p.add_argument("--solver-log", action="store_true")
    p.add_argument("--scenarios-dir", default="scenarios")
    p.add_argument("--results-dir", default="results")
    a = p.parse_args(argv)
    zone_map = None
    if a.zone_map:
        try:
            zone_map = tuple(int(z) for z in a.zone_map.split(","))
        except ValueError:
            p.error(f"--zone-map must be comma-separated zone numbers, got {a.zone_map!r}")

    settings = RunSettings(
        scenario=a.scenario, baseline_rule=a.baseline_rule, seed=a.seed,
        baseline_supply_temp_C=a.baseline_supply_temp, reassignment=a.reassignment,
        placement="thermal_blind" if a.thermal_blind else d.placement,
        supply_temp_mode=a.supply_temp_mode, fixed_supply_temp_C=a.fixed_supply_temp,
        H_min_C=a.h_min, time_limit_s=a.time_limit, mip_rel_gap=a.mip_gap,
        cold_plates=a.cold_plates, cold_plate_count=a.cold_plate_count,
        baseline_cp_flow_Lmin=a.baseline_cp_flow, baseline_cp_coolant_C=a.baseline_cp_coolant,
        cp_mode=a.cp_mode, fixed_cp_flow_Lmin=a.fixed_cp_flow, fixed_cp_coolant_C=a.fixed_cp_coolant,
        priority=not a.no_priority, zone_map=zone_map, carbon_file=a.carbon_file,
        ci_flat=a.ci_flat, fan_power=a.fan_power, per_tick=a.per_tick,
        tick_time_limit_s=a.tick_time_limit, tick_no_schedule_limit_s=a.tick_no_schedule_limit,
    )
    settings.run_name = a.run_name or default_run_name(settings)
    return settings, a


def default_run_name(settings: RunSettings) -> str:
    """Run folder name built from the switches, e.g. ``least_loaded_base18_free_optimize``.

    Every switch that can change the result and differs from its default is in the name,
    so two runs with different switches never share a folder. A per-tick run ends in
    ``_pertick`` (and ``_ttl<s>`` / ``_nsl<s>`` for a tick time limit or no-schedule limit other
    than the default); the whole-day
    time limit does not apply to it and is left out."""
    st, d = settings, RunSettings()
    fixed_temp = st.optimized_fixed_supply_temp_C()
    cp_fixed = st.cold_plates and st.cp_mode == "fixed"
    fixed_cp = (None if (st.fixed_cp_flow_Lmin, st.fixed_cp_coolant_C) == (None, None)
                else st.optimized_fixed_cp())
    cp_values = ((st.baseline_cp_flow_Lmin, st.baseline_cp_coolant_C)
                 != (d.baseline_cp_flow_Lmin, d.baseline_cp_coolant_C))
    return "_".join([
        st.baseline_rule, f"base{st.baseline_supply_temp_C:g}", st.reassignment,
        st.supply_temp_mode if st.supply_temp_mode == "optimize" else f"fixed{fixed_temp:g}",
    ] + (["tblind"] if st.placement == "thermal_blind" else [])
      + (["nocp"] if not st.cold_plates else
         [] if st.cold_plate_count is None else [f"cp{st.cold_plate_count}"])
      + ([f"bcp{st.baseline_cp_flow_Lmin:g}-{st.baseline_cp_coolant_C:g}"]
         if st.cold_plates and cp_values else [])
      + ([] if not cp_fixed else
         ["cpfixed" if fixed_cp is None else f"cpfixed{fixed_cp[0]:g}-{fixed_cp[1]:g}"])
      + (["noprio"] if not st.priority else [])
      + ([f"zones{''.join(map(str, st.zone_map))}"] if st.zone_map else [])
      + (["ciflat"] if st.ci_flat else [])
      + ([f"ci-{Path(st.carbon_file).stem}"] if st.carbon_file else [])
      + ([f"hmin{st.H_min_C:g}"] if st.H_min_C is not None else [])
      + ([f"seed{st.seed}"] if st.baseline_rule == "random" else [])
      + ([f"tl{st.time_limit_s:g}"] if st.time_limit_s != d.time_limit_s and not st.per_tick
         else [])
      + ([f"gap{st.mip_rel_gap:g}"] if st.mip_rel_gap != d.mip_rel_gap else [])
      + (["pertick"] if st.per_tick else [])
      + ([f"ttl{st.tick_time_limit_s:g}"]
         if st.per_tick and st.tick_time_limit_s != d.tick_time_limit_s else [])
      + ([f"nsl{st.tick_no_schedule_limit_s:g}"]
         if st.per_tick and st.tick_no_schedule_limit_s != d.tick_no_schedule_limit_s else []))


def solve_staged(scenario, original, settings, base_sched, base_supply, warm_start, log,
                 time_limit_s=None, ignore_temperature=False):
    """Solve the optimized model, offering the solver a starting schedule.

    The start is the baseline. When servers may change, the same model is first solved
    with every server fixed (a much smaller problem whose solutions are all valid for
    the full one) and its schedule is offered instead. A start only gives the solver a
    first incumbent; it does not change the model or its optimum. ``time_limit_s``
    (default: the run's) covers both stages; ``ignore_temperature`` is passed to solve.
    """
    limit = settings.time_limit_s if time_limit_s is None else time_limit_s
    if not warm_start:
        return solve(scenario, original, settings, time_limit_s=limit, log=log,
                     ignore_temperature=ignore_temperature)
    start_sched, start_supply = base_sched, base_supply
    stage_time, stage_note = 0.0, None
    if settings.reassignment != "none":
        stage = solve(scenario, original, replace(settings, reassignment="none"),
                      warm_start=base_sched, warm_start_supply_temp_C=base_supply,
                      time_limit_s=STAGE_SHARE * limit, log=log,
                      ignore_temperature=ignore_temperature)
        stage_time = stage.solve_time_s
        if stage.has_schedule:
            start_sched, start_supply = stage.schedule, stage.supply_temp_C
            stage_note = (
                f"Starting schedule: the same model with reassignment=none, solved first "
                f"({stage.status}, {stage.objective_g:,.2f} g, {stage_time:.1f} s); its time "
                f"is included in the solve time.")
        else:
            stage_note = (f"Starting schedule: the baseline (the reassignment=none stage "
                          f"ended with status {stage.status} after {stage_time:.1f} s).")
    result = solve(scenario, original, settings, warm_start=start_sched,
                   warm_start_supply_temp_C=start_supply,
                   time_limit_s=max(1.0, limit - stage_time), log=log,
                   ignore_temperature=ignore_temperature)
    result.solve_time_s += stage_time
    if stage_note:
        result.notes.append(stage_note)
    return result


def solve_thermal_blind(scenario, original, settings, base_sched, base_supply, warm_start, log):
    """Thermal-blind placement in two passes. Returns the result and the (servers, settings)
    the result was solved with, for the infeasibility diagnosis.

    Pass 1 solves the model with server temperature left out (no recursion, no thermal cap,
    DPF = 1) and keeps only the server of each workload. Pass 2 fixes every workload to that
    server and solves the full model, every thermal limit included, for start times and
    cooling. If pass 2 has no valid schedule, that is the result: no fallback.
    """
    pass1 = solve_staged(scenario, original, settings, base_sched, base_supply, warm_start, log,
                         time_limit_s=BLIND_SHARE * settings.time_limit_s,
                         ignore_temperature=True)
    gap = "-" if pass1.mip_gap is None else f"{pass1.mip_gap:.2%}"
    if not pass1.has_schedule:
        pass1.notes.append(
            f"Thermal-blind placement: pass 1 (servers chosen with server temperature left out) "
            f"ended with status {pass1.status} after {pass1.solve_time_s:.1f} s and no schedule, "
            "so pass 2 was not run.")
        return pass1, (original, settings)
    servers = pass1.schedule.server_of()
    moved = sum(1 for i, sv in servers.items() if sv != original[i])
    fixed = replace(settings, reassignment="none")
    pass2 = solve(scenario, servers, fixed,
                  warm_start=pass1.schedule if warm_start else None,
                  warm_start_supply_temp_C=pass1.supply_temp_C if warm_start else None,
                  time_limit_s=max(1.0, settings.time_limit_s - pass1.solve_time_s), log=log)
    pass2.solve_time_s += pass1.solve_time_s
    pass2.notes.append(
        f"Thermal-blind placement, pass 1: servers chosen with server temperature left out "
        f"(no temperature recursion, no thermal cap, DPF = 1); {pass1.status}, "
        f"{pass1.objective_g:,.2f} g, gap {gap}, {pass1.solve_time_s:.1f} s; {moved} workload(s) "
        f"on another server than the baseline's."
        + "".join(f" Pass 1 {n[0].lower()}{n[1:]}" for n in pass1.notes
                  if n.startswith("Starting schedule")))
    pass2.notes.append(
        "Thermal-blind placement, pass 2: every workload fixed to its pass-1 server; start times "
        "and cooling solved with every thermal limit. The status, gap and objective above are "
        "pass 2's; the solve time includes pass 1.")
    return pass2, (servers, fixed)


def optimized_checks(scenario, settings, original, base_eval, result, opt_eval) -> dict:
    """Checks on an optimized schedule: the solver's objective equals the evaluator's carbon,
    the evaluator finds no broken limit, running workloads stay put, and with nothing to
    optimise but timing the result is no worse than a valid baseline."""
    out: dict = {}
    diff = abs(result.objective_g - opt_eval.total_carbon_g)
    out["optimizer_objective_equals_evaluator_carbon"] = {
        "passed": diff <= OBJECTIVE_REL_TOL * abs(opt_eval.total_carbon_g),
        "optimizer_objective_g": result.objective_g,
        "evaluator_carbon_g": opt_eval.total_carbon_g,
        "relative_difference": diff / abs(opt_eval.total_carbon_g)}
    out["optimized_schedule_valid_under_evaluator"] = {
        "passed": opt_eval.valid, "violations": opt_eval.compliance["violations"]}
    running = [w.i for w in scenario.workloads if w.B == 1]
    if running:
        moved = [i for i in running
                 if result.schedule.placements[i - 1].server != original[i]]
        out["running_workloads_not_reassigned"] = {
            "passed": not moved, "running": len(running), "moved": moved}
    same_temp = (settings.supply_temp_mode == "fixed" and
                 settings.optimized_fixed_supply_temp_C() == settings.baseline_supply_temp_C)
    if settings.reassignment == "none" and same_temp:
        name = "optimized_no_worse_than_valid_baseline"
        if settings.per_tick:
            out[name] = {"passed": None, "not_applicable":
                         "per tick, the optimizer does not know later arrivals, so it can do "
                         "worse than the baseline", "baseline_carbon_g": base_eval.total_carbon_g,
                         "optimized_carbon_g": opt_eval.total_carbon_g}
        elif base_eval.valid:
            out[name] = {
                "passed": opt_eval.total_carbon_g <=
                base_eval.total_carbon_g * (1 + NO_WORSE_REL_TOL),
                "baseline_carbon_g": base_eval.total_carbon_g,
                "optimized_carbon_g": opt_eval.total_carbon_g}
        else:
            out[name] = {"passed": None, "not_applicable":
                         "the baseline breaks limits, so it is not a valid schedule"}
    return out


def format_checks(checks: dict) -> str:
    lines = ["Checks:"]
    for name, c in checks.items():
        state = {True: "PASS", False: "FAIL", None: "n/a"}.get(c.get("passed"), "info")
        detail = {k: v for k, v in c.items() if k != "passed"}
        lines.append(f"  [{state}] {name}" + (f"  {detail}" if detail else ""))
    return "\n".join(lines)


def run(settings: RunSettings, *, scenarios_dir="scenarios", results_dir="results",
        baseline_only=False, warm_start=True, diagnosis_time_limit_s=120.0,
        solver_log=False, quiet=False, made_with="command line",
        wall_time_limit_s: float | None = None) -> dict:
    """Run one comparison and write its results. Returns the summary dict.

    Per tick (``settings.per_tick``), a run stopped by ``wall_time_limit_s`` writes no summary
    and returns ``{"per_tick": {"outcome": "stopped", ...}}``; the same call resumes it."""
    started_at = time.monotonic()
    settings.validate()
    scenario = apply_overrides(load_scenario(settings.scenario, scenarios_dir), settings)
    checks: dict = {}

    # baseline
    base_sched = build_baseline(scenario, settings)
    if settings.baseline_rule == "least_loaded":
        ref = load_reference_baseline(settings.scenario, scenarios_dir)
        checks["least_loaded_reproduces_baseline_schedule_csv"] = {
            "passed": base_sched == ref}
    base_supply = constant_supply_temp(scenario, settings.baseline_supply_temp_C)
    base_eval = evaluate(scenario, base_sched, base_supply, settings)
    original = base_sched.server_of()

    # optimized
    out_dir = Path(results_dir) / settings.scenario / settings.run_name
    result = opt_eval = infeasibility = None
    per_tick = None
    if baseline_only:
        pass
    elif any(sv is None for sv in original.values()):
        unplaced = [i for i, sv in original.items() if sv is None]
        checks["optimized_run_skipped"] = {
            "reason": f"the baseline could not place workloads {unplaced}, so they have no "
                      "original server"}
    elif settings.per_tick:
        outcome = run_per_tick(scenario, original, settings, out_dir, warm_start=warm_start,
                               log=solver_log, wall_time_limit_s=wall_time_limit_s,
                               started_at=started_at, diagnosis_time_limit_s=diagnosis_time_limit_s,
                               quiet=quiet)
        per_tick = per_tick_block(outcome, settings)
        if outcome.status == OUTCOME_STOPPED:
            if not quiet:
                print(f"\nStopped by the wall-time limit before tick {outcome.next_tick} of "
                      f"{scenario.T}; checkpoint saved in {out_dir}/checkpoint/. Run the same "
                      "command again to resume.")
            return {"per_tick": {**per_tick, "next_tick": outcome.next_tick}}
        if outcome.status == OUTCOME_DONE:
            result = outcome.result
            opt_eval = evaluate(scenario, result.schedule, result.supply_temp_C, settings,
                                result.cold_plate_ops)
            checks.update(optimized_checks(scenario, settings, original, base_eval,
                                           result, opt_eval))
        else:
            result = stuck_result(outcome, scenario)
            infeasibility = outcome.stuck.get("relaxed_one_at_a_time")
    else:
        if settings.placement == "thermal_blind":
            result, (diag_servers, diag_settings) = solve_thermal_blind(
                scenario, original, settings, base_sched, base_supply, warm_start, solver_log)
        else:
            result = solve_staged(scenario, original, settings, base_sched, base_supply,
                                  warm_start, solver_log)
            diag_servers, diag_settings = original, settings
        if result.status == STATUS_INFEASIBLE:
            infeasibility = diagnose_infeasibility(scenario, diag_servers, diag_settings,
                                                   time_limit_s=diagnosis_time_limit_s)
        if result.has_schedule:
            opt_eval = evaluate(scenario, result.schedule, result.supply_temp_C, settings,
                                result.cold_plate_ops)
            checks.update(optimized_checks(scenario, settings, original, base_eval,
                                           result, opt_eval))

    summary = write_results(
        out_dir, scenario, settings, original, base_sched, base_eval,
        result.schedule if result else None, opt_eval, result, infeasibility, checks,
        made_with=made_with, per_tick=per_tick)
    if not quiet:
        print(format_summary(summary))
        print("\n" + format_checks(checks))
        print(f"\nResults written to {out_dir}/")
    return summary


def main(argv=None) -> int:
    settings, a = parse_args(argv)
    try:
        summary = run(settings, scenarios_dir=a.scenarios_dir, results_dir=a.results_dir,
                      baseline_only=a.baseline_only, warm_start=not a.no_warm_start,
                      diagnosis_time_limit_s=a.diagnosis_time_limit, solver_log=a.solver_log,
                      wall_time_limit_s=a.wall_time_limit)
    except NotDefinedYet as e:
        print(f"Not defined yet: {e}", file=sys.stderr)
        return EXIT_NOT_DEFINED
    except Exception:
        # Python's own exit code for a crash is 1, the same as "done, a check failed"
        traceback.print_exc()
        return EXIT_ERROR
    outcome = (summary.get("per_tick") or {}).get("outcome")
    if outcome == OUTCOME_STOPPED:
        return EXIT_WALL_TIME
    if outcome == OUTCOME_STUCK:
        st = summary["per_tick"]["stuck"]
        print(f"\nSTUCK at tick {st['tick']}: {st['limit_text']}", file=sys.stderr)
        return EXIT_STUCK
    failed = [n for n, c in summary.get("checks", {}).items() if c.get("passed") is False]
    if failed:
        print(f"\nFAILED checks: {failed}", file=sys.stderr)
        return EXIT_CHECK_FAILED
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile {LIB}/scheduler/validate.py
"""Validation checks: small, controlled tests that show the model does what the thesis says.

Each check answers one question of the thesis group and returns a table: a list of rows
(dicts) whose keys are the column headers, with units. Where a check has an expected
answer, the row says whether it passed.

    uv run python -m scheduler.validate --scenario servers6

writes every table to ``results/validation/<scenario>/`` as CSV, with ``validation.json``
(release, date, scenario build, and the result of every check).
``notebooks/solver.ipynb`` computes the same tables in its last section.

The checks that solve build small scenarios from the real servers and constants of the
scenario, so every number comes from the model's own code: the optimizer (optimize.py) and
the independent evaluator (evaluate.py). The checks that read runs take a run folder under
``results/`` and are skipped when it is missing.
"""

from __future__ import annotations

import argparse
import csv
import dataclasses
import json
import math
import sys
from datetime import datetime, timezone
from pathlib import Path

import numpy as np

from .baseline import build_baseline, periods_needed
from .evaluate import evaluate
from .io import load_scenario
from .optimize import diagnose_infeasibility, solve
from .types import (
    STATUS_INFEASIBLE, STATUS_OPTIMAL, Placement, RunSettings, Scenario, Schedule, Workload,
    constant_supply_temp, cop, status_text,
)
from .version import VERSION

POWER_TOL_W = 0.01
TOY_TIME_LIMIT_S = 60.0

# Examples of the thesis group (validation questions of 2026-10-21), posed on thesis10, the
# earlier 10-server scenario. On another scenario the checks use the workloads of these
# numbers it has, and its first and last server.
EXAMPLE_WORKLOADS = (211, 232, 9, 5)
EXAMPLE_SERVERS = ("qh2-rcc165", "qh2-rcc183")


def _clock(period: int, start_hour: float | None, dt_seconds: float) -> str | None:
    """Local clock time at the start of ``period`` (1-based); None without a start hour."""
    if start_hour is None:
        return None
    minutes = round(start_hour * 60 + (period - 1) * dt_seconds / 60) % (24 * 60)
    return f"{minutes // 60:02d}:{minutes % 60:02d}"


def _server(scenario: Scenario, name: str):
    for sv in scenario.servers:
        if sv.name == name:
            return sv
    raise ValueError(f"no server named {name!r} in scenario {scenario.name!r}")


def _workload(scenario: Scenario, i: int) -> Workload:
    if not 1 <= i <= scenario.I:
        raise ValueError(f"workload {i} is not in scenario {scenario.name!r} (1..{scenario.I})")
    return scenario.workloads[i - 1]


def _toy(scenario: Scenario, servers, workloads, T: int, name: str) -> Scenario:
    """A small scenario: the given servers (renumbered, all in zone 1) and workloads, the
    first T periods of the carbon curve, and the scenario's constants."""
    svs = tuple(dataclasses.replace(sv, s=k, zone=1) for k, sv in enumerate(servers, 1))
    return dataclasses.replace(
        scenario, name=name, T=T, servers=svs, workloads=tuple(workloads), n_zones=1,
        CI_g_per_kWh=np.asarray(scenario.CI_g_per_kWh[:T], dtype=float), AV=None,
        fingerprint=None)


def _top_supply(scenario: Scenario) -> float:
    """Highest whole-degree supply temperature allowed: what every optimized run picks."""
    c = scenario.constants
    return float(math.floor(min(c.T_crac_max_C, c.T_max_zone_C) + 1e-9))


# ---------------------------------------------------------------------------
# Inputs: the workload table of the thesis (Section 5.2.2)
# ---------------------------------------------------------------------------

def workload_inputs(scenario: Scenario, start_hour: float | None = None) -> list[dict]:
    """One row per workload: what it asks for and when, with units."""
    dt_min = scenario.dt_seconds / 60.0
    rows = []
    for w in scenario.workloads:
        rows.append({
            "Workload": w.i,
            "Workload ID": w.workload_id,
            "Flavor": w.flavor,
            "Type": "time-sensitive" if w.RT else "delay-tolerant",
            "Requested CPU cores (vCPU)": w.C_req,
            "Requested RAM (GB)": round(w.M_req_MB / 1024.0, 3),
            "Arrival (period)": w.A,
            "Arrival (clock)": _clock(w.A, start_hour, scenario.dt_seconds),
            "Deadline (period)": w.D,
            "Deadline (clock)": _clock(w.D, start_hour, scenario.dt_seconds),
            "Base runtime (min)": w.ET_base * dt_min,
            "Allowed delay (min)": w.AD * dt_min,
            "Work required (MHz·s)": w.W_MHz_s,
        })
    return rows


# ---------------------------------------------------------------------------
# B9: server power at 0, 50 and 100 % CPU use
# ---------------------------------------------------------------------------

def power_check(scenario: Scenario) -> list[dict]:
    """For every server: power at 0, 50 and 100 % CPU use, expected from its listed idle and
    maximum power, against the optimizer's and the evaluator's value.

    Each server is solved alone over two periods with one workload that fills half or all of
    its cores in period 1; period 2 is idle (0 %). Its starting temperature is set to 30 °C
    so that heat cannot get in the way; power does not depend on temperature."""
    rows = []
    dt = scenario.dt_seconds
    for sv in scenario.servers:
        measured: dict[float, tuple[float, float]] = {}
        for share in (0.5, 1.0):
            cores = max(1, round(share * sv.C_cores))
            w = Workload(i=1, workload_id="power-test", flavor="test", A=1, D=2, AD=0,
                         ET_base=1, RT=0, B=0, X0=1, C_req=cores, M_req_MB=1.0,
                         W_MHz_s=cores * sv.f_MHz * dt)
            toy = _toy(scenario, [dataclasses.replace(sv, T0_C=30.0)], [w], 2,
                       f"power-{sv.name}")
            st = RunSettings(scenario=toy.name, reassignment="none",
                             time_limit_s=TOY_TIME_LIMIT_S)
            res = solve(toy, {1: 1}, st)
            if not res.has_schedule:
                raise RuntimeError(f"power check of {sv.name}: no schedule ({res.status})")
            ev = evaluate(toy, res.schedule, res.supply_temp_C, st)
            opt_p = res.model_values["P_IT_W"][0]
            measured[cores / sv.C_cores] = (float(opt_p[0]), float(ev.P_IT_W[0, 0]))
            measured[0.0] = (float(opt_p[1]), float(ev.P_IT_W[0, 1]))
        for share in sorted(measured):
            opt_w, ev_w = measured[share]
            expected = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * share
            ok = abs(opt_w - expected) <= POWER_TOL_W and abs(ev_w - expected) <= POWER_TOL_W
            rows.append({
                "Server": sv.name, "Speed (MHz)": sv.f_MHz,
                "CPU use (%)": round(share * 100, 1),
                "Idle power (W)": sv.P_idle_W, "Maximum power (W)": sv.P_max_W,
                "Expected power (W)": round(expected, 3),
                "Optimizer power (W)": round(opt_w, 3),
                "Evaluator power (W)": round(ev_w, 3),
                "Pass": ok,
            })
    return rows


# ---------------------------------------------------------------------------
# B4: server speed in the execution time; B5: the same task on server A or B
# ---------------------------------------------------------------------------

def _single_run(scenario: Scenario, w: Workload, sv, n: int, supply_C: float,
                settings: RunSettings):
    """Evaluate ``w`` alone on ``sv`` (in its own fleet) from arrival for ``n`` periods. A run
    past the end of the horizon is evaluated on a horizon extended to its end (the carbon
    intensity of the last period repeated), so its work is still counted."""
    one = dataclasses.replace(w, i=1)
    T = max(scenario.T, w.A + n - 1)
    ci = np.asarray(scenario.CI_g_per_kWh, dtype=float)
    ci = np.concatenate([ci, np.full(T - scenario.T, ci[-1])])
    sc = dataclasses.replace(scenario, workloads=(one,), T=T, CI_g_per_kWh=ci, AV=None)
    sched = Schedule((Placement(1, sv.s, w.A, w.A + n),))
    return sc, evaluate(sc, sched, constant_supply_temp(sc, supply_C), settings)


def execution_check(scenario: Scenario, workloads=EXAMPLE_WORKLOADS, servers=EXAMPLE_SERVERS,
                    start_hour: float | None = None) -> list[dict]:
    """The same workload (same work, cores, start at arrival) on each server: the run length
    from the model's rule, ceil(W / (cores x MHz x DPF x period)), and whether the evaluator
    agrees that this is the shortest run that delivers the work, and whether it meets the
    deadline. Each workload runs alone, at the top supply temperature, so DPF is the
    evaluator's (1 unless the server passes T_dvfs)."""
    st = RunSettings(scenario=scenario.name)
    dt = scenario.dt_seconds
    supply = _top_supply(scenario)
    rows = []
    for i in workloads:
        w = _workload(scenario, i)
        for name in servers:
            sv = _server(scenario, name)
            exact = w.W_MHz_s / (w.C_req * sv.f_MHz * dt)
            n = periods_needed(w, sv, dt)
            _, ev = _single_run(scenario, w, sv, n, supply, st)
            enough = bool(ev.work_complete[0])
            short = n > 1 and bool(_single_run(scenario, w, sv, n - 1, supply, st)[1].work_complete[0])
            end = w.A + n
            rows.append({
                "Workload": w.i, "Type": "time-sensitive" if w.RT else "delay-tolerant",
                "Cores": w.C_req, "Work required (MHz·s)": w.W_MHz_s,
                "Server": sv.name, "Speed (MHz)": sv.f_MHz,
                "Lowest DPF during the run": round(float(ev.min_DPF_run[0]), 4),
                "Periods, exact": round(exact, 2), "Periods, rounded up": n,
                "Execution time (min)": n * dt / 60,
                "Start (period)": w.A, "Start (clock)": _clock(w.A, start_hour, dt),
                "Finish (period)": end, "Finish (clock)": _clock(end, start_hour, dt),
                "Deadline (period)": w.D, "Deadline (clock)": _clock(w.D, start_hour, dt),
                "Meets deadline": end <= w.D,
                "Pass": enough and not short,
            })
    return rows


def equal_time_count(scenario: Scenario, servers=EXAMPLE_SERVERS) -> dict:
    """How many workloads get the same number of periods on every server in ``servers``,
    because the run length is rounded up to whole periods."""
    svs = [_server(scenario, n) for n in servers]
    same = [w.i for w in scenario.workloads
            if len({periods_needed(w, sv, scenario.dt_seconds) for sv in svs}) == 1]
    return {"servers": list(servers), "workloads": scenario.I, "same_periods": len(same),
            "examples": same[:10]}


def server_choice_check(scenario: Scenario, workloads=EXAMPLE_WORKLOADS,
                        servers=EXAMPLE_SERVERS) -> list[dict]:
    """The same workload on server A or B: how long it runs, the server electricity it
    causes (idle part and added part), the fleet's electricity over the whole day, and
    whether it meets the deadline. Every server draws its idle power all day whatever runs
    on it (the model has no on/off decision), so the fleet's idle electricity is the same
    in every option. A run past the end of the day counts in full for the server, and up to
    the end of the day for the fleet."""
    st = RunSettings(scenario=scenario.name)
    dt = scenario.dt_seconds
    supply = _top_supply(scenario)
    to_kwh = dt / 3.6e6
    idle_day = sum(sv.P_idle_W for sv in scenario.servers) * scenario.T * to_kwh
    rows = []
    for i in workloads:
        w = _workload(scenario, i)
        for name in servers:
            sv = _server(scenario, name)
            n = periods_needed(w, sv, dt)
            _, ev = _single_run(scenario, w, sv, n, supply, st)
            k, lo, hi = sv.s - 1, w.A - 1, w.A - 1 + n
            during = float(ev.P_IT_W[k, lo:hi].sum()) * to_kwh
            idle_part = sv.P_idle_W * (hi - lo) * to_kwh
            rows.append({
                "Workload": w.i, "Cores": w.C_req, "Server": sv.name, "Speed (MHz)": sv.f_MHz,
                "Execution time (min)": n * dt / 60,
                "Server electricity during the run (kWh)": round(during, 5),
                "of which idle (kWh)": round(idle_part, 5),
                "of which added by the workload (kWh)": round(during - idle_part, 5),
                "Fleet idle electricity, whole day (kWh)": round(idle_day, 3),
                "Fleet IT electricity, whole day (kWh)": round(float(ev.E_IT_kWh[:scenario.T].sum()), 5),
                "Meets deadline": w.A + n <= w.D,
            })
    return rows


# ---------------------------------------------------------------------------
# B2: reassignment where a move is needed or helps
# ---------------------------------------------------------------------------

def _pick_pair(scenario: Scenario, key) -> tuple[str, str]:
    """(worst, best) server by ``key``, lowest number first among equals."""
    ranked = sorted(scenario.servers, key=lambda sv: (key(sv), sv.s))
    return ranked[-1].name, ranked[0].name


def reassignment_check(scenario: Scenario, necessary=None, helpful=None,
                       T: int = 24) -> list[dict]:
    """Two small cases built from the scenario's real servers, each solved with
    reassignment "none" and "free".

    *Move needed*: a time-sensitive workload starts on a slow server where it cannot meet its
    deadline; on the fast server it can. Expected: "none" has no valid schedule, "free"
    moves it and meets the deadline.
    *Move helps*: a delay-tolerant workload starts on the server that uses the most
    electricity per unit of work; it fits on both. Expected: "free" moves it to the most
    efficient server and emits less carbon than "none".

    ``necessary`` and ``helpful`` are (original server, other server) names; by default the
    slowest and fastest, and the least and most efficient servers."""
    dt = scenario.dt_seconds
    necessary = necessary or _pick_pair(scenario, lambda sv: -sv.f_MHz)
    helpful = helpful or _pick_pair(
        scenario, lambda sv: (sv.P_max_W - sv.P_idle_W) / (sv.C_cores * sv.f_MHz))

    cases = []
    slow, fast = (_server(scenario, n) for n in necessary)
    n_fast = 4
    cores = min(8, slow.C_cores, fast.C_cores)
    work = cores * fast.f_MHz * dt * n_fast
    w = Workload(i=1, workload_id="move-needed", flavor="test", A=2, D=2 + n_fast, AD=0,
                 ET_base=n_fast, RT=1, B=0, X0=1, C_req=cores, M_req_MB=8192.0, W_MHz_s=work)
    cases.append(("Move needed", "time-sensitive", (slow, fast), w))

    worse, better = (_server(scenario, n) for n in helpful)
    n_better = 12
    cores = min(16, worse.C_cores, better.C_cores)
    work = cores * better.f_MHz * dt * n_better
    w = Workload(i=1, workload_id="move-helps", flavor="test", A=2, D=T + 1,
                 AD=T + 1 - 2 - n_better, ET_base=n_better, RT=0, B=0, X0=1, C_req=cores,
                 M_req_MB=16384.0, W_MHz_s=work)
    cases.append(("Move helps", "delay-tolerant", (worse, better), w))

    rows = []
    for case, kind, (orig, other), w in cases:
        toy = _toy(scenario, [orig, other], [w], T, f"reassign-{case}")
        results = {}
        for mode in ("none", "free"):
            st = RunSettings(scenario=toy.name, reassignment=mode, time_limit_s=TOY_TIME_LIMIT_S)
            res = solve(toy, {1: 1}, st)
            ev = (evaluate(toy, res.schedule, res.supply_temp_C, st)
                  if res.has_schedule else None)
            results[mode] = (res, ev)
        for mode, (res, ev) in results.items():
            p = res.schedule.placements[0] if res.has_schedule else None
            on = toy.servers[p.server - 1].name if p else None
            if case == "Move needed":
                expected = ("no valid schedule" if mode == "none"
                            else f"moves to {other.name}, on time")
                ok = (res.status == STATUS_INFEASIBLE if mode == "none" else
                      res.status == STATUS_OPTIMAL and on == other.name and p.end <= w.D)
            else:
                expected = (f"stays on {orig.name}" if mode == "none"
                            else f"moves to {other.name}, less carbon")
                none_ev = results["none"][1]
                ok = (res.status == STATUS_OPTIMAL and on == orig.name if mode == "none" else
                      res.status == STATUS_OPTIMAL and on == other.name and none_ev is not None
                      and ev.total_carbon_g < none_ev.total_carbon_g)
            rows.append({
                "Case": case, "Workload type": kind, "Cores": w.C_req,
                "Starts on": orig.name, "Other server": other.name,
                "Reassignment": mode, "Solver status": status_text(res.status),
                "Runs on": on, "Moved": None if p is None else on != orig.name,
                "Start (period)": p.start if p else None,
                "Finish (period)": p.end if p else None, "Deadline (period)": w.D,
                "Carbon (g CO2)": round(ev.total_carbon_g, 3) if ev else None,
                "Expected": expected, "Pass": bool(ok),
            })
    return rows


# ---------------------------------------------------------------------------
# B1: the CRAC supply temperature is chosen by the solver
# ---------------------------------------------------------------------------

def crac_check(scenario: Scenario, h_mins=(None, 18.0, 19.0, 20.0),
               time_limit_s: float = 120.0) -> tuple[list[dict], list[dict], list[dict]]:
    """Solve the scenario with reassignment "none" (proven best in seconds) at several safety
    margins H_min, with the supply temperature optimized per zone and period. Returns a
    summary row per H_min, the supply temperature of every zone and period, and the CRAC's
    COP at each whole-degree supply temperature.

    With a wide margin no temperature limit binds and the solver keeps the supply at the top
    of its range, where the CRAC's COP is highest. Narrow enough, a server would pass its cap
    and the solver lowers the supply in just those periods. Too narrow, and no schedule
    exists (the infeasibility diagnosis names the limit)."""
    c = scenario.constants
    top = _top_supply(scenario)
    rows, supply_rows = [], []
    for h in h_mins:
        st = RunSettings(scenario=scenario.name, reassignment="none", H_min_C=h,
                         time_limit_s=time_limit_s)
        base = build_baseline(scenario, st)
        original = base.server_of()
        res = solve(scenario, original, st, warm_start=base,
                    warm_start_supply_temp_C=constant_supply_temp(scenario, st.baseline_supply_temp_C))
        h_val = st.h_min(scenario)
        row = {"H_min (°C)": h_val, "Thermal cap (°C)": st.thermal_cap_C(scenario),
               "Solver status": status_text(res.status)}
        if res.has_schedule:
            ev = evaluate(scenario, res.schedule, res.supply_temp_C, st)
            sup = np.asarray(res.supply_temp_C)
            below = []
            for z in range(scenario.Z):
                lows = [(t + 1, sup[z, t]) for t in range(scenario.T) if sup[z, t] < top - 1e-9]
                below += [f"zone {z + 1}, period {t}: {v:g} °C" for t, v in lows]
                row[f"Zone {z + 1} supply, lowest (°C)"] = float(sup[z].min())
                row[f"Zone {z + 1} periods below {top:g} °C"] = len(lows)
                for t in range(scenario.T):
                    supply_rows.append({"H_min (°C)": h_val, "Zone": z + 1, "Period": t + 1,
                                        "Supply temperature (°C)": float(sup[z, t])})
            row.update({
                "Hottest server (°C)": round(float(ev.T_C.max()), 2),
                "Carbon (kg CO2)": round(ev.total_carbon_g / 1000, 4),
                "Lower supply in": "; ".join(below) or "none",
            })
        elif res.status == STATUS_INFEASIBLE:
            diag = diagnose_infeasibility(scenario, original, st, time_limit_s=time_limit_s)
            row["Lower supply in"] = ("no valid schedule; feasible only without: "
                                      + (", ".join(f for f, v in diag.items() if v == "feasible")
                                         or "none of the limits"))
        rows.append(row)
    cop_rows = [{"Supply temperature (°C)": float(k), "COP": round(float(cop(c, k)), 4)}
                for k in range(math.ceil(c.T_crac_min_C), int(top) + 1)]
    return rows, supply_rows, cop_rows


# ---------------------------------------------------------------------------
# B6: a hot server slows down and the run gets longer
# ---------------------------------------------------------------------------

def slowdown_check(scenario: Scenario, server: str | None = None, share: float = 0.75,
                   periods: int = 6, supplies=(18.0, None), h_min: float = 5.0) -> list[dict]:
    """One workload that fills ``share`` of a server's cores and needs exactly ``periods``
    periods at full speed, solved alone on that server with the supply temperature fixed.
    At the cooler supply the server stays below T_dvfs and the run takes ``periods``; at the
    warmer one it passes T_dvfs, DPF drops below 1 and the run needs an extra period.
    ``h_min`` 5 °C (the 6.3 low margin) leaves room above T_dvfs before the cap. A row passes
    when the run takes exactly ``periods`` with DPF 1 throughout, and longer once DPF drops.
    By default the server is the one without a cold plate that heats up fastest under this load
    (cold plates are on in every run, and a plate keeps its server below T_dvfs)."""
    c = scenario.constants
    dt = scenario.dt_seconds
    top = _top_supply(scenario)
    supplies = tuple(top if s is None else float(s) for s in supplies)
    if server is None:
        def heat(sv):
            p = sv.P_idle_W + (sv.P_max_W - sv.P_idle_W) * share
            temp = sv.T0_C
            for _ in range(periods):
                temp = sv.th_alpha * temp + (1 - sv.th_alpha) * (top + sv.th_a_C + sv.th_R_C_per_W * p)
            return temp
        air_only = [sv for sv in scenario.servers if sv.plates == 0] or list(scenario.servers)
        server = max(air_only, key=heat).name
    sv = _server(scenario, server)
    cores = max(1, round(share * sv.C_cores))
    work = cores * sv.f_MHz * dt * periods
    T = periods + 4
    w = Workload(i=1, workload_id="slowdown", flavor="test", A=1, D=T + 1, AD=0,
                 ET_base=periods, RT=0, B=0, X0=1, C_req=cores, M_req_MB=1.0, W_MHz_s=work)
    toy = _toy(scenario, [sv], [w], T, f"slowdown-{sv.name}")
    rows = []
    for sup in supplies:
        st = RunSettings(scenario=toy.name, reassignment="none", supply_temp_mode="fixed",
                         fixed_supply_temp_C=sup, H_min_C=h_min, time_limit_s=TOY_TIME_LIMIT_S)
        res = solve(toy, {1: 1}, st)
        row = {"Server": sv.name, "Cores": cores, "CPU use (%)": round(cores / sv.C_cores * 100, 1),
               "Supply temperature (°C)": sup, "H_min (°C)": h_min,
               "T_dvfs (°C)": c.T_dvfs_C, "Periods at full speed": periods,
               "Solver status": status_text(res.status)}
        if res.has_schedule:
            ev = evaluate(toy, res.schedule, res.supply_temp_C, st)
            p = res.schedule.placements[0]
            idx = slice(p.start - 1, p.end - 1)
            row.update({
                "Hottest temperature during the run (°C)": round(float(ev.T_C[0, idx].max()), 2),
                "Lowest DPF during the run": round(float(ev.DPF[0, idx].min()), 4),
                "Periods run": p.end - p.start,
                "Execution time (min)": (p.end - p.start) * dt / 60,
                "Work delivered / required": round(float(ev.work_delivered_MHz_s[0]) / work, 4),
                "Valid under the evaluator": ev.valid,
            })
        rows.append(row)
    for r in rows:
        if "Periods run" in r:
            # at full speed the work fits exactly, so any slowdown needs an extra period
            slowed = r["Lowest DPF during the run"] < 1.0
            r["Pass"] = r["Valid under the evaluator"] and (
                r["Periods run"] > periods if slowed else r["Periods run"] == periods)
    return rows


def throttled_run_rows(run_dir: Path, scenario: Scenario) -> tuple[list[dict], list[dict]]:
    """From a solved run: every server that slowed (DPF < 1), and every workload that ran
    while its server was slowed, with its run length against the full-speed length."""
    run_dir = Path(run_dir)
    sp = _read_csv(run_dir / "optimized" / "server_period.csv")
    sched = _read_csv(run_dir / "optimized" / "schedule.csv")
    names = {sv.s: sv.name for sv in scenario.servers}
    slow: dict[int, list[tuple[int, float, float]]] = {}
    for r in sp:
        d = float(r["DPF"])
        if d < 1.0 - 1e-9:
            slow.setdefault(int(r["s"]), []).append((int(r["t"]), d, float(r["temperature_C"])))
    servers = [{
        "Server": names.get(s, s), "Periods slowed": len(v),
        "Periods": ", ".join(str(t) for t, _, _ in v),
        "Lowest DPF": round(min(d for _, d, _ in v), 4),
        "Hottest temperature (°C)": round(max(x for _, _, x in v), 2),
    } for s, v in sorted(slow.items())]
    slow_set = {(s, t) for s, v in slow.items() for t, _, _ in v}
    work = []
    for r in sched:
        if r.get("placed") in ("0", "") or not r.get("server"):
            continue
        s, start, n = int(r["server"]), int(r["start"]), int(r["periods_run"])
        hit = [t for t in range(start, start + n) if (s, t) in slow_set]
        if not hit:
            continue
        w = scenario.workloads[int(r["i"]) - 1]
        full = periods_needed(w, scenario.servers[s - 1], scenario.dt_seconds)
        work.append({
            "Workload": w.i, "Server": names.get(s, s), "Periods slowed": len(hit),
            "Periods at full speed": full, "Periods run": n, "Extra periods": n - full,
            "Spare work from rounding up (%)": round(
                (full * w.C_req * scenario.servers[s - 1].f_MHz * scenario.dt_seconds
                 / w.W_MHz_s - 1) * 100, 2),
            "Work delivered / required": round(float(r["work_delivered_MHz_s"])
                                               / float(r["work_required_MHz_s"]), 4),
            "Finish (period)": int(r["finish"]), "Deadline (period)": int(r["deadline"]),
        })
    return servers, work


# ---------------------------------------------------------------------------
# B8: cold plates, checked on a run that has them
# ---------------------------------------------------------------------------

def cold_plate_rows(run_dir: Path, scenario: Scenario,
                    no_plate_run_dir: Path | None = None) -> list[dict]:
    """Every limit of the cold plates, per server with plates, from a solved run; and the
    server's hottest temperature against a run without cold plates, if given."""
    run_dir = Path(run_dir)
    summary = json.loads((run_dir / "summary.json").read_text())
    cps = summary.get("cold_plates") or {}
    if not cps.get("enabled"):
        raise ValueError(f"{run_dir.name} has cold plates off")
    plates = cps["plates_per_server"]
    flows = [float(f) for f in cps["flow_settings_Lmin"]]
    lo_c, hi_c = cps["coolant_limits_C"]
    q_max = float(cps["Q_max_per_plate_W"])
    sp = _read_csv(run_dir / "optimized" / "server_period.csv")
    hot_without = {}
    if no_plate_run_dir is not None and (Path(no_plate_run_dir) / "optimized").is_dir():
        for r in _read_csv(Path(no_plate_run_dir) / "optimized" / "server_period.csv"):
            s = int(r["s"])
            hot_without[s] = max(hot_without.get(s, -1e9), float(r["temperature_C"]))
    dt_h = scenario.dt_seconds / 3600.0
    rows = []
    for sv in scenario.servers:
        n = int(plates[sv.s - 1])
        if n == 0:
            continue
        mine = [r for r in sp if int(r["s"]) == sv.s]
        q = [float(r["Q_cp_W"]) for r in mine]
        p_it = [float(r["P_IT_W"]) for r in mine]
        flow = [float(r["cp_flow_Lmin"]) for r in mine if r["cp_flow_Lmin"] != ""]
        cool = [float(r["cp_coolant_C"]) for r in mine if r["cp_coolant_C"] != ""]
        pump_wh = sum(float(r["P_cp_W"]) for r in mine) * dt_h
        hot = max(float(r["temperature_C"]) for r in mine)
        checks = {
            "Heat removed ≤ server heat": all(a <= b + 1e-6 for a, b in zip(q, p_it)),
            f"Heat removed ≤ {q_max:g} W per plate": max(q) <= n * q_max + 1e-6,
            "Flow is one of the settings": all(any(abs(f - x) < 1e-6 for x in flows) for f in flow),
            "Coolant within limits": all(lo_c - 1e-6 <= x <= hi_c + 1e-6 for x in cool),
        }
        rows.append({
            "Server": sv.name, "Cold plates": n,
            "Peak heat removed (W)": round(max(q), 2), "Peak server heat (W)": round(max(p_it), 2),
            "Heat removed over the day (kWh)": round(sum(q) * dt_h / 1000, 4),
            "Flow used (L/min)": ", ".join(f"{x:g}" for x in sorted(set(flow))),
            "Coolant used (°C)": f"{min(cool):g}–{max(cool):g}" if cool else "",
            "Pump electricity (Wh)": round(pump_wh, 5),
            "Hottest temperature (°C)": round(hot, 2),
            "Hottest without cold plates (°C)": (round(hot_without[sv.s], 2)
                                                  if sv.s in hot_without else None),
            **checks,
            "Pass": all(checks.values()),
        })
    return rows


# ---------------------------------------------------------------------------
# B3 and A1: how complete each solver result is
# ---------------------------------------------------------------------------

def runs_of(results_dir: Path, scenario_name: str) -> list[Path]:
    """Every run folder under ``<results_dir>/<folder>/<run>/`` solved on the scenario, by the
    name in its summary.json: ``results/<scenario>/`` as the notebooks write it, or a folder
    such as ``results/servers10/`` that holds renamed runs."""
    out = []
    for d in sorted(Path(results_dir).glob("*/*/summary.json")):
        if d.parent.parent.name.startswith("archive-"):
            continue
        try:
            name = (json.loads(d.read_text()).get("scenario") or {}).get("name")
        except (OSError, ValueError):
            continue
        if (name or d.parent.parent.name) == scenario_name:
            out.append(d.parent)
    return out


def _named_run(runs: list[Path], names: tuple[str, ...]) -> Path | None:
    for name in names:
        for d in runs:
            if d.name == name:
                return d
    return None


def _plates_on(run_dir: Path) -> bool:
    """True if the run modelled cold plates (a run made before they were on by default may not)."""
    summary = json.loads((Path(run_dir) / "summary.json").read_text())
    return bool((summary.get("cold_plates") or {}).get("enabled"))


def solver_rows(results_dir: Path, scenario_name: str) -> list[dict]:
    """One row per run of the scenario: what it allowed, the solver's status in plain words,
    the gap, the solve time, carbon, the solver's lower bound, and the most a better
    schedule could still save (carbon minus bound)."""
    rows = []
    for run in runs_of(results_dir, scenario_name):
        d = run / "summary.json"
        s = json.loads(d.read_text())
        st, sol = s.get("settings") or {}, s.get("solver") or {}
        opt = (s.get("optimized") or {}).get("metrics") or {}
        carbon = opt.get("carbon_kg") if s.get("has_optimized_schedule") else None
        bound = sol.get("best_bound_g")
        supply = ("optimized" if st.get("supply_temp_mode") == "optimize" else
                  f"fixed {st.get('fixed_supply_temp_C') or st.get('baseline_supply_temp_C'):g} °C")
        rows.append({
            "Run": d.parent.name,
            "Release": (s.get("provenance") or {}).get("thermasched_version"),
            "Reassignment": str(st.get("reassignment", "")).replace("_", " "),
            "CRAC supply": supply,
            "Cold plates": bool(st.get("cold_plates")),
            "H_min (°C)": st.get("H_min_C"),
            "Time limit (s)": st.get("time_limit_s"),
            # a thermal-blind run reports its second pass, which keeps the servers fixed
            "Solver status": status_text(sol.get("status")) + (
                " (servers fixed in pass 2)" if st.get("placement") == "thermal_blind" else ""),
            "MIP gap (%)": None if sol.get("mip_gap") is None else round(sol["mip_gap"] * 100, 3),
            "Solve time (s)": None if sol.get("solve_time_s") is None else round(sol["solve_time_s"], 1),
            "Carbon (kg CO2)": None if carbon is None else round(carbon, 4),
            "Lower bound (kg CO2)": None if bound is None else round(bound / 1000, 4),
            "Most a better schedule could save (kg CO2)": (
                None if carbon is None or bound is None else round(max(0.0, carbon - bound / 1000), 4)),
            "Workloads moved": opt.get("n_reassigned"),
        })
    return rows


# ---------------------------------------------------------------------------
# Writing every table
# ---------------------------------------------------------------------------

def _read_csv(path: Path) -> list[dict]:
    with open(path, newline="", encoding="utf-8-sig") as f:
        return list(csv.DictReader(f))


def write_table(path: Path, rows: list[dict]) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    fields: list[str] = []
    for r in rows:
        fields += [k for k in r if k not in fields]
    with open(path, "w", newline="", encoding="utf-8") as f:
        w = csv.DictWriter(f, fieldnames=fields)
        w.writeheader()
        for r in rows:
            w.writerow({k: ("" if v is None else v) for k, v in r.items()})


# Runs the run-based checks read, under results/<scenario>/ (or another scenario's folder).
# The run with cold plates on, by the names a run may have: the short Chapter 6 name, or the
# name the solver gives it. Cold plates are on by default, so it is the base run.
COLD_PLATE_RUN = ("base", "least_loaded_base18_free_optimize_tl1800")
WARM_RUN = ("thesis10_warm", "least_loaded_base18_free_optimize")


def _passed(rows, key="Pass"):
    vals = [r[key] for r in rows if r.get(key) is not None]
    return None if not vals else all(vals)


def write_all(scenario_name: str = "servers6", scenarios_dir="scenarios", results_dir="results",
              out_dir=None, log=print) -> dict:
    """Run every check on the scenario and write the tables to
    ``<results_dir>/validation/<scenario>/``. Returns what validation.json holds."""
    scenarios_dir, results_dir = Path(scenarios_dir), Path(results_dir)
    out = Path(out_dir) if out_dir else results_dir / "validation" / scenario_name
    sc = load_scenario(scenario_name, scenarios_dir)
    params = json.loads((scenarios_dir / scenario_name / "params.json").read_text())
    hour = (params.get("time") or {}).get("period_1_local_hour")
    checks: dict = {}

    def table(key, rows, note=None):
        write_table(out / f"{key}.csv", rows)
        checks[key] = {"rows": len(rows), "passed": _passed(rows), **({"note": note} if note else {})}
        log(f"  {key}: {len(rows)} rows, passed = {checks[key]['passed']}")

    log(f"Validation of {scenario_name} (release {VERSION}) -> {out}/")
    table("workload_inputs", workload_inputs(sc, hour))
    table("power", power_check(sc))
    examples = [n for n in EXAMPLE_SERVERS if any(sv.name == n for sv in sc.servers)]
    servers = examples if len(examples) == 2 else [sc.servers[0].name, sc.servers[-1].name]
    work = [i for i in EXAMPLE_WORKLOADS if i <= sc.I] or [1]
    table("execution_time", execution_check(sc, work, servers, hour))
    checks["execution_time"]["equal_periods"] = equal_time_count(sc, servers)
    table("server_choice", server_choice_check(sc, work, servers))
    table("reassignment", reassignment_check(sc, necessary=tuple(servers)))
    crac, supply, cops = crac_check(sc)
    table("crac", crac)
    table("crac_supply", supply)
    table("crac_cop", cops)
    table("slowdown", slowdown_check(sc))

    warm_dir = results_dir / WARM_RUN[0] / WARM_RUN[1]
    if (warm_dir / "summary.json").is_file() and (scenarios_dir / WARM_RUN[0]).is_dir():
        srv, wl = throttled_run_rows(warm_dir, load_scenario(WARM_RUN[0], scenarios_dir))
        table("throttled_servers", srv, note=f"{WARM_RUN[0]}/{WARM_RUN[1]}")
        table("throttled_workloads", wl, note=f"{WARM_RUN[0]}/{WARM_RUN[1]}")
    runs = runs_of(results_dir, scenario_name)
    cp_dir = _named_run(runs, COLD_PLATE_RUN)
    if cp_dir is not None and _plates_on(cp_dir):
        table("cold_plates", cold_plate_rows(cp_dir, sc),
              note=f"{cp_dir.parent.name}/{cp_dir.name}")
    table("solver", solver_rows(results_dir, scenario_name))

    meta = {"scenario": scenario_name, "scenario_fingerprint": sc.fingerprint,
            "thermasched_version": VERSION,
            "created_utc": datetime.now(timezone.utc).strftime("%Y-%m-%dT%H:%M:%SZ"),
            "period_1_local_hour": hour, "dt_seconds": sc.dt_seconds,
            "checks": checks}
    (out / "validation.json").write_text(json.dumps(meta, indent=2) + "\n")
    return meta


def main(argv=None) -> int:
    p = argparse.ArgumentParser(prog="scheduler.validate", description=__doc__.split("\n\n")[0])
    p.add_argument("--scenario", default="servers6")
    p.add_argument("--scenarios-dir", default="scenarios")
    p.add_argument("--results-dir", default="results")
    a = p.parse_args(argv)
    meta = write_all(a.scenario, a.scenarios_dir, a.results_dir)
    failed = [k for k, v in meta["checks"].items() if v["passed"] is False]
    if failed:
        print(f"FAILED: {failed}", file=sys.stderr)
        return 1
    return 0


if __name__ == "__main__":
    sys.exit(main())


In [ ]:
%%writefile {LIB}/scheduler/experiments.py
"""Chapter 6 experiments: the fleet configurations and the tests each one goes through.

One list for the solver, the batch notebook and the site. A run belongs to a test when its
configuration and its switches match the test's, so a folder name never decides it: a run
made in the batch notebook or uploaded on the site is recognized too.

A configuration is a fleet on a load level. Fleets: 10 servers (3 slow / 4 mid / 3 fast),
6 servers (2 / 2 / 2) and 2 servers (2 mid). The slow (2.1 GHz) and mid (2.4 GHz) servers are
Dell R840s cooled by CRAC air; the fast (2.6 GHz) ones are Dell C6320s cooled by CRAC air and a
cold plate. The frequencies are those of SPEC CPU results for these models. Load levels: a
light, normal and heavy day, whose workloads are drawn until peak demand reaches 30, 60 or
85 % of the fleet's cores or memory, whichever fills first. The normal day of a fleet is scenario ``servers10`` (``servers6``,
``servers2``), the others ``servers10_light`` and ``servers10_heavy``; each has a ``_swing``
twin with a deeper daily carbon swing (6.4 Test 3b). ``thesis10`` and ``consolidated6`` (and
their ``_swing``) are the earlier names of the 10- and 6-server normal days.
"""

from __future__ import annotations

from dataclasses import dataclass

from scheduler.types import RunSettings

# Server tiers, slowest first: name -> clock frequency (MHz).
TIERS = {"slow": 2100.0, "mid": 2400.0, "fast": 2600.0}
COLD_PLATE_TIER = "fast"        # servers of this tier carry a cold plate (CT_s = 1)

Switches = tuple[tuple[str, object], ...]
SWING: Switches = (("ci_amplitude", 0.5),)


@dataclass(frozen=True)
class Level:
    key: str                    # "light", "normal", "heavy"
    label: str                  # "Normal day"
    target: float               # peak demand, as a share of the fleet's cores or memory

    @property
    def description(self) -> str:
        return (f"workloads drawn until peak demand reaches {self.target:.0%} of the fleet's "
                "cores or memory, whichever fills first")


LEVELS: tuple[Level, ...] = (
    Level("light", "Light day", 0.30),
    Level("normal", "Normal day", 0.60),
    Level("heavy", "Heavy day", 0.85),
)
LEVEL_BY_KEY = {lv.key: lv for lv in LEVELS}
NORMAL = "normal"


@dataclass(frozen=True)
class Configuration:
    key: str                    # results/<key>/ and the scenario name: servers10, servers10_light
    tiers: tuple[str, ...]      # tier of each server, in server order
    aliases: tuple[tuple[str, Switches], ...]   # scenario names holding it -> switches implied
    regroup_zone_map: tuple[int, ...] | None    # 6.1 Test 1; None: not applicable
    fleet_key: str = ""         # servers10, servers6, servers2
    level: str = NORMAL         # a LEVELS key

    @property
    def n_servers(self) -> int:
        return len(self.tiers)

    @property
    def size_label(self) -> str:
        return f"{self.n_servers} servers"

    @property
    def level_label(self) -> str:
        return LEVEL_BY_KEY[self.level].label

    @property
    def label(self) -> str:
        """'10 servers · normal day'."""
        return f"{self.size_label} · {self.level_label.lower()}"

    @property
    def fleet(self) -> str:
        """'4 slow · 3 mid · 3 fast'."""
        return " · ".join(f"{self.tiers.count(t)} {t}" for t in TIERS if t in self.tiers)

    @property
    def has_cold_plates(self) -> bool:
        return COLD_PLATE_TIER in self.tiers


def scenario_name(fleet_key: str, level: str) -> str:
    """The scenario of a fleet on a load level: servers10, servers10_light, servers10_heavy."""
    return fleet_key if level == NORMAL else f"{fleet_key}_{level}"


def _fleet(fleet_key: str, tiers: tuple[str, ...], regroup: tuple[int, ...] | None,
           earlier: tuple[str, ...] = ()) -> tuple[Configuration, ...]:
    """The fleet on each load level; ``earlier`` names its normal day had before."""
    out = []
    for level in LEVELS:
        names = [scenario_name(fleet_key, level.key)]
        if level.key == NORMAL:
            names += earlier
        aliases = tuple(a for n in names for a in ((n, ()), (f"{n}_swing", SWING)))
        out.append(Configuration(names[0], tiers, aliases, regroup, fleet_key, level.key))
    return tuple(out)


CONFIGURATIONS: tuple[Configuration, ...] = (
    # 6.1 Test 1 regroups by tier: the slow tier in zone 1, the others in zone 2.
    *_fleet("servers10", ("slow",) * 3 + ("mid",) * 4 + ("fast",) * 3,
            (1, 1, 1, 2, 2, 2, 2, 2, 2, 2), earlier=("thesis10",)),
    *_fleet("servers6", ("slow",) * 2 + ("mid",) * 2 + ("fast",) * 2,
            (1, 1, 2, 2, 2, 2), earlier=("consolidated6",)),
    # Base zones alternate (server 1 in zone 1, server 2 in zone 2); regrouping puts both in
    # one zone. No fast server, so no cold plate: 6.2 does not apply.
    *_fleet("servers2", ("mid", "mid"), (1, 1)),
)
CONFIGURATION_BY_KEY = {c.key: c for c in CONFIGURATIONS}
# The fleets, in order: key -> its normal-day configuration (for the size and fleet labels).
FLEETS = {c.fleet_key: c for c in CONFIGURATIONS if c.level == NORMAL}


def configuration(fleet_key: str, level: str = NORMAL) -> Configuration:
    return CONFIGURATION_BY_KEY[scenario_name(fleet_key, level)]


# A switch whose value comes from the configuration (6.1 Test 1's zone map).
REGROUP = "<regroup zone map>"


@dataclass(frozen=True)
class Test:
    __test__ = False            # not a pytest test class
    id: str                     # also the run folder: results/<configuration>/<id>/
    name: str
    switches: Switches = ()
    needs: str | None = None    # "regroup" or "cold_plates": not applicable without it


TESTS: tuple[Test, ...] = (
    Test("base", "Base"),
    Test("s1_t1_zones", "Same-type servers regrouped into one zone",
         (("zone_map", REGROUP),), needs="regroup"),
    Test("s1_t2_same_zone", "Moves only within the original zone",
         (("reassignment", "same_zone"),)),
    # Cold plates are on in every run (RunSettings default), so 6.2's reference is the Base.
    Test("s2_t1_crac", "CRAC changed, cold plates fixed",
         (("cp_mode", "fixed"),), needs="cold_plates"),
    Test("s2_t2_cold_plate", "Cold plates changed, CRAC fixed",
         (("supply_temp_mode", "fixed"),), needs="cold_plates"),
    Test("s3_t1_hmin_high", "High safety margin, 19 °C", (("H_min_C", 19.0),)),
    Test("s3_t2_hmin_med", "Medium safety margin, 15 °C", (("H_min_C", 15.0),)),
    Test("s3_t3_hmin_low", "Low safety margin, 5 °C", (("H_min_C", 5.0),)),
    Test("s4_t1_no_moves", "No reassignment (r_is = 0)", (("reassignment", "none"),)),
    Test("s4_t2_blind", "Moves chosen without temperature",
         (("placement", "thermal_blind"),)),
    Test("s4_t3a_ci_flat", "Flat carbon intensity", (("ci_amplitude", 0.0),)),
    Test("s4_t3b_ci_swing", "Deeper daily carbon swing (amplitude 0.50)",
         (("ci_amplitude", 0.5),)),
)
TEST_BY_ID = {t.id: t for t in TESTS}


@dataclass(frozen=True)
class Section:
    key: str                    # "6.1"
    title: str
    question: str
    goal: str
    tests: tuple[tuple[str, str], ...]      # (tag, test id): the first is the reference
    note: str = ""


SECTIONS: tuple[Section, ...] = (
    Section("6.1", "Effect of cooling zone restrictions on workload reassignment",
            "How does the zoning of servers interact with workload reassignment?",
            "Determine whether grouping servers into cooling zones affects where workloads "
            "are assigned or reassigned, and whether restricting reassignment to the same "
            "zone affects performance.",
            (("Base", "base"), ("Test 1", "s1_t1_zones"), ("Test 2", "s1_t2_same_zone")),
            "Same servers, workloads, cooling settings and carbon intensity in every run. "
            "Test 1 changes only the zone of each server; Test 2 lets a waiting workload "
            "move only within its original zone, and workloads already running stay put."),
    Section("6.2", "Effect of cooling configuration on server temperature",
            "Which cooling system has a greater impact on reducing server temperature?",
            "Determine which cooling system has a greater effect on reducing server "
            "temperature.",
            (("Base", "base"), ("Test 1", "s2_t1_crac"), ("Test 2", "s2_t2_cold_plate")),
            "Both cooling systems stay active, as in every run; the Base optimizes both. "
            "Test 1 lets the CRAC supply air temperature change with the cold plates fixed; "
            "Test 2 lets the cold-plate coolant temperature and flow change with the CRAC "
            "fixed at 18 °C. Airflow is not modelled, so the CRAC changes only its supply air "
            "temperature."),
    Section("6.3", "Effect of thermal headroom on workload completion and QoS",
            "How does controlling thermal stress affect workload completion time and QoS "
            "under deadline-aware scheduling?",
            "Determine whether requiring servers to stay cooler affects how quickly "
            "workloads finish and whether deadlines are met.",
            (("Base", "base"), ("Test 1", "s3_t1_hmin_high"), ("Test 2", "s3_t2_hmin_med"),
             ("Test 3", "s3_t3_hmin_low")),
            "Only the minimum thermal headroom H_min changes: 10 °C in the base, then 19, 15 "
            "and 5 °C. Each server must stay H_min below its temperature limit."),
    Section("6.4", "Effect of thermal-aware reassignment on carbon footprint and QoS",
            "How does thermal-aware reassignment of workloads affect total carbon footprint "
            "and QoS?",
            "Determine whether considering temperature when reassigning waiting workloads "
            "reduces total carbon emissions while still meeting deadlines.",
            (("Base", "base"), ("Test 1", "s4_t1_no_moves"), ("Test 2", "s4_t2_blind"),
             ("Test 3a", "s4_t3a_ci_flat"), ("Test 3b", "s4_t3b_ci_swing")),
            "The base weighs server capacity, temperature and effective speed. Test 2 "
            "chooses servers without temperature, then applies every safety limit. Test 3 "
            "changes the shape of carbon intensity over the day: flat (3a) or a deeper swing "
            "(3b). Scaling it up or down by a constant would leave the schedule unchanged."),
)
SECTION_BY_KEY = {s.key: s for s in SECTIONS}


def sections_of(test_id: str) -> list[tuple[Section, str]]:
    """Every (section, tag) a test appears in; the base appears in every section."""
    return [(s, tag) for s in SECTIONS for tag, tid in s.tests if tid == test_id]


def title_of(test: Test) -> str:
    """'6.1 · Test 1 · Same-type servers regrouped into one zone'; the shared base is 'Base'."""
    where = sections_of(test.id)
    if test.id == "base" or not where:
        return test.name
    section, tag = where[0]
    return f"{section.key} · {tag} · {test.name}"


def applicable(test: Test, config: Configuration) -> bool:
    if test.needs == "regroup":
        return config.regroup_zone_map is not None
    if test.needs == "cold_plates":
        return config.has_cold_plates
    return True


def switches_for(test: Test, config: Configuration) -> dict:
    """The test's switches for this configuration (the regroup zone map filled in)."""
    return {k: (config.regroup_zone_map if v == REGROUP else v) for k, v in test.switches}


# ---------------------------------------------------------------------------
# Matching a run to its configuration and tests
# ---------------------------------------------------------------------------

# Settings that never decide a test: names, solver limits, the seed of a rule not used, and
# whether the optimizer re-plans every tick or plans the whole day at once.
IGNORED = frozenset({"scenario", "run_name", "time_limit_s", "mip_rel_gap", "seed", "per_tick",
                     "tick_time_limit_s"})


def configuration_of(scenario: str | None) -> tuple[Configuration, dict] | None:
    """The configuration a scenario holds, and the switches the scenario implies."""
    for config in CONFIGURATIONS:
        for name, implied in config.aliases:
            if name == scenario:
                return config, dict(implied)
    return None


def _value(key: str, value):
    if key == "zone_map" and value is not None:
        return tuple(int(z) for z in value)
    if isinstance(value, bool) or value is None or isinstance(value, str):
        return value
    if isinstance(value, (int, float)):
        return float(value)
    return value


def _defaults() -> dict:
    d = {k: v for k, v in RunSettings().as_dict().items() if k not in IGNORED}
    d.pop("ci_flat", None)
    d.pop("carbon_file", None)
    d["ci_amplitude"] = None            # the scenario's own carbon profile
    return {k: _value(k, v) for k, v in d.items()}


def effective_switches(settings: dict, implied: dict | None = None) -> dict:
    """A run's switches in the form the tests use. The carbon intensity becomes one switch,
    ``ci_amplitude``: 0 for a flat profile, the scenario's implied value, the run's own value,
    or None for the scenario's profile. A run with its own carbon file matches no test."""
    defaults = _defaults()
    have = {k: _value(k, settings.get(k, d)) for k, d in defaults.items()}
    amplitude = settings.get("ci_amplitude", (implied or {}).get("ci_amplitude"))
    if settings.get("ci_flat"):
        amplitude = 0.0
    have["ci_amplitude"] = _value("ci_amplitude", amplitude)
    if settings.get("carbon_file"):
        have["ci_amplitude"] = f"file:{settings['carbon_file']}"
    for k, v in (implied or {}).items():
        if k != "ci_amplitude":
            have[k] = _value(k, v)
    return have


def identify(settings: dict, scenario: str | None) -> tuple[Configuration | None, list[Test]]:
    """The configuration of a run (from its scenario) and the tests it matches: every test
    switch equal, every other switch at its default. A random baseline matches no test."""
    found = configuration_of(scenario)
    if found is None:
        return None, []
    config, implied = found
    if settings.get("baseline_rule", "least_loaded") == "random":
        return config, []
    have = effective_switches(settings, implied)
    defaults = _defaults()
    out = []
    for test in TESTS:
        if not applicable(test, config):
            continue
        want = dict(defaults)
        want.update({k: _value(k, v) for k, v in switches_for(test, config).items()})
        if have == want:
            out.append(test)
    return config, out


In [ ]:
%%writefile {LIB}/dashboard/data.py
"""Read-only loaders for ThermaSched run outputs.

A run lives in ``results/<scenario>/<run_name>/`` (see ``scheduler/report.py``). This module
turns one into tidy pandas frames: per-variant CSVs are stacked into one long frame with a
``variant`` column ("baseline" / "optimized"), servers carry their names from
``scenarios/<scenario>/servers.csv``, and every per-period frame gets the local clock time
from ``scenarios/<scenario>/carbon.csv``.

There is no Streamlit here so the loaders can be tested directly; the UI caches them.
"""

from __future__ import annotations

import dataclasses
import json
import os
from dataclasses import dataclass, field
from pathlib import Path

import numpy as np
import pandas as pd

from scheduler import experiments as ex
from scheduler.build import scenario_fingerprint
from scheduler.types import STATUS_OPTIMAL, STATUS_TIME_LIMIT, status_text
from scheduler.version import SCHEMA_VERSION, VERSION

REPO_ROOT = Path(__file__).resolve().parents[1]
RESULTS_DIR = Path(os.environ.get("THERMASCHED_RESULTS_DIR", REPO_ROOT / "results"))
SCENARIOS_DIR = Path(os.environ.get("THERMASCHED_SCENARIOS_DIR", REPO_ROOT / "scenarios"))
# Validation tables, results/validation/<scenario>/ (written by the validation checks).
VALIDATION_DIR = RESULTS_DIR / "validation"

VARIANTS = ("baseline", "optimized")
VARIANT_LABELS = {"baseline": "Baseline", "optimized": "Optimized"}
# The variant name of the Base in against_base: the test charts draw it dashed.
BASE_LABEL = "Base"

# Energy components in stacking order: column -> label.
ENERGY_COMPONENTS = {
    "E_IT_kWh": "IT",
    "E_crac_kWh": "CRAC",
    "E_cp_kWh": "Cold-plate pumps",
    "E_fan_kWh": "Fans",
}

# Workload types as written in schedule.csv.
WORKLOAD_TYPES = {"time_sensitive": "Time-sensitive", "delay_tolerant": "Delay-tolerant"}

# Constraint families relaxed one at a time by scheduler.optimize.diagnose_infeasibility.
RELAX_FAMILY_LABELS = {
    "deadlines_and_waiting": "Deadlines and waiting limits, Eq. (27)-(28)",
    "thermal_cap": "Thermal cap T <= T_max - H_min, Eq. (31)-(33)",
    "capacity": "CPU and memory capacity, Eq. (12)-(13)",
    "cooling_capacity": "CRAC cooling capacity, Eq. (54)",
    "priority": "Priority order, Eq. (56)-(57)",
}

# Monday 2000-01-03 as a dummy date: only the clock time is shown.
CLOCK_ORIGIN = pd.Timestamp("2000-01-03")
ZERO_TOL = 1e-12

# Server model of each tier: Dell PowerEdge R840 (CRAC) at the slow and mid clock, C6320
# (CRAC + cold plate) at the fast one. The clock of each tier is in ex.TIERS.
TIER_MODELS = {"slow": "R840", "mid": "R840", "fast": "C6320"}


def tier_ghz(tier: str) -> str:
    """'2.6 GHz'."""
    return f"{ex.TIERS[tier] / 1000:.1f} GHz"


def tier_text(tier: str) -> str:
    """'R840, 2.1 GHz'; the cold-plate tier adds ', cold plate'."""
    text = f"{TIER_MODELS[tier]}, {tier_ghz(tier)}"
    return text + (", cold plate" if tier == ex.COLD_PLATE_TIER else "")


def fleet_text(config: ex.Configuration) -> str:
    """'3 slow (R840, 2.1 GHz) · 4 mid (R840, 2.4 GHz) · 3 fast (C6320, 2.6 GHz, cold plate)'."""
    return " · ".join(f"{config.tiers.count(t)} {t} ({tier_text(t)})"
                      for t in ex.TIERS if t in config.tiers)


def cold_plate_server_text() -> str:
    """'a C6320 at 2.6 GHz': the servers that carry a cold plate."""
    return f"a {TIER_MODELS[ex.COLD_PLATE_TIER]} at {tier_ghz(ex.COLD_PLATE_TIER)}"


# ---------------------------------------------------------------------------
# Run discovery
# ---------------------------------------------------------------------------

@dataclass(frozen=True, order=True)
class RunRef:
    scenario: str
    name: str
    path: Path = field(compare=False)

    @property
    def key(self) -> str:
        return f"{self.scenario}/{self.name}"


# Folders under results/ named "archive-<scenario>" hold runs set aside (made before release
# 1.0.0). They are listed after the current scenarios; their summary.json still names the
# real scenario, whose servers.csv and carbon.csv they use.
ARCHIVE_PREFIX = "archive-"


def is_archive(scenario: str) -> bool:
    return scenario.startswith(ARCHIVE_PREFIX)


def scenario_order(scenario: str) -> tuple:
    """Current scenarios first, then the archived ones, each alphabetically."""
    return is_archive(scenario), scenario


def run_order(ref: RunRef) -> tuple:
    return scenario_order(ref.scenario) + (ref.name,)


def discover_runs(results_dir: str | Path = RESULTS_DIR) -> list[RunRef]:
    """Every ``<results_dir>/<scenario>/<run>/summary.json``, sorted by scenario and run."""
    root = Path(results_dir)
    if not root.is_dir():
        return []
    return sorted((RunRef(p.parent.parent.name, p.parent.name, p.parent)
                   for p in root.glob("*/*/summary.json")), key=run_order)


def load_summary(run_dir: str | Path) -> dict:
    with open(Path(run_dir) / "summary.json") as f:
        return json.load(f)


# ---------------------------------------------------------------------------
# Scenario inputs: servers and clock
# ---------------------------------------------------------------------------

def scenario_file(scenario: str, name: str, scenarios_dir: str | Path = SCENARIOS_DIR,
                  run_dir: str | Path | None = None) -> Path | None:
    """``scenarios/<scenario>/<name>``, else ``<name>`` next to the run folders
    (``results/<scenario>/<name>``, where an unzipped run export puts it), else None."""
    for path in (Path(scenarios_dir) / scenario / name,
                 Path(run_dir).parent / name if run_dir is not None else None):
        if path is not None and path.is_file():
            return path
    return None


def load_servers(scenario: str, run_dir: str | Path | None = None,
                 scenarios_dir: str | Path = SCENARIOS_DIR) -> pd.DataFrame:
    """Columns s, name, zone, label, zone_label, and model when the scenario's servers.csv
    has one (scenarios built before release 1.4.0 have none); sorted by zone then s (display
    order).

    Names come from the scenario's ``servers.csv`` (see ``scenario_file``), else from the
    run's ``compliance_servers.csv``, else "server <s>".
    """
    path = scenario_file(scenario, "servers.csv", scenarios_dir, run_dir)
    if path is not None:
        df = pd.read_csv(path, usecols=lambda c: c in ("s", "server", "zone", "model")
                         ).rename(columns={"server": "name"})
    else:
        df = pd.DataFrame(columns=["s", "name", "zone"])
        if run_dir is not None:
            comp = Path(run_dir) / "baseline" / "compliance_servers.csv"
            if comp.is_file():
                df = pd.read_csv(comp, usecols=["s", "name", "zone"])
    df = df.astype({"s": int, "zone": int})
    df["name"] = df["name"].fillna(df["s"].map(lambda s: f"server {s}")).astype(str)
    df["zone_label"] = "Zone " + df["zone"].astype(str)
    df["label"] = df["name"] + " (Z" + df["zone"].astype(str) + ")"
    return df.sort_values(["zone", "s"]).reset_index(drop=True)


def _with_zones(servers: pd.DataFrame, zones: list[int]) -> pd.DataFrame:
    df = servers.copy()
    df["zone"] = df["s"].map(dict(zip(range(1, len(zones) + 1), zones))).astype(int)
    df["zone_label"] = "Zone " + df["zone"].astype(str)
    df["label"] = df["name"] + " (Z" + df["zone"].astype(str) + ")"
    return df.sort_values(["zone", "s"]).reset_index(drop=True)


def hours_to_clock(hour_local: pd.Series) -> pd.Series:
    """Decimal local hours -> Timestamps on a dummy date, rolling to the next day at midnight.

    ``hour_local`` is printed to 3 decimals (3.6 s), so it is snapped to the nearest 30 s.
    """
    seconds = (np.round(hour_local.to_numpy(float) * 120) * 30).astype(np.int64)
    day = np.concatenate([[0], np.cumsum(np.diff(seconds) < 0)])
    return pd.Series(CLOCK_ORIGIN + pd.to_timedelta(day * 86400 + seconds, unit="s"),
                     index=hour_local.index)


def load_clock(scenario: str, T: int, dt_seconds: float,
               scenarios_dir: str | Path = SCENARIOS_DIR,
               run_dir: str | Path | None = None) -> pd.DataFrame:
    """Columns t, hour_local, time, clock for t = 1..T.

    ``time`` is the local time at the middle of period t (``carbon.csv:hour_local``, found
    by ``scenario_file``); a period covers ``time +/- dt/2``. Without carbon.csv the clock
    starts at 00:00.
    """
    path = scenario_file(scenario, "carbon.csv", scenarios_dir, run_dir)
    t = pd.DataFrame({"t": np.arange(1, T + 1)})
    if path is not None:
        carbon = pd.read_csv(path, usecols=["t", "hour_local"])
        t = t.merge(carbon, on="t", how="left")
    else:
        t["hour_local"] = np.nan
    if t["hour_local"].isna().any():
        # Fill gaps (or a missing file) by extending the known periods at dt per period.
        known = t.dropna(subset=["hour_local"])
        h0 = (known["hour_local"].iloc[0] - (known["t"].iloc[0] - 1) * dt_seconds / 3600
              if len(known) else 0.5 * dt_seconds / 3600)
        t["hour_local"] = t["hour_local"].fillna(((h0 + (t["t"] - 1) * dt_seconds / 3600) % 24))
    t["time"] = hours_to_clock(t["hour_local"])
    t["clock"] = t["time"].dt.strftime("%H:%M")
    return t


def period_bounds(clock: pd.DataFrame, dt_seconds: float):
    """Function t -> Timestamp of the *start* of period t, extrapolated past 1..T.

    Used for schedule bars: a workload starting in period ``start`` with exclusive end
    ``finish`` spans ``bound(start)`` to ``bound(finish)``.
    """
    t0 = clock["time"].iloc[0] - pd.Timedelta(seconds=dt_seconds / 2)

    def bound(t):
        return t0 + pd.to_timedelta((np.asarray(t, dtype=float) - 1) * dt_seconds, unit="s")
    return bound


# ---------------------------------------------------------------------------
# One run
# ---------------------------------------------------------------------------

def _read_variants(run_dir: Path, filename: str, variants: tuple[str, ...]) -> pd.DataFrame:
    frames = []
    for v in variants:
        path = run_dir / v / filename
        if path.is_file():
            df = pd.read_csv(path)
            df.insert(0, "variant", v)
            frames.append(df)
    if not frames:
        return pd.DataFrame(columns=["variant"])
    return pd.concat(frames, ignore_index=True)


@dataclass
class Run:
    ref: RunRef
    summary: dict
    variants: tuple[str, ...]          # variants with a folder on disk, baseline first
    servers: pd.DataFrame
    clock: pd.DataFrame
    period: pd.DataFrame               # variant, t, energy, CI, carbon_g, time, clock
    comparison_period: pd.DataFrame    # t, CI, baseline_/optimized_/diff_ columns, time, clock
    server_period: pd.DataFrame        # variant, s, t, ..., name, zone, label, time, clock
    zone_period: pd.DataFrame          # variant, z, t, ..., zone_label, time, clock
    schedule: pd.DataFrame             # variant, i, ..., server_name, original_server_name
    compliance_servers: pd.DataFrame
    compliance_zones: pd.DataFrame
    compliance_workloads: pd.DataFrame
    compliance_cold_plates: pd.DataFrame = field(default_factory=pd.DataFrame)
    reassignment: pd.DataFrame = field(default_factory=pd.DataFrame)   # optimized only

    # -- summary shortcuts ------------------------------------------------------------
    @property
    def settings(self) -> dict:
        return self.summary.get("settings", {})

    @property
    def solver(self) -> dict:
        return self.summary.get("solver") or {}

    @property
    def provenance(self) -> dict:
        return provenance(self.summary)

    @property
    def has_optimized(self) -> bool:
        return bool(self.summary.get("has_optimized_schedule")) and "optimized" in self.variants

    @property
    def dt_seconds(self) -> float:
        return float(self.summary.get("scenario", {}).get("dt_seconds", 300.0))

    @property
    def thermal_cap_C(self) -> float:
        return float(self.summary.get("thermal_cap_C", 80.0))

    @property
    def T_dvfs_C(self) -> float:
        return float(self.summary.get("T_dvfs_C", 75.0))

    @property
    def T_max_C(self) -> float | None:
        """Server temperature limit T_max (headroom is measured from it)."""
        sp = self.server_period
        if sp.empty or "headroom_C" not in sp:
            return None
        return float((sp["temperature_C"] + sp["headroom_C"]).median())

    @property
    def H_min_C(self) -> float | None:
        t_max = self.T_max_C
        return None if t_max is None else t_max - self.thermal_cap_C

    def server_name(self, s) -> str:
        if s is None:
            return "–"
        m = self.servers.set_index("s")["name"]
        return str(m.get(int(s), f"server {s}"))


def load_run(run_dir: str | Path, scenarios_dir: str | Path = SCENARIOS_DIR) -> Run:
    run_dir = Path(run_dir)
    ref = RunRef(run_dir.parent.name, run_dir.name, run_dir)
    summary = load_summary(run_dir)
    sc = summary.get("scenario", {})
    scenario = sc.get("name", ref.scenario)
    variants = tuple(v for v in VARIANTS if (run_dir / v).is_dir())

    servers = load_servers(scenario, run_dir, scenarios_dir)
    zones = sc.get("zones")       # the run's own zones (a --zone-map run differs from servers.csv)
    if zones and len(zones) == len(servers):
        servers = _with_zones(servers, zones)
    dt = float(sc.get("dt_seconds", 300.0))
    T = int(sc.get("T") or _max_t(run_dir / "baseline" / "period.csv"))
    clock = load_clock(scenario, T, dt, scenarios_dir, run_dir)
    tcols = clock[["t", "time", "clock"]]

    def with_time(df: pd.DataFrame) -> pd.DataFrame:
        return df.merge(tcols, on="t", how="left") if "t" in df else df

    period = with_time(_read_variants(run_dir, "period.csv", variants))
    cmp_path = run_dir / "comparison_period.csv"
    comparison = with_time(pd.read_csv(cmp_path)) if cmp_path.is_file() else pd.DataFrame()

    server_cols = servers[["s", "name", "zone", "zone_label", "label"]]
    server_period = _read_variants(run_dir, "server_period.csv", variants)
    if not server_period.empty:
        server_period = with_time(server_period.merge(server_cols, on="s", how="left"))

    zone_period = _read_variants(run_dir, "zone_period.csv", variants)
    if not zone_period.empty:
        zone_period["zone_label"] = "Zone " + zone_period["z"].astype(str)
        if "airflow" in zone_period:
            # "not modelled" when airflow is off; keep a numeric column either way.
            zone_period["airflow"] = pd.to_numeric(zone_period["airflow"], errors="coerce")
        zone_period = with_time(zone_period)

    names = servers.set_index("s")["name"]
    schedule = _read_variants(run_dir, "schedule.csv", variants)
    if not schedule.empty:
        for col in ("server", "original_server"):
            schedule[f"{col}_name"] = schedule[col].map(names)
        schedule["type_label"] = schedule["type"].map(WORKLOAD_TYPES).fillna(schedule["type"])

    comp_servers = _read_variants(run_dir, "compliance_servers.csv", variants)
    if not comp_servers.empty and "name" not in comp_servers:
        comp_servers["name"] = comp_servers["s"].map(names)

    return Run(
        ref=ref, summary=summary, variants=variants, servers=servers, clock=clock,
        period=period, comparison_period=comparison, server_period=server_period,
        zone_period=zone_period, schedule=schedule, compliance_servers=comp_servers,
        compliance_zones=_read_variants(run_dir, "compliance_zones.csv", variants),
        compliance_workloads=_read_variants(run_dir, "compliance_workloads.csv", variants),
        compliance_cold_plates=_read_variants(run_dir, "compliance_cold_plates.csv", variants),
        reassignment=_read_reassignment(run_dir / "optimized" / "reassignment.csv", names),
    )


def _read_reassignment(path: Path, names: pd.Series) -> pd.DataFrame:
    """optimized/reassignment.csv with server names; empty for older runs without it."""
    if not path.is_file():
        return pd.DataFrame()
    df = pd.read_csv(path)
    for col in ("original_server", "final_server"):
        df[f"{col}_name"] = df[col].map(names)
    return df


def _max_t(path: Path) -> int:
    return int(pd.read_csv(path, usecols=["t"])["t"].max()) if path.is_file() else 0


# ---------------------------------------------------------------------------
# Derived views
# ---------------------------------------------------------------------------

def is_all_zero(series: pd.Series) -> bool:
    """True when a column has no non-zero number (all zero, blank or non-numeric)."""
    values = pd.to_numeric(series, errors="coerce").to_numpy(float)
    return bool(np.all(np.isnan(values) | (np.abs(values) <= ZERO_TOL)))


def nonzero_columns(df: pd.DataFrame, columns) -> list[str]:
    """The given columns that exist in ``df`` and are not all zero, in the given order."""
    return [c for c in columns if c in df and not is_all_zero(df[c])]


def active_energy_components(run: Run) -> list[str]:
    """Energy columns with any non-zero value in any variant (E_IT and E_crac always)."""
    always = ["E_IT_kWh", "E_crac_kWh"]
    extra = nonzero_columns(run.period, [c for c in ENERGY_COMPONENTS if c not in always])
    return always + extra


# (key, label, unit, better) for the key results tiles; better is "lower", "higher" or
# "neutral".
KPIS = [
    ("carbon_kg", "Total carbon", "kg CO₂", "lower"),
    ("E_total_kWh", "Total energy", "kWh", "lower"),
    ("E_IT_kWh", "IT energy", "kWh", "lower"),
    ("E_crac_kWh", "CRAC energy", "kWh", "lower"),
    ("E_cp_kWh", "Cold-plate pumps", "Wh", "lower"),
    ("hottest_T_C", "Hottest server", "°C", "lower"),
    ("min_headroom_C", "Lowest headroom", "°C", "higher"),
    ("n_late", "Missed deadlines", "", "lower"),
    ("n_reassigned", "Workloads moved", "", "neutral"),
]
# Stored in kWh but shown in Wh: the pumps use a fraction of a watt-hour a day.
KPI_SCALE = {"E_cp_kWh": 1000.0}


def _optimized_metrics(summary: dict | None) -> dict:
    if not summary or not summary.get("has_optimized_schedule"):
        return {}
    return (summary.get("optimized") or {}).get("metrics", {})


def kpi_frame(summary: dict, base: dict | None = None) -> pd.DataFrame:
    """One row per KPI of the optimized schedule: key, label, unit, better, value, base,
    diff, pct.

    ``base`` is the summary of the run to compare with (the Base of the test's section);
    ``base``/``diff``/``pct`` are NaN without it, ``value`` without an optimized schedule,
    and ``pct`` when the Base's value is 0. Cold-plate pump energy is dropped when it is zero
    in both (plates off).
    """
    mine, other = _optimized_metrics(summary), _optimized_metrics(base)
    rows = []
    for key, label, unit, better in KPIS:
        v, b = mine.get(key), other.get(key)
        if key == "E_cp_kWh" and not v and not b:
            continue
        scale = KPI_SCALE.get(key, 1.0)
        v = np.nan if v is None else float(v) * scale
        b = np.nan if b is None else float(b) * scale
        diff = v - b
        pct = diff / abs(b) * 100 if b and not np.isnan(diff) else np.nan
        rows.append(dict(key=key, label=label, unit=unit, better=better,
                         value=v, base=b, diff=diff, pct=pct))
    return pd.DataFrame(rows, columns=["key", "label", "unit", "better", "value", "base",
                                       "diff", "pct"])


def checks_frame(summary: dict) -> pd.DataFrame:
    """Self-checks: name, label, passed (True/False/None), detail."""
    rows = []
    for name, c in (summary.get("checks") or {}).items():
        c = c if isinstance(c, dict) else {"passed": c}
        detail = {k: v for k, v in c.items() if k != "passed"}
        rows.append(dict(name=name, label=name.replace("_", " ").capitalize(),
                         passed=c.get("passed"),
                         detail="; ".join(f"{k.replace('_', ' ')}: {_fmt_detail(v)}"
                                          for k, v in detail.items())))
    return pd.DataFrame(rows, columns=["name", "label", "passed", "detail"])


def _fmt_detail(v) -> str:
    if isinstance(v, float):
        return f"{v:,.6g}"
    if isinstance(v, list):
        return "none" if not v else "; ".join(map(str, v))
    return str(v)


def infeasibility_frame(summary: dict) -> pd.DataFrame:
    """family, label, result: one row per constraint family the diagnosis relaxed."""
    diag = summary.get("infeasibility") or {}
    return pd.DataFrame(
        [dict(family=k, label=RELAX_FAMILY_LABELS.get(k, k.replace("_", " ")), result=v)
         for k, v in diag.items()], columns=["family", "label", "result"])


def gantt_lanes(schedule: pd.DataFrame) -> pd.Series:
    """Lane index per row so workloads running at once on a server get separate rows.

    Greedy interval partitioning per (variant, server) on [start, finish); unplaced
    workloads (no server or start) get lane -1.
    """
    lanes = pd.Series(-1, index=schedule.index, dtype=int)
    ok = schedule["server"].notna() & schedule["start"].notna() & schedule["finish"].notna()
    for _, grp in schedule[ok].groupby(["variant", "server"]):
        ends: list[float] = []
        for idx, row in grp.sort_values(["start", "finish", "i"]).iterrows():
            for k, end in enumerate(ends):
                if end <= row["start"]:
                    ends[k] = row["finish"]
                    lanes[idx] = k
                    break
            else:
                ends.append(row["finish"])
                lanes[idx] = len(ends) - 1
    return lanes


def same_schedule(a: Run, b: Run) -> bool:
    """True when two runs' optimized schedules put every workload on the same server at the
    same start. Cooling and carbon can still differ (other zones, another carbon intensity)."""
    def key(run: Run) -> pd.DataFrame:
        sch = run.schedule
        sch = sch[sch["variant"] == "optimized"] if not sch.empty else sch
        return sch[["i", "server", "start"]].sort_values("i").reset_index(drop=True)
    if not (a.has_optimized and b.has_optimized):
        return False
    ka, kb = key(a), key(b)
    return len(ka) > 0 and ka.equals(kb)


def against_base(run: Run, label: str, base: Run | None = None) -> Run:
    """The optimized schedule of ``run`` as variant ``label`` and, when given, that of
    ``base`` as variant BASE_LABEL, in one Run for the test charts (the first variant drawn
    solid, the Base dashed). The Base's per-server rows take this run's server names and
    zones, so a test that regroups the zones lines up server by server. The limits (thermal
    cap, DVFS temperature, H_min) stay this run's."""
    pairs = [(run, label)]
    if base is not None and base.has_optimized:
        pairs.append((base, BASE_LABEL))
    server_cols = run.servers[["s", "name", "zone", "zone_label", "label"]]

    def stack(attr: str) -> pd.DataFrame:
        frames = []
        for r, name in pairs:
            df = getattr(r, attr)
            if df.empty or "variant" not in df:
                continue
            df = df[df["variant"] == "optimized"].assign(variant=name)
            if attr == "server_period" and r is not run:
                df = (df.drop(columns=[c for c in server_cols if c != "s" and c in df])
                      .merge(server_cols, on="s", how="left"))
            frames.append(df)
        return pd.concat(frames, ignore_index=True) if frames else getattr(run, attr).iloc[0:0]

    return dataclasses.replace(
        run, variants=tuple(name for _, name in pairs), period=stack("period"),
        server_period=stack("server_period"), zone_period=stack("zone_period"),
        schedule=stack("schedule"), compliance_zones=stack("compliance_zones"))


def throttled_servers(run: Run) -> pd.DataFrame:
    """Servers with DPF < 1 anywhere: variant, s, name, periods_throttled, min_DPF."""
    sp = run.server_period
    if sp.empty or "DPF" not in sp:
        return pd.DataFrame(columns=["variant", "s", "name", "periods_throttled", "min_DPF"])
    low = sp[sp["DPF"] < 1 - 1e-9]
    if low.empty:
        return pd.DataFrame(columns=["variant", "s", "name", "periods_throttled", "min_DPF"])
    return (low.groupby(["variant", "s", "name"], as_index=False)
            .agg(periods_throttled=("t", "size"), min_DPF=("DPF", "min")))


UNVERSIONED = "before 1.0.0"     # runs made before summary.json had a provenance block
UNKNOWN_BUILD = "unknown"


def provenance(summary: dict) -> dict:
    """summary.json's provenance block; empty for runs made before release 1.0.0."""
    p = summary.get("provenance")
    return p if isinstance(p, dict) else {}


def provenance_caption(summary: dict) -> str:
    """One line on what made a run: release, where, when, and the scenario build."""
    p = provenance(summary)
    if not p:
        return (f"Made {UNVERSIONED}: this run records no release or scenario build, so it "
                "cannot be checked against other runs.")
    where = {"command line": "from the command line", "dashboard": "on the dashboard"}.get(
        p.get("made_with"), f"in the {p.get('made_with')}")
    when = str(p.get("created_utc", "")).replace("T", " ").replace("Z", " UTC")
    return (f"Made with ThermaSched {p.get('thermasched_version')} {where}"
            + (f" on {when}" if when else "")
            + (f" (commit {p['git_commit']})" if p.get("git_commit") else "")
            + (f"; labelled {p['relabelled_from']} when solved, before the release's version "
               "was set" if p.get("relabelled_from") else "")
            + f" · scenario build `{p.get('scenario_fingerprint') or UNKNOWN_BUILD}`")


def provenance_warnings(summary: dict, scenarios_dir: str | Path = SCENARIOS_DIR) -> list[str]:
    """Reasons to be careful with a run: a newer summary layout than this dashboard reads,
    or a scenario build that differs from the one in ``scenarios_dir``."""
    p, out = provenance(summary), []
    if int(p.get("schema_version", 0)) > SCHEMA_VERSION:
        out.append(f"This run was made with ThermaSched {p.get('thermasched_version')}, newer "
                   f"than this dashboard ({VERSION}). Some figures may be missing until the "
                   "dashboard is updated.")
    name = (summary.get("scenario") or {}).get("name")
    run_fp = p.get("scenario_fingerprint")
    here = scenario_fingerprint(Path(scenarios_dir) / name) if name and run_fp else None
    if run_fp and here and run_fp != here:
        out.append(f"This run was solved on a different build of `{name}` (run: `{run_fp}`, "
                   f"this dashboard: `{here}`), for example one made with other settings or "
                   "other data. Compare it only with runs of the same build.")
    return out


def per_tick_info(summary: dict) -> dict | None:
    """The per-tick block of a run re-planned at every tick (n_ticks, n_ticks_time_limit,
    tick_time_limit_s, max_mip_gap, ...); None for a run planned for the whole day at once."""
    return (summary.get("per_tick") or None) if summary.get("mode") == "per_tick" else None


def ticks_at_cap(ticks: dict) -> int:
    return int(ticks.get("n_ticks_time_limit") or 0)


def ticks_no_bound(ticks: dict) -> int:
    """Ticks at the cap stopped before the solver had a bound: their gap is unknown."""
    return int(ticks.get("n_ticks_no_bound") or 0)


def load_ticks(run_dir: str | Path) -> pd.DataFrame | None:
    """ticks.csv of a per-tick run (one row per tick: status, solve_s, mip_gap, n_waiting,
    ...); None when the run has none. A tick at the cap with no gap stopped before the solver
    had a bound: its gap stays NaN (unknown)."""
    path = Path(run_dir) / "ticks.csv"
    if not path.is_file():
        return None
    df = pd.read_csv(path)
    gap = pd.to_numeric(df["mip_gap"], errors="coerce")
    df["mip_gap"] = gap.where(gap.notna() | (df["status"] == STATUS_TIME_LIMIT), 0.0)
    return df


# The limit a per-tick run names when a tick has no schedule (scheduler/pertick.py), in a few
# words. A stuck run stops at that tick: it keeps the ticks before it, not a whole day.
TICK_TIME_LIMIT = "tick_time_limit"
STUCK_LIMIT_WORDS = {
    "deadlines_and_waiting": "deadlines", "thermal_cap": "thermal cap",
    "capacity": "server capacity", "cooling_capacity": "cooling capacity",
    "priority": "priority order", "supply_ramp": "supply-air ramps",
    "several_limits_together": "several limits together", "model_input": "model inputs",
}


def stuck_info(summary: dict) -> dict | None:
    """Where and why a per-tick run stopped (tick, limits, limit_text, detail, n_waiting,
    n_running, relaxed_one_at_a_time); None for a run that did not get stuck."""
    ticks = per_tick_info(summary)
    if ticks is None or ticks.get("outcome") != "stuck":
        return None
    return ticks.get("stuck") or None


def stuck_reason(stuck: dict, cap_s: float | None = None) -> str:
    """Why a run stopped at its stuck tick, in a few words: 'no schedule within the 60 s cap'
    (the solver ran out of time; a schedule may exist) or 'no valid schedule (deadlines vs.
    server capacity)' (none exists, and relaxing any one of these alone would give one)."""
    limits = list(stuck.get("limits") or [])
    if limits == [TICK_TIME_LIMIT]:
        return f"no schedule within the {cap_s:g} s cap" if cap_s else \
            "no schedule within the per-tick cap"
    return "no valid schedule (" + " vs. ".join(
        STUCK_LIMIT_WORDS.get(x, x.replace("_", " ")) for x in limits) + ")"


def stuck_sentence(summary: dict, clock: str | None = None) -> str:
    """Where and why a stuck run stopped, and what it keeps, in a few sentences; ``clock``
    is the local time of the stuck tick's period."""
    stuck, ticks = stuck_info(summary) or {}, per_tick_info(summary) or {}
    tick, T = stuck.get("tick"), (summary.get("scenario") or {}).get("T")
    cap_s, cap = ticks.get("tick_time_limit_s"), ticks_at_cap(ticks)
    solved = int(ticks.get("n_ticks_optimal") or 0) + cap
    gap = ticks.get("max_mip_gap")
    if list(stuck.get("limits") or []) == [TICK_TIME_LIMIT]:
        why = (f"the solver found no schedule for it within the {cap_s:g} s per-tick cap, so "
               "whether one exists is not known")
    else:
        why = ("no valid schedule exists for it: " + (stuck.get("detail") or
                                                       stuck.get("limit_text") or "unknown"))
    waiting = (f" {stuck['n_waiting']} workloads were waiting and {stuck['n_running']} "
               "running at that tick." if stuck.get("n_waiting") is not None else "")
    before = (f"The {solved} ticks before it were solved"
              + (f" ({cap} at the cap, worst gap {gap * 100:.2f} %)" if cap and gap is not None
                 else f" ({cap} at the cap)" if cap else "")
              + " and are kept as a partial schedule")
    return (f"Stuck at tick {tick}" + (f" of {T}" if T else "")
            + (f" ({clock})" if clock else "") + f": {why}.{waiting} "
            f"{before}. The run stops there, so it has no full-day schedule and no full-day "
            "results.")


def solver_short(solver: dict, ticks: dict | None = None) -> str:
    """The solver's outcome in a few words: 'Proven best', or 'Stopped at time limit, gap
    1.60 %' (the gap tells how far from the best possible the schedule may be). A per-tick
    run (``ticks``, from per_tick_info): 'Every tick solved', or '48 of 288 ticks at the
    cap, worst gap 24.62 %'; a stuck one 'Stuck at tick 55: no schedule within the 60 s
    cap'."""
    status, gap = solver.get("status"), solver.get("mip_gap")
    if ticks is not None and ticks.get("stuck"):
        stuck = ticks["stuck"]
        return (f"Stuck at tick {stuck.get('tick')}: "
                + (stuck.get("reason") or stuck_reason(stuck, ticks.get("tick_time_limit_s"))))
    if ticks is not None and status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT):
        n, cap = ticks.get("n_ticks"), ticks_at_cap(ticks)
        if not cap:
            return "Every tick solved"
        nb = ticks_no_bound(ticks)
        return f"{cap} of {n} ticks at the cap" + (
            f", worst gap {gap * 100:.2f} %" if gap is not None else "") + (
            f", {nb} with no bound" if nb else "")
    text = status_text(status)
    if status == STATUS_TIME_LIMIT and gap is not None:
        text += f", gap {gap * 100:.2f} %"
    return text


def solver_sentence(solver: dict, ticks: dict | None = None) -> str:
    """How complete the result is, in one sentence for the thesis."""
    status, gap, t = solver.get("status"), solver.get("mip_gap"), solver.get("solve_time_s")
    took = f" in {t:,.0f} s" if t is not None else ""
    if ticks is not None and ticks.get("stuck"):    # stuck_sentence says more, from the summary
        return f"{solver_short(solver, ticks)}. The run has no full-day schedule."
    if ticks is not None and status in (STATUS_OPTIMAL, STATUS_TIME_LIMIT):
        n, cap, limit = ticks.get("n_ticks"), ticks_at_cap(ticks), ticks.get("tick_time_limit_s")
        slow = ticks.get("slowest_tick_solve_s")
        head = (f"Re-planned at each of {n} ticks{took} in all"
                + (f" (slowest tick {slow:,.0f} s, tick {ticks.get('slowest_tick')})"
                   if slow is not None else "") + ". ")
        if not cap:
            body = "Every tick was solved to the requested gap. "
        else:
            nb = ticks_no_bound(ticks)
            body = (f"{n - cap} ticks were solved to the requested gap and {cap} stopped at the "
                    f"{limit:g} s per-tick cap; the worst of these is within "
                    f"{(gap or 0) * 100:.2f} % of the best plan for what that tick could see. ")
            if nb == cap:
                body = (f"{n - cap} ticks were solved to the requested gap and {cap} stopped at "
                        f"the {limit:g} s per-tick cap before the solver had a bound, so how "
                        "close their plans are to the best is unknown. ")
            elif nb:
                body = body[:-2] + (f", except {nb} that stopped before the solver had a bound, "
                                    "whose gap is unknown. ")
        return body.join([head, "Each tick sees only the workloads that have arrived, so the "
                          "day as a whole is not proven best."])
    if status == STATUS_OPTIMAL:
        return f"Proven best{took}: no valid schedule emits less carbon (within the requested gap)."
    if status == STATUS_TIME_LIMIT and gap is not None:
        return (f"Stopped at the time limit{took}. This is the best schedule found, within "
                f"{gap * 100:.2f} % of the best possible (the solver's lower bound); it is not "
                "proven best.")
    return f"{status_text(status)}{took}."


def compare_runs(runs: list[RunRef]) -> pd.DataFrame:
    """One row per run (summary.json only): settings, solver and key metrics."""
    rows = []
    for ref in runs:
        try:
            s = load_summary(ref.path)
        except (OSError, json.JSONDecodeError) as e:
            rows.append(dict(scenario=ref.scenario, run=ref.name, status=f"unreadable: {e}"))
            continue
        st, sol = s.get("settings", {}), s.get("solver") or {}
        base = (s.get("baseline") or {}).get("metrics", {})
        has_opt = bool(s.get("has_optimized_schedule"))
        opt = (s.get("optimized") or {}).get("metrics", {}) if has_opt else {}
        supply = (f"fixed {_supply_fixed(st):g} °C" if st.get("supply_temp_mode") == "fixed"
                  else "optimized")
        b_c, o_c = base.get("carbon_kg"), opt.get("carbon_kg")
        b_e, o_e = base.get("E_total_kWh"), opt.get("E_total_kWh")
        pv = provenance(s)
        info = run_info(ref, s)
        rows.append(dict(
            scenario=ref.scenario, run=ref.name,
            configuration=info.config.label if info.config else None,
            test=run_title(info, ref) if info.tests else None,
            scenario_build=pv.get("scenario_fingerprint") or UNKNOWN_BUILD,
            version=pv.get("thermasched_version") or UNVERSIONED,
            made_with=pv.get("made_with"),
            baseline_rule=st.get("baseline_rule"),
            baseline_supply_C=st.get("baseline_supply_temp_C"),
            reassignment=st.get("reassignment"), supply_temp=supply,
            H_min_C=st.get("H_min_C"), cold_plates=bool(st.get("cold_plates")),
            cp_mode=st.get("cp_mode", "optimize") if st.get("cold_plates") else None,
            priority=st.get("priority", False),
            zone_map=",".join(map(str, st["zone_map"])) if st.get("zone_map") else None,
            carbon=_carbon_label(st),
            status=sol.get("status"), solve_time_s=sol.get("solve_time_s"),
            mip_gap=sol.get("mip_gap"),
            baseline_valid=(s.get("baseline") or {}).get("valid"),
            optimized_valid=(s.get("optimized") or {}).get("valid") if has_opt else None,
            baseline_carbon_kg=b_c, optimized_carbon_kg=o_c,
            carbon_saved_kg=(b_c - o_c) if has_opt and b_c is not None else None,
            carbon_saved_pct=(b_c - o_c) / b_c * 100 if has_opt and b_c else None,
            baseline_E_total_kWh=b_e, optimized_E_total_kWh=o_e,
            energy_saved_pct=(b_e - o_e) / b_e * 100 if has_opt and b_e else None,
            baseline_hottest_C=base.get("hottest_T_C"), optimized_hottest_C=opt.get("hottest_T_C"),
            n_reassigned=opt.get("n_reassigned"), n_cross_zone=opt.get("n_cross_zone"),
            optimized_n_noncompliant=opt.get("n_noncompliant"),
        ))
    return pd.DataFrame(rows)


# ---------------------------------------------------------------------------
# Chapter 6 of the thesis: configurations, tests and the results tables
# ---------------------------------------------------------------------------

# The configurations, tests and sections live in scheduler/experiments.py, shared with the
# solver and the batch notebook. A run is matched to a test by its settings.
CONFIGURATIONS = ex.CONFIGURATIONS
SECTIONS = ex.SECTIONS
OTHER = "other"                 # runs of no configuration (another scenario, an archive)
NOT_RUN_YET = "not run yet"
NOT_APPLICABLE = "not applicable"
# A configuration with no run at all: left out of this release, not waiting for one test.
NOT_IN_RELEASE = "not run in this release"
NO_RUN = (NOT_RUN_YET, NOT_APPLICABLE, NOT_IN_RELEASE)


@dataclass(frozen=True)
class RunInfo:
    """What a run is: its configuration (None for another scenario), the tests it matches
    and when it was made (for picking the newest of several matches)."""
    config: ex.Configuration | None
    tests: tuple[ex.Test, ...]
    created: str

    @property
    def config_key(self) -> str:
        return self.config.key if self.config else OTHER


def run_info(ref: RunRef, summary: dict | None = None) -> RunInfo:
    try:
        summary = summary if summary is not None else load_summary(ref.path)
    except (OSError, json.JSONDecodeError):
        return RunInfo(None, (), "")
    created = str(provenance(summary).get("created_utc") or "")
    if is_archive(ref.scenario):
        return RunInfo(None, (), created)
    scenario = (summary.get("scenario") or {}).get("name") or ref.scenario
    config, tests = ex.identify(summary.get("settings") or {}, scenario)
    return RunInfo(config, tuple(tests), created)


def run_infos(runs: list[RunRef]) -> dict[str, RunInfo]:
    return {ref.key: run_info(ref) for ref in runs}


def run_title(info: RunInfo, ref: RunRef) -> str:
    """'6.1 · Test 1 · …' for a test run, else its folder name."""
    return " / ".join(ex.title_of(t) for t in info.tests) or ref.name


def run_long_title(info: RunInfo, ref: RunRef) -> str:
    """'10 servers · 6.1 · Test 1 · …', for page headers."""
    title = run_title(info, ref) if info.tests else ref.name
    return f"{info.config.label} · {title}" if info.config else title


def catalogue(runs: list[RunRef], infos: dict[str, RunInfo] | None = None
              ) -> dict[tuple[str, str], tuple[RunRef | None, list[RunRef]]]:
    """(configuration key, test id) -> (the run used: the newest match, or None; every
    match, oldest first), for every configuration and test."""
    infos = infos if infos is not None else run_infos(runs)
    found: dict[tuple[str, str], list[tuple[str, RunRef]]] = {}
    for ref in runs:
        info = infos.get(ref.key)
        if info is None or info.config is None:
            continue
        for test in info.tests:
            found.setdefault((info.config.key, test.id), []).append((info.created, ref))
    out = {}
    for config in CONFIGURATIONS:
        for test in ex.TESTS:
            matches = [r for _, r in sorted(found.get((config.key, test.id), []),
                                            key=lambda m: m[0])]
            out[(config.key, test.id)] = (matches[-1] if matches else None, matches)
    return out


def configs_with_runs(cat) -> set[str]:
    """Keys of the configurations with at least one test run, from ``catalogue``."""
    return {key for (key, _), (used, _) in cat.items() if used is not None}


def cell_status(config: ex.Configuration, test: ex.Test, used: RunRef | None,
                in_release: bool = True) -> str | None:
    """None when the test has a run, else NOT_APPLICABLE, NOT_IN_RELEASE (the configuration
    has no run at all) or NOT_RUN_YET."""
    if not ex.applicable(test, config):
        return NOT_APPLICABLE
    if used is not None:
        return None
    return NOT_RUN_YET if in_release else NOT_IN_RELEASE


def config_runs(runs: list[RunRef], infos: dict[str, RunInfo] | None = None
                ) -> dict[str, list[RunRef]]:
    """Configuration key (or OTHER) -> its runs for the run picker: the run used for each
    test, in test order, then the configuration's other runs (validation runs, older solves
    of a test) by folder name. Configurations without runs are left out."""
    infos = infos if infos is not None else run_infos(runs)
    cat = catalogue(runs, infos)
    out: dict[str, list[RunRef]] = {}
    for config in CONFIGURATIONS:
        used = [cat[(config.key, t.id)][0] for t in ex.TESTS]
        used = list(dict.fromkeys(r for r in used if r is not None))
        rest = sorted((r for r in runs if infos[r.key].config_key == config.key
                       and r not in used), key=lambda r: r.name)
        if used or rest:
            out[config.key] = used + rest
    other = [r for r in runs if infos[r.key].config is None]
    if other:
        out[OTHER] = other
    return out


# Columns of the Chapter 6 results table: key -> header (with its unit).
CHAPTER6_COLUMNS = {
    "configuration": "Configuration",
    "servers": "Servers",
    "level": "Load level",
    "section": "Section",
    "tag": "Test",
    "label": "Run",
    "run": "Run folder",
    "release": "Release",
    "wait_mean_min": "Average waiting time (min)",
    "wait_max_min": "Longest waiting time (min)",
    "exec_mean_min": "Average execution time (min)",
    "turnaround_mean_min": "Average completion time, from arrival (min)",
    "turnaround_max_min": "Longest completion time, from arrival (min)",
    "util_mean_pct": "Average server utilization (%)",
    "E_crac_zone1_kWh": "CRAC electricity, Zone 1 (kWh)",
    "E_crac_zone2_kWh": "CRAC electricity, Zone 2 (kWh)",
    "E_crac_kWh": "CRAC electricity (kWh)",
    "E_IT_kWh": "IT electricity (kWh)",
    "E_cp_Wh": "Cold-plate pump electricity (Wh)",
    "E_cooling_kWh": "Cooling electricity, CRAC + cold plates (kWh)",
    "E_total_kWh": "Total electricity (kWh)",
    "carbon_kg": "Total carbon emissions (kg CO2)",
    "T_mean_C": "Average server temperature (°C)",
    "T_max_C": "Highest server temperature (°C)",
    "H_mean_C": "Average thermal headroom (°C)",
    "n_moved_server": "Workloads moved to another server",
    "n_moved_zone": "Workloads moved to another zone",
    "n_late": "Missed deadlines",
    "pct_late": "Missed deadlines (%)",
    "n_throttled_servers": "Servers slowed by temperature",
    "DPF_min": "Lowest DPF",
    "n_unplaced": "Unplaced workloads",
    "status": "Solver status",
    "mip_gap": "MIP gap (per tick: the worst tick's)",
    "solve_time_s": "Solve time (s; per tick: all ticks)",
    "time_limit_s": "Time limit (s)",
    "n_ticks": "Ticks re-planned",
    "n_ticks_at_cap": "Ticks at the per-tick cap",
    "tick_time_limit_s": "Per-tick cap (s)",
    "stuck_tick": "Stuck at tick",
    "stuck_reason": "Why it got stuck",
}
# Columns that hold results, for the change against each section's reference.
CHAPTER6_METRICS = tuple(k for k in CHAPTER6_COLUMNS if k not in {
    "configuration", "servers", "level", "section", "tag", "label", "run", "release",
    "n_unplaced", "status", "mip_gap", "solve_time_s", "time_limit_s", "n_ticks",
    "n_ticks_at_cap", "tick_time_limit_s", "stuck_tick", "stuck_reason"})
# What each section measures (the Chapter 6 document's "What to measure"), in its order.
SECTION_MEASURES = {
    "6.1": ("n_moved_server", "n_moved_zone", "util_mean_pct", "E_crac_zone1_kWh",
            "E_crac_zone2_kWh", "E_total_kWh", "carbon_kg", "n_late", "pct_late", "T_mean_C",
            "T_max_C"),
    "6.2": ("T_mean_C", "T_max_C", "H_mean_C", "E_crac_kWh", "E_cp_Wh", "E_cooling_kWh",
            "carbon_kg"),
    "6.3": ("exec_mean_min", "wait_mean_min", "turnaround_mean_min", "n_late", "pct_late",
            "T_mean_C", "T_max_C", "DPF_min", "n_throttled_servers", "E_total_kWh",
            "carbon_kg"),
    "6.4": ("n_moved_server", "E_IT_kWh", "E_crac_kWh", "E_cp_Wh", "carbon_kg",
            "turnaround_mean_min", "wait_mean_min", "n_late", "T_mean_C", "T_max_C",
            "DPF_min"),
}


def schedule_times(schedule_csv: str | Path, dt_seconds: float) -> dict:
    """Waiting time (start - arrival), execution time (end - start) and time from arrival to
    completion (end - arrival), the end being exclusive, in minutes, over the placed workloads
    of one schedule.csv."""
    df = pd.read_csv(schedule_csv)
    placed = df[df["placed"] == 1] if "placed" in df else df.dropna(subset=["start"])
    per_min = dt_seconds / 60.0
    wait = placed["wait"].astype(float) * per_min
    run = (placed["finish"].astype(float) - placed["start"].astype(float)) * per_min
    turn = (placed["finish"].astype(float) - placed["arrival"].astype(float)) * per_min
    nan = float("nan")
    return dict(
        wait_mean_min=wait.mean() if len(placed) else nan,
        wait_max_min=wait.max() if len(placed) else nan,
        exec_mean_min=run.mean() if len(placed) else nan,
        turnaround_mean_min=turn.mean() if len(placed) else nan,
        turnaround_max_min=turn.max() if len(placed) else nan,
        n_unplaced=int(len(df) - len(placed)))


def chapter6_row(ref: RunRef, summary: dict) -> dict:
    """The Chapter 6 figures of one run, from its optimized result."""
    pv, sol, st = provenance(summary), summary.get("solver") or {}, summary.get("settings") or {}
    row = dict(run=ref.name, release=pv.get("thermasched_version") or UNVERSIONED,
               status=sol.get("status"), mip_gap=sol.get("mip_gap"),
               solve_time_s=sol.get("solve_time_s"), time_limit_s=st.get("time_limit_s"))
    ticks = per_tick_info(summary)
    if ticks is not None:       # the whole-day time limit plays no part in a per-tick run
        row.update(time_limit_s=None, n_ticks=ticks.get("n_ticks"),
                   n_ticks_at_cap=ticks_at_cap(ticks),
                   tick_time_limit_s=ticks.get("tick_time_limit_s"))
        stuck = stuck_info(summary)
        if stuck is not None:
            row.update(stuck_tick=stuck.get("tick"),
                       stuck_reason=stuck_reason(stuck, ticks.get("tick_time_limit_s")))
    if not summary.get("has_optimized_schedule"):
        return row
    opt = summary.get("optimized") or {}
    tot, met = opt.get("totals") or {}, opt.get("metrics") or {}
    zones = list(tot.get("E_crac_by_zone_kWh") or [])
    n = tot.get("n_workloads") or (summary.get("scenario") or {}).get("I")
    e_cp, e_crac = tot.get("E_cp_kWh"), tot.get("E_crac_kWh")
    row.update(
        E_crac_zone1_kWh=zones[0] if len(zones) > 0 else None,
        E_crac_zone2_kWh=zones[1] if len(zones) > 1 else None,
        E_crac_kWh=e_crac,
        E_IT_kWh=tot.get("E_IT_kWh"),
        E_cp_Wh=None if e_cp is None else e_cp * 1000.0,
        E_cooling_kWh=None if e_crac is None else e_crac + (e_cp or 0.0),
        E_total_kWh=tot.get("E_total_kWh"),
        carbon_kg=tot.get("carbon_kg"),
        T_mean_C=met.get("mean_T_C"), T_max_C=met.get("hottest_T_C"),
        n_moved_server=met.get("n_reassigned"), n_moved_zone=met.get("n_cross_zone"),
        n_late=met.get("n_late"),
        pct_late=(met.get("n_late") / n * 100.0) if n and met.get("n_late") is not None else None,
        n_throttled_servers=met.get("n_throttled_servers"),
    )
    dt = (summary.get("scenario") or {}).get("dt_seconds", 300.0)
    schedule = Path(ref.path) / "optimized" / "schedule.csv"
    if schedule.is_file():
        row.update(schedule_times(schedule, float(dt)))
    sp_path = Path(ref.path) / "optimized" / "server_period.csv"
    if sp_path.is_file():
        sp = pd.read_csv(sp_path, usecols=["utilization", "headroom_C", "DPF"])
        row.update(util_mean_pct=float(sp["utilization"].mean()) * 100.0,
                   H_mean_C=float(sp["headroom_C"].mean()), DPF_min=float(sp["DPF"].min()))
    return row


# Per-server figures of one schedule, for the Chapter 6 section detail: column -> header.
SERVER_STATS = {
    "util_mean_pct": "Average CPU utilization (%)",
    "util_max_pct": "Peak CPU utilization (%)",
    "T_mean_C": "Average temperature (°C)",
    "T_max_C": "Highest temperature (°C)",
    "H_mean_C": "Average thermal headroom (°C)",
    "H_min_C": "Lowest thermal headroom (°C)",
    "DPF_mean": "Average DPF",
    "DPF_min": "Lowest DPF",
    "periods_throttled": "Periods slowed (DPF < 1)",
    "E_IT_kWh": "IT electricity (kWh)",
}
# The per-server figures each section shows first.
SECTION_SERVER_STATS = {
    "6.1": "util_mean_pct",
    "6.2": "T_max_C",
    "6.3": "DPF_min",
    "6.4": "DPF_min",
}


def server_columns(servers: pd.DataFrame) -> list[str]:
    """s, name, zone, and model when the scenario's servers.csv names it."""
    return ["s", "name", "zone", *(["model"] if "model" in servers else [])]


def server_stats(run: Run, variant: str = "optimized") -> pd.DataFrame:
    """One row per server (display order): s, name, zone, model (when known) and the
    SERVER_STATS columns."""
    sp = run.server_period
    sp = sp[sp["variant"] == variant] if not sp.empty else sp
    if sp.empty:
        return pd.DataFrame(columns=["s", "name", "zone", *SERVER_STATS])
    g = sp.groupby("s")
    out = pd.DataFrame({
        "util_mean_pct": g["utilization"].mean() * 100,
        "util_max_pct": g["utilization"].max() * 100,
        "T_mean_C": g["temperature_C"].mean(),
        "T_max_C": g["temperature_C"].max(),
        "H_mean_C": g["headroom_C"].mean(),
        "H_min_C": g["headroom_C"].min(),
        "DPF_mean": g["DPF"].mean(),
        "DPF_min": g["DPF"].min(),
        "periods_throttled": g["DPF"].apply(lambda d: int((d < 1 - 1e-9).sum())),
        "E_IT_kWh": g["P_IT_W"].sum() * run.dt_seconds / 3600 / 1000,
    }).reset_index()
    return run.servers[server_columns(run.servers)].merge(out, on="s", how="left")


def reassignment_table(run: Run) -> pd.DataFrame:
    """Every workload's original and final server and zone (optimized schedule only)."""
    ra = run.reassignment
    if ra.empty:
        return pd.DataFrame()
    ra = ra.sort_values("i")
    yes_no = {1: "yes", 0: "no"}
    return pd.DataFrame({
        "Workload": ra["workload_id"],
        "Type": ra["type"].map(WORKLOAD_TYPES).fillna(ra["type"]),
        "Running at start (B_i)": ra["B_running"].map(yes_no),
        "Original server": ra["original_server_name"],
        "Original zone": ra["original_zone"],
        "Moved": ra["reassigned"].map(yes_no),
        "Final server": ra["final_server_name"],
        "Final zone": ra["final_zone"],
        "Cross-zone": ra["cross_zone"].map(yes_no),
        "Reason": ra["reason"],
    }).reset_index(drop=True)


def chapter6_table(runs: list[RunRef], infos: dict[str, RunInfo] | None = None,
                   configs: list[str] | None = None) -> pd.DataFrame:
    """One row per configuration, section and test, in that order, with the columns of
    CHAPTER6_COLUMNS. The base run appears in every section, so every test sits right
    below its section's reference. A test without a run says why in *status*."""
    infos = infos if infos is not None else run_infos(runs)
    cat = catalogue(runs, infos)
    in_release = configs_with_runs(cat)
    rows, cache = [], {}
    for config in CONFIGURATIONS:
        if configs is not None and config.key not in configs:
            continue
        for section in SECTIONS:
            for tag, tid in section.tests:
                test = ex.TEST_BY_ID[tid]
                used, _ = cat[(config.key, tid)]
                row = dict(configuration=config.label, servers=config.size_label,
                           level=config.level_label, section=section.key, tag=tag,
                           label=test.name)
                missing = cell_status(config, test, used, config.key in in_release)
                if missing:
                    row.update(run=None, status=missing)
                else:
                    if used.key not in cache:
                        cache[used.key] = chapter6_row(used, load_summary(used.path))
                    row.update(cache[used.key])
                rows.append(row)
    return pd.DataFrame(rows, columns=list(CHAPTER6_COLUMNS))


def has_result(table: pd.DataFrame) -> pd.Series:
    """Rows with a run (a stuck run too, though its measures are blank)."""
    return ~table["status"].isin(NO_RUN)


def is_stuck(table: pd.DataFrame) -> pd.Series:
    """Rows whose run stopped at a stuck tick."""
    return pd.to_numeric(table["stuck_tick"], errors="coerce").notna()


def _clean(x: float, tol: float = 1e-9) -> float:
    """Round float noise (e.g. -7e-16) to an exact 0 so it does not print as -0.00."""
    return 0.0 if abs(x) < tol else x


def chapter6_changes(table: pd.DataFrame) -> pd.DataFrame:
    """Each result against the first row of its configuration and section (the section's
    reference), in %. Zero against zero is 0 %; anything against zero is left blank."""
    out = table.copy()
    groups = table.groupby(["configuration", "section"], sort=False).groups
    for _, idx in groups.items():
        ref = table.loc[idx[0], list(CHAPTER6_METRICS)]
        for i in idx:
            for col in CHAPTER6_METRICS:
                v, r = table.at[i, col], ref[col]
                if pd.isna(v) or pd.isna(r):
                    out.at[i, col] = None
                elif r == 0:
                    out.at[i, col] = 0.0 if v == 0 else None
                else:
                    out.at[i, col] = _clean((float(v) - float(r)) / abs(float(r)) * 100.0)
    return out


def _supply_fixed(settings: dict) -> float:
    v = settings.get("fixed_supply_temp_C")
    return float(settings.get("baseline_supply_temp_C", 18.0) if v is None else v)


def _carbon_label(settings: dict) -> str:
    if settings.get("carbon_file"):
        return Path(settings["carbon_file"]).name
    return "flat (mean)" if settings.get("ci_flat") else "scenario"


def settings_badges(summary: dict) -> list[tuple[str, str]]:
    """(label, value) pairs that describe a run's switches, in display order."""
    st = summary.get("settings", {})
    supply = (f"fixed at {_supply_fixed(st):g} °C" if st.get("supply_temp_mode") == "fixed"
              else "optimized")
    cp = "off"
    if st.get("cold_plates"):
        n = st.get("cold_plate_count")
        cp = "on (servers.csv)" if n is None else f"{n} per server"
        if st.get("cp_mode") == "fixed":
            flow = st.get("fixed_cp_flow_Lmin") or st.get("baseline_cp_flow_Lmin")
            cool = st.get("fixed_cp_coolant_C") or st.get("baseline_cp_coolant_C")
            cp += f", fixed at {flow:g} L/min, {cool:g} °C"
    h_min = st.get("H_min_C")
    carbon = _carbon_label(st)
    if carbon == "scenario":
        carbon = f"scenario {(summary.get('scenario') or {}).get('name', '?')}"
    return [
        ("Reassignment", str(st.get("reassignment", "?")).replace("_", " ")),
        ("Server choice", "temperature ignored (two passes)"
         if st.get("placement") == "thermal_blind" else "temperature-aware"),
        ("Supply temperature", supply),
        ("H_min", "scenario default" if h_min is None else f"{h_min:g} °C"),
        ("Cold plates", cp),
        ("Zones", "servers.csv" if not st.get("zone_map")
         else ",".join(map(str, st["zone_map"]))),
        ("Carbon intensity", carbon),
        *([("Planning", "re-planned every period"),
           ("Per-tick cap", f"{st.get('tick_time_limit_s') or 0:g} s")]
          if st.get("per_tick") else [("Time limit", f"{st.get('time_limit_s', 0):g} s")]),
        ("MIP gap", f"{st.get('mip_rel_gap', 0):g}"),
    ]


# ---------------------------------------------------------------------------
# Validation tables
# ---------------------------------------------------------------------------

def validation_scenarios(root: str | Path = VALIDATION_DIR) -> list[str]:
    """Scenarios with validation tables, ``<root>/<scenario>/validation.json``."""
    root = Path(root)
    return sorted(p.parent.name for p in root.glob("*/validation.json")) if root.is_dir() else []


def load_validation(scenario: str, root: str | Path = VALIDATION_DIR
                    ) -> tuple[dict, dict[str, pd.DataFrame]] | None:
    """validation.json and every table of one scenario's validation, or None if there is none."""
    folder = Path(root) / scenario
    meta_path = folder / "validation.json"
    if not meta_path.is_file():
        return None
    meta = json.loads(meta_path.read_text())
    tables = {}
    for p in sorted(folder.glob("*.csv")):
        try:
            tables[p.stem] = pd.read_csv(p)
        except pd.errors.EmptyDataError:
            tables[p.stem] = pd.DataFrame()
    return meta, tables


In [ ]:
%%writefile {LIB}/dashboard/theme.py
"""Colours and shared Plotly styling.

Categorical colours come from a validated colour-blind-safe palette with a light and a
dark step per hue. Baseline and optimized keep the same hue everywhere (orange and blue).
Other dimensions (energy component, workload type) use hues distinct from those two.
Charts with one line per server or zone use ``series``; there a schedule is told apart by
the dash (baseline dashed), not the hue.
Thresholds are drawn in neutral ink with a text label, never in a series colour.
"""

from __future__ import annotations

from dataclasses import dataclass, field

import plotly.graph_objects as go


@dataclass(frozen=True)
class Palette:
    mode: str
    baseline: str
    optimized: str
    components: dict[str, str]
    types: dict[str, str]
    ink: str
    ink2: str
    muted: str
    grid: str
    axis: str
    surface: str
    seq: tuple[str, ...]           # sequential ramp, low -> high (low recedes into surface)
    series: tuple[str, ...] = ()   # one hue per server or zone, in display order
    status: dict[str, str] = field(default_factory=lambda: dict(STATUS))

    def variant(self, v: str) -> str:
        return self.baseline if v == "baseline" else self.optimized

    def nth(self, k: int) -> str:
        """The k-th series colour (servers, zones), cycling past the end."""
        return self.series[k % len(self.series)]


STATUS = {"good": "#0ca30c", "warning": "#fab219", "serious": "#ec835a", "critical": "#d03b3b"}

_BLUE_RAMP = ("#cde2fb", "#9ec5f4", "#6da7ec", "#3987e5", "#256abf", "#184f95", "#0d366b")

LIGHT = Palette(
    mode="light", baseline="#eb6834", optimized="#2a78d6",
    components={"E_IT_kWh": "#1baf7a", "E_crac_kWh": "#6250d6",
                "E_cp_kWh": "#e87ba4", "E_fan_kWh": "#eda100"},
    types={"time_sensitive": "#e87ba4", "delay_tolerant": "#008300"},
    ink="#0b0b0b", ink2="#52514e", muted="#898781", grid="#e1e0d9", axis="#c3c2b7",
    surface="#fcfcfb", seq=_BLUE_RAMP,
    series=("#2a78d6", "#eb6834", "#1baf7a", "#6250d6", "#e87ba4", "#c98500", "#0f8f8f",
            "#8c564b", "#7a7974", "#b8336a"),
)
DARK = Palette(
    mode="dark", baseline="#d95926", optimized="#3987e5",
    components={"E_IT_kWh": "#199e70", "E_crac_kWh": "#9085e9",
                "E_cp_kWh": "#d55181", "E_fan_kWh": "#c98500"},
    types={"time_sensitive": "#d55181", "delay_tolerant": "#008300"},
    ink="#f0efec", ink2="#c3c2b7", muted="#898781", grid="#2c2c2a", axis="#383835",
    surface="#1a1a19", seq=tuple(reversed(_BLUE_RAMP)),
    series=("#3987e5", "#d95926", "#199e70", "#9085e9", "#d55181", "#eda100", "#2bb5b5",
            "#b07a6a", "#a8a7a1", "#e0628f"),
)


def palette(mode: str | None) -> Palette:
    return DARK if mode == "dark" else LIGHT


def rgba(hex_color: str, alpha: float) -> str:
    h = hex_color.lstrip("#")
    r, g, b = (int(h[i:i + 2], 16) for i in (0, 2, 4))
    return f"rgba({r},{g},{b},{alpha})"


FONT = 'system-ui, -apple-system, "Segoe UI", sans-serif'
TIME_FORMAT = "%H:%M"


def style(fig: go.Figure, pal: Palette, *, height: int = 360, legend: bool = True,
          time_axis: bool = True, hovermode: str | bool = "x unified",
          titled: bool = False) -> go.Figure:
    """Recessive grid and axes, transparent background, legend above the plot.

    ``titled``: the figure has subplot titles, so the legend sits one row higher.
    Charts are drawn with Streamlit's theme off (it replaces colour scales), so every
    colour here comes from the palette for the active light or dark mode.
    """
    lift = 30 if titled and legend else 0
    top = (36 if legend else 16) + lift
    plot_h = max(height - top - 60, 100)
    fig.update_layout(
        template="none", height=height, font=dict(family=FONT, color=pal.ink2, size=13),
        paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(0,0,0,0)",
        margin=dict(l=8, r=8, t=top, b=8),
        hovermode=hovermode, showlegend=legend,
        legend=dict(orientation="h", yanchor="bottom", y=1.0 + lift / plot_h, xanchor="left",
                    x=0, title_text="", font=dict(color=pal.ink2), bgcolor="rgba(0,0,0,0)"),
        hoverlabel=dict(font=dict(family=FONT, color=pal.ink), bgcolor=pal.surface,
                        bordercolor=pal.axis),
    )
    fig.update_xaxes(gridcolor=pal.grid, linecolor=pal.axis, zeroline=False, showgrid=False,
                     tickfont=dict(color=pal.muted), title_font=dict(color=pal.ink2),
                     showline=True, ticks="outside", tickcolor=pal.axis, ticklen=4,
                     automargin=True)
    fig.update_yaxes(gridcolor=pal.grid, linecolor=pal.axis, zerolinecolor=pal.axis,
                     showgrid=True, zeroline=False, tickfont=dict(color=pal.muted),
                     title_font=dict(color=pal.ink2), automargin=True)
    if time_axis:
        fig.update_xaxes(tickformat=TIME_FORMAT, hoverformat=TIME_FORMAT,
                         showspikes=True, spikemode="across", spikethickness=1,
                         spikecolor=pal.muted, spikedash="solid")
    for ann in fig.layout.annotations or ():
        ann.font = dict(family=FONT, color=pal.ink2, size=13)
    return fig


def threshold(fig: go.Figure, pal: Palette, y: float, label: str, *, dash: str = "dash",
              row: int | str | None = None, col: int | str | None = None) -> None:
    """A horizontal limit line in neutral ink with its label at the right end."""
    kw = {} if row is None else dict(row=row, col=col)
    fig.add_hline(y=y, line=dict(color=pal.ink2, width=1, dash=dash),
                  annotation=dict(text=label, font=dict(color=pal.ink2, size=11),
                                  bgcolor=rgba(pal.surface, 0.7)),
                  annotation_position="top right", **kw)


PLOTLY_CONFIG = {"displaylogo": False, "modeBarButtonsToRemove": ["lasso2d", "select2d"]}


In [ ]:
%%writefile {LIB}/dashboard/uploads.py
"""Zip export of run folders, to send runs to the project owner.

The zip mirrors ``results/``: ``<scenario>/<run>/summary.json`` plus the run's CSVs, and the
scenario's ``servers.csv`` / ``carbon.csv`` next to the run folders
(``<scenario>/servers.csv``). Unzipped into ``results/``, the runs show on the site with their
server names and clock times.
"""

from __future__ import annotations

import io
import json
import zipfile
from pathlib import Path

from dashboard.data import RunRef

# The only scenario inputs the site reads (server names and the clock).
SCENARIO_FILES = ("servers.csv", "carbon.csv")


def _inputs_scenario(ref: RunRef) -> str:
    """The scenario whose files a run uses: the name in its summary.json, which differs
    from its folder for an archived run (``archive-<scenario>``)."""
    try:
        name = (json.loads((ref.path / "summary.json").read_text()).get("scenario") or {}).get("name")
    except (OSError, ValueError, AttributeError):
        name = None
    return name or ref.scenario


def runs_zip(runs: list[tuple[RunRef, Path]]) -> bytes:
    """Zip of ``(run, scenarios_dir)`` pairs laid out as ``<scenario>/<run>/...``.

    ``<scenario>`` is the scenario named in the run's summary.json, so an archived run
    (``archive-<scenario>/``) is zipped under its real scenario.
    The scenario's servers.csv and carbon.csv are added as ``<scenario>/<file>`` so the
    zip shows server names and clock times on a machine without that scenario.
    """
    buf = io.BytesIO()
    added: set[str] = set()
    with zipfile.ZipFile(buf, "w", zipfile.ZIP_DEFLATED) as zf:
        for ref, scenarios_dir in runs:
            scenario = _inputs_scenario(ref)
            if f"{scenario}/{ref.name}/" in added:      # an archived and a current run share a name
                scenario = ref.scenario
            added.add(f"{scenario}/{ref.name}/")
            for f in sorted(ref.path.rglob("*")):
                if f.is_file() and not f.name.startswith("."):
                    rel = f.relative_to(ref.path).as_posix()
                    zf.write(f, f"{scenario}/{ref.name}/{rel}")
            for fname in SCENARIO_FILES:
                arc = f"{scenario}/{fname}"
                src = Path(scenarios_dir) / _inputs_scenario(ref) / fname
                if arc not in added and src.is_file():
                    zf.write(src, arc)
                    added.add(arc)
    return buf.getvalue()


## 1. Plan

Every run of the list, checked before anything is solved: its switches are valid, its
scenario exists, and no two entries are the same run. *done* means the run is already in
`results/` from this release on the same scenario build, and is skipped (with
`SKIP_DONE = True`). *stale* means a run with that name is there from another release or
scenario build; it is solved again and replaced.

In [ ]:
import json
import time

import pandas as pd
from IPython.display import display

from dashboard.data import RunRef, compare_runs
from dashboard.uploads import runs_zip
from scheduler.io import load_scenario
from scheduler.run import default_run_name, run
from scheduler.types import RunSettings
from scheduler.version import VERSION

plan = []
for label, switches in RUNS:
    settings = RunSettings(**(DEFAULTS | dict(per_tick=PER_TICK, tick_time_limit_s=TICK_TIME_LIMIT_S,
                                              tick_no_schedule_limit_s=TICK_NO_SCHEDULE_LIMIT_S)
                              | switches))
    settings.validate()
    settings.run_name = "nb_" + default_run_name(settings)
    plan.append((label, settings))
keys = [f"{s.scenario}/{s.run_name}" for _, s in plan]
repeated = sorted({k for k in keys if keys.count(k) > 1})
assert not repeated, f"entries of RUNS with the same switches: {repeated}"
# Loading each scenario also stops here if one is missing or broken.
loaded = {name: load_scenario(name, SCENARIOS_DIR) for name in {s.scenario for _, s in plan}}
fingerprint = {name: sc.fingerprint for name, sc in loaded.items()}


def run_dir(settings):
    return RESULTS_DIR / settings.scenario / settings.run_name


def state(settings):
    path = run_dir(settings) / "summary.json"
    if not path.is_file():
        return "to run"
    made = json.loads(path.read_text()).get("provenance") or {}
    same = (made.get("thermasched_version") == VERSION
            and made.get("scenario_fingerprint") == fingerprint[settings.scenario])
    return "done" if same else "stale"


rows = []
for label, s in plan:
    st = state(s)
    todo = not (SKIP_DONE and st == "done")
    if s.per_tick:
        minutes = loaded[s.scenario].T * s.tick_time_limit_s / 60 + 0.5 if todo else 0
    else:
        minutes = (0.5 if s.reassignment == "none" else s.time_limit_s / 60 + 0.5) if todo else 0
    rows.append({"label": label, "scenario": s.scenario, "run": s.run_name, "state": st,
                 "solve": "yes" if todo else "skip", "up to (min)": round(minutes, 1)})
plan_df = pd.DataFrame(rows)
display(plan_df)
n_todo = int((plan_df["solve"] == "yes").sum())
print(f"ThermaSched {VERSION}: {n_todo} of {len(plan)} runs to solve, "
      f"up to about {plan_df['up to (min)'].sum():.0f} minutes.")

## 2. Run

Each run is solved and written to `results/` before the next one starts. A run that fails
with an error is reported and the list goes on, and so does a per-tick run that gets stuck
at a period with no valid schedule (its limit is shown); the last cell fails if any run had an error
or a failed check.

In [ ]:
outcome = {}
started = time.time()
for k, (label, s) in enumerate(plan, 1):
    head = f"[{k}/{len(plan)}] {label} ({s.scenario}/{s.run_name})"
    if SKIP_DONE and state(s) == "done":
        outcome[label] = "skipped: already done"
        print(f"{head}: already done, skipped")
        continue
    print(f"{head}: solving...", flush=True)
    t0 = time.time()
    try:
        summary = run(s, scenarios_dir=SCENARIOS_DIR, results_dir=RESULTS_DIR, quiet=True,
                      made_with="batch notebook")
    except Exception as e:  # noqa: BLE001 - reported, and the list goes on
        outcome[label] = f"error: {e!r}"
        print(f"    error: {e!r}; going on with the next run")
        continue
    stuck = (summary.get("per_tick") or {}).get("stuck")
    if stuck:
        outcome[label] = f"stuck at period {stuck['tick']}: {stuck['limit_text']}"
        print(f"    {outcome[label]} (no optimized schedule)")
        continue
    status = (summary.get("solver") or {}).get("status", "baseline only")
    failed = [n for n, c in summary["checks"].items() if c.get("passed") is False]
    outcome[label] = f"checks failed: {failed}" if failed else "ok"
    print(f"    {status} in {time.time() - t0:.0f} s" + (f"; FAILED checks: {failed}" if failed else ""))
print(f"\nFinished in {(time.time() - started) / 60:.1f} minutes.")

## 3. Results

One row per run of the list that has results. The carbon saved is each run's optimized
schedule against its own baseline. Read the solver status before the saving: `time_limit`
is a valid schedule that is not proven best, and its gap says how far from best it can be.
The scenario build is the fingerprint of the scenario the run was solved on; runs of one
scenario are comparable only when it is the same.

In [ ]:
refs = {label: RunRef(s.scenario, s.run_name, run_dir(s)) for label, s in plan
        if (run_dir(s) / "summary.json").is_file()}
results_df = compare_runs(list(refs.values()))
results_df.insert(0, "label", list(refs))
results_df["outcome"] = [outcome.get(label, "not solved in this session") for label in refs]
cols = ["label", "scenario", "run", "status", "mip_gap", "baseline_carbon_kg", "optimized_carbon_kg",
        "carbon_saved_pct", "n_reassigned", "version", "scenario_build", "outcome"]
display(results_df[cols].style.format({"mip_gap": "{:.2%}", "baseline_carbon_kg": "{:.3f}",
                                       "optimized_carbon_kg": "{:.3f}", "carbon_saved_pct": "{:.2f}%"},
                                      na_rep="-"))

## 4. Zip for the site

Every run of the list that has results goes into `exports/<ZIP_NAME>.zip`, with each
scenario's `servers.csv` and `carbon.csv` so server names and clock times show correctly. On
Colab the zip is also downloaded.

To put the runs on the ThermaSched site, send the zip to the project owner.

This cell fails if any run had an error or a failed check.

In [ ]:
zip_path = EXPORTS_DIR / f"{ZIP_NAME}.zip"
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)
zip_path.write_bytes(runs_zip([(ref, SCENARIOS_DIR) for ref in refs.values()]))
print(f"{len(refs)} runs in {zip_path} ({zip_path.stat().st_size / 1e6:.1f} MB)")
if IN_COLAB and DOWNLOAD_ZIP:
    from google.colab import files
    files.download(str(zip_path))
problems = {label: o for label, o in outcome.items() if o.startswith(("error", "checks failed"))}
assert not problems, f"runs with problems: {problems}"